<a href="https://colab.research.google.com/github/Sophialllin/CQF-/blob/claude/fix-lookahead-bias-backtest-79h5ii/CQF_Volatility_Direction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Colab setup: these packages are not preinstalled on a fresh Colab runtime
!pip install -q yfinance pandas_datareader lightgbm xgboost


# CQF Final Project: Machine Learning for Volatility-Direction Prediction

**Variant of the price-direction notebook** — same data, features, and blending-ensemble architecture; only the prediction target changed (see Section 1.7). Built after the price-direction model showed AUC≈0.50 (no real edge) even after correcting a look-ahead bias in its backtest.

## Author: Jie Lin
## Date: January 2026

---

# Table of Contents

1. **Problem Statement**
2. **Feature Engineering**
3. **Exploratory Data Analysis (EDA)** — includes missing-value handling, multicollinearity analysis, feature transformation, and dimensionality reduction (K-Means)
4. **Model Building (Blending Ensemble)**
5. **Performance Evaluation & Backtesting**

---

# 1. Problem Statement

## 1.1 Objective

This project aims to build a **machine learning classification model** to predict the **direction of 20-trading-day-forward realized volatility** (rising vs falling, relative to the trailing 20-day realized volatility) of the **QQQ ETF** (Nasdaq-100 tracking fund). This targets volatility clustering/mean-reversion instead of price direction — a prior variant of this notebook targeting next-day price direction achieved AUC≈0.50 (no real edge) even after correcting a look-ahead bias in its backtest.

---

## 1.2 Asset Selection

**Underlying Asset:** QQQ ETF (Invesco QQQ Trust)
- Tracks the Nasdaq-100 Index
- Highly liquid technology-focused ETF
- Strong representation of US large-cap growth stocks
- Sufficient volatility for meaningful directional prediction

---

## 1.3 Data Specification

### **Data Source:**
- **Price Data:** Yahoo Finance API (`yfinance`)
- **Macro Data:** Federal Reserve Economic Data (FRED) API
- **Sentiment Data:** VIX (Yahoo Finance)

### **Timeframe:**
- **Start Date:** 2020-12-29
- **End Date:** 2025-12-29
- **Total Duration:** ~5 years of daily data
- **Rationale:** Sufficient for daily return prediction

### **Frequency:**
- **Daily OHLCV data** (Open, High, Low, Close, Adjusted Close, Volume)
- Prediction target: **1-day ahead return direction**

---

## 1.4 Prediction Target

**Target Variable:** Binary classification of forward vs. trailing 20-day realized volatility

- **Class 1 (CALM AHEAD):** Realized vol over [t+1, t+20] < trailing realized vol over [t-19, t]
- **Class 0 (STORM AHEAD):** Realized vol over [t+1, t+20] >= trailing realized vol over [t-19, t]
- **Near-flat filtering:** Days within the 10th percentile of |forward vol − trailing vol| are excluded from training to reduce label noise (mirrors the near-zero return filter used in the price-direction version)
- **Label polarity chosen so `prob > 0.5 -> long QQQ` is still the correct trading rule** for every backtest cell reused from the price-direction version (Section 4 onward)

**Label Definition:**
```python
vol_now  = ret_log.rolling(20).std()               # uses only data through day t
vol_fwd  = ret_log.rolling(20).std().shift(-20)     # realized vol over [t+1, t+20], future data only
label = 1 if vol_fwd < vol_now else 0               # undefined at the series' head/tail -> excluded, not a fake 0/1
```

**Class Imbalance Handling:**
- No prior assumption of a natural skew (unlike price direction's long-run upward drift) — check the printed class balance in Section 1.7 once this notebook is run
- Re-evaluate near-flat threshold / resampling if the realized split turns out heavily imbalanced

---

## 1.5 Data Collection

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

# 1. Define parameters
TICKER = "QQQ"
START = "2020-12-29"
END = "2025-12-30"  # to include the 2025/12/29

# 2. Download daily OHLCV with Adjusted Close for QQQ ETF
df = yf.download(
    TICKER,
    start=START,
    end=END,
    interval="1d",
    auto_adjust=False,   # Keep Adjusted Close column
    actions=False,
    progress=False
)

if df.empty:
    raise ValueError("Downloaded dataframe is empty")

# 3. Import data to base table
OHLCV = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
missing = [c for c in OHLCV if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in downloaded data: {missing}. Columns found: {list(df.columns)}")

df = df[OHLCV].copy()

# Flatten column names if MultiIndex (remove ticker suffix like 'QQQ')
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.index = pd.to_datetime(df.index) # Ensure index is datetime
df = df[~df.index.duplicated(keep="first")] # Remove duplicate indices
df = df.sort_index() # Sort ascending by date

In [5]:
# 4. Check data
print(f"Data shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Missing values per column:\n{df.isnull().sum()}")
print(f"Data head:{df.head()}")

Data shape: (1256, 6)
Date range: 2020-12-29 00:00:00 to 2025-12-29 00:00:00
Missing values per column:
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64
Data head:                  Open        High         Low       Close   Adj Close  \
Date                                                                     
2020-12-29  314.049988  314.690002  312.029999  312.959991  302.989777   
2020-12-30  314.160004  314.489990  312.329987  312.970001  302.999451   
2020-12-31  312.869995  314.239990  311.760010  313.739990  303.744873   
2021-01-04  315.109985  315.290009  305.179993  309.309998  299.456024   
2021-01-05  308.290009  312.140015  308.290009  311.859985  301.924835   

              Volume  
Date                  
2020-12-29  25871900  
2020-12-30  18138100  
2020-12-31  21611400  
2021-01-04  45305900  
2021-01-05  29323400  


In [6]:
# Verify column format and data structure
print("Data Structure Verification")

# 1. Check column name format
print("\nColumn name types:")
print(f"Column index type: {type(df.columns)}")
print(f"Is MultiIndex: {isinstance(df.columns, pd.MultiIndex)}")

# 2. Check shape of each column
print("\nColumn shapes:")
for col in df.columns:
    print(f"df['{col}']:  Type={type(df[col]).__name__:15s}  Shape={df[col].shape}")

# 3. Check .values operation results
print("\nShape after .values operation:")
print(f"df['Adj Close'].values:  Shape={df['Adj Close'].values.shape}")
print(f"df['Volume'].values:     Shape={df['Volume'].values.shape}")


Data Structure Verification

Column name types:
Column index type: <class 'pandas.Index'>
Is MultiIndex: False

Column shapes:
df['Open']:  Type=Series           Shape=(1256,)
df['High']:  Type=Series           Shape=(1256,)
df['Low']:  Type=Series           Shape=(1256,)
df['Close']:  Type=Series           Shape=(1256,)
df['Adj Close']:  Type=Series           Shape=(1256,)
df['Volume']:  Type=Series           Shape=(1256,)

Shape after .values operation:
df['Adj Close'].values:  Shape=(1256,)
df['Volume'].values:     Shape=(1256,)


# 1.6 Initial Data Exploration

In [7]:
df["Adj Close"].plot(title="QQQ Adjusted Close Price")
plt.show()

In [8]:
import numpy as np
import matplotlib.pyplot as plt

# simple return
df["ret_simple"] = df["Adj Close"].pct_change()

# log return
df["ret_log"] = np.log(df["Adj Close"] / df["Adj Close"].shift(1))

rets = df[["ret_simple", "ret_log"]].dropna()

# Basic statistics
print(rets.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

# Distribution comparison (histogram)
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.hist(rets["ret_simple"], bins=100)
plt.title("Simple Return Distribution")

plt.subplot(1,2,2)
plt.hist(rets["ret_log"], bins=100)
plt.title("Log Return Distribution")

plt.tight_layout()
plt.show()


        ret_simple      ret_log
count  1255.000000  1255.000000
mean      0.000672     0.000570
std       0.014282     0.014265
min      -0.062109    -0.064121
1%       -0.038404    -0.039161
5%       -0.023693    -0.023978
50%       0.001134     0.001134
95%       0.022411     0.022163
99%       0.033700     0.033145
max       0.120031     0.113356


### Return Metric Selection

Both simple returns and log returns are examined during exploratory data analysis. While their first and second moments are largely similar at the daily frequency, log returns exhibit a more symmetric distribution with reduced influence from extreme positive values.

**Decision: Use Log Returns**
- Additive property: easier to calculate multi-period returns
- Better statistical stability
- More symmetric distribution
- Standard practice in quantitative finance



## 1.7 Target Label Definition — Volatility Direction

**This variant of the notebook predicts volatility direction instead of price direction** (see the [look-ahead-bias fix and Section 7.4 discussion] in the price-direction version for why: raw price-direction AUC sat at ~0.50 — indistinguishable from a coin flip — while this feature set (VIX, VXN, realized-vol, range features) is far more naturally suited to forecasting volatility, which is known to cluster/mean-revert rather than follow a random walk.

### 1.7.1: Handle Near-Flat Volatility Changes
Days where the forward/trailing volatility ratio is close to 1 (ambiguous — neither a real expansion nor contraction) are filtered, the same way near-zero returns were filtered in the price-direction version.

In [9]:
# 1. Analyze near-flat volatility-change threshold
# vol_now: trailing 20d realized vol as of day T (uses only data through T -> no look-ahead)
# vol_fwd: realized vol over T+1..T+20 (future window; label/target only, matched window length to vol_now
#          so we are not comparing a noisy short estimate against a smooth long one)
vol_now = df["ret_log"].rolling(20).std()
vol_fwd = df["ret_log"].rolling(20).std().shift(-20)
vol_valid = vol_now.notna() & vol_fwd.notna()
vol_chg = (vol_fwd - vol_now)[vol_valid]
abs_vol_chg = vol_chg.abs()

# Calculate data-driven thresholds
thresholds = {
    "10th percentile": abs_vol_chg.quantile(0.10),
    "20th percentile": abs_vol_chg.quantile(0.20),
    "25th percentile": abs_vol_chg.quantile(0.25),
}

print("Potential near-flat volatility-change thresholds:")
for name, thresh in thresholds.items():
    count = (abs_vol_chg <= thresh).sum()
    pct = count / len(abs_vol_chg) * 100
    print(f"{name:15}: {thresh:.6f} ({count:4d} obs, {pct:4.1f}%)")

# Visualize threshold impact
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(abs_vol_chg, bins=100, alpha=0.7, edgecolor='black')
for name, thresh in thresholds.items():
    plt.axvline(thresh, color='red', linestyle='--', alpha=0.7, label=f"{name}: {thresh:.4f}")
plt.xlabel("|Forward 20d Vol - Trailing 20d Vol|")
plt.ylabel("Frequency")
plt.title("Distribution of Volatility Changes")
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(abs_vol_chg, bins=100, alpha=0.7, edgecolor='black')
plt.xlim(0, abs_vol_chg.quantile(0.95))  # Zoom into the bulk of the distribution
for name, thresh in thresholds.items():
    plt.axvline(thresh, color='red', linestyle='--', alpha=0.7, label=f"{name}: {thresh:.4f}")
plt.xlabel("|Forward 20d Vol - Trailing 20d Vol|")
plt.ylabel("Frequency")
plt.title("Near-Flat Region (Zoomed)")
plt.legend()

plt.tight_layout()
plt.show()


Potential near-flat volatility-change thresholds:
10th percentile: 0.000410 ( 122 obs, 10.0%)
20th percentile: 0.000842 ( 244 obs, 20.1%)
25th percentile: 0.001050 ( 304 obs, 25.0%)


### Rationale for Near-Flat Volatility-Change Threshold

Volatility is a slow-moving, mean-reverting series (unlike returns, which are close to a random walk), so a meaningful fraction of day-to-day changes in the forward-vs-trailing 20-day realized vol comparison are small and directionally ambiguous rather than a genuine regime shift.

**Selection:** 10th percentile of |forward vol − trailing vol|
- Filters cases where the vol "direction" is essentially noise
- Removes ~10% of observations, consistent with the price-direction version's filter
- Reduces label noise while maintaining statistical power

In [10]:
# Choose 10th percentile threshold
CHOSEN_THRESHOLD = thresholds["10th percentile"]
print(f"Selected threshold: {CHOSEN_THRESHOLD:.6f}")
print(f"This will affect {(abs_vol_chg <= CHOSEN_THRESHOLD).sum()} observations ({(abs_vol_chg <= CHOSEN_THRESHOLD).mean()*100:.1f}%)")


Selected threshold: 0.000410
This will affect 122 observations (10.0%)


### 1.7.2: Create Volatility-Direction Target Labels

**Label polarity, chosen deliberately so every downstream cell (backtest, threshold overlay, etc.) works unchanged:** `label_binary = 1` means volatility is expected to **fall** over the next 20 trading days relative to the trailing 20 days ("calm ahead" → safe to hold QQQ), `label_binary = 0` means volatility is expected to **rise** ("storm ahead" → de-risk to cash). This is the opposite sign convention from a naive "1 = vol up" label, but it means `prob > 0.5 -> long` (the rule every cell from Section 4 onward already uses) is still the economically correct trading rule.

Compare impact of labeling strategies on class balance:
- **Strategy 1:** Simple Binary (all days)
- **Strategy 2:** Drop Near-Flat (training only)

In [11]:
# LABEL CREATION PROCESS

df_fe = df.copy()

# --- INTERMEDIATE VARIABLES (kept for structural compatibility with the rest of the notebook;
#     these two columns are already excluded from feature_cols further down, same as before) ---
df_fe["ret_fwd"] = df_fe["ret_log"].shift(-1)      # next-day return (unused by the vol label; kept as a
df_fe["abs_ret_fwd"] = df_fe["ret_fwd"].abs()      # placeholder so Section 3's EXCLUDE_COLS logic is untouched

# --- LABELS (y) - What we actually predict ---
# vol_now / vol_fwd are LOCAL variables (not stored as df_fe columns), so they never leak into feature_cols.
# vol_now uses only data through day T (no look-ahead); vol_fwd is strictly future data (T+1..T+20) and is
# used only to build the label, never as a feature.

# Strategy 1: Simple Binary Label
# label_binary = 1  ->  volatility is expected to FALL (calm ahead, safe to be long)
# label_binary = 0  ->  volatility is expected to RISE (storm ahead, de-risk to cash)
label_binary_raw = (vol_fwd < vol_now).astype(float)
label_binary_raw[~vol_valid] = np.nan   # undefined at the head/tail of the window -> exclude, not a fake 0
df_fe["label_binary"] = label_binary_raw

# Strategy 2: Filtered Label (Drop Near-Flat)
near_flat_mask = (abs(vol_fwd - vol_now) <= CHOSEN_THRESHOLD) & vol_valid
df_fe["label_drop_nearzero"] = df_fe["label_binary"].copy()
df_fe.loc[near_flat_mask, "label_drop_nearzero"] = np.nan  # Mark for exclusion

# Only label_binary or label_drop_nearzero will be used as y

# Compare label strategies
strategies = {
    "Binary (Original)": "label_binary",
    "Drop Near-Flat": "label_drop_nearzero"
}

print("Label Strategy Comparison (1 = vol falling / calm ahead, 0 = vol rising / storm ahead):")
for name, col in strategies.items():
    valid_mask = ~df_fe[col].isna()
    counts = df_fe.loc[valid_mask, col].value_counts().sort_index()
    total = valid_mask.sum()
    excluded = (~valid_mask).sum()
    print(f"\n{name}:")
    print(f"  Excluded (NaN / near-flat): {excluded} ({excluded/len(df_fe)*100:.1f}%)")
    print(f"  Training samples: {total}")
    for cls, count in counts.items():
        print(f"  Class {int(cls)}: {count} ({count/total*100:.1f}%)")


Label Strategy Comparison (1 = vol falling / calm ahead, 0 = vol rising / storm ahead):

Binary (Original):
  Excluded (NaN / near-flat): 40 (3.2%)
  Training samples: 1216
  Class 0: 622 (51.2%)
  Class 1: 594 (48.8%)

Drop Near-Flat:
  Excluded (NaN / near-flat): 162 (12.9%)
  Training samples: 1094
  Class 0: 557 (50.9%)
  Class 1: 537 (49.1%)


In [12]:
# Visualize label distribution comparison
plt.figure(figsize=(12, 8))

# Plot 1: Binary Original Distribution
plt.subplot(2, 2, 1)
binary_counts = df_fe["label_binary"].value_counts().sort_index()
plt.bar(binary_counts.index.astype(str), binary_counts.values, color=['red', 'green'], alpha=0.7)
plt.title("Binary (Original) Label Distribution")
plt.xlabel("Class (0=Vol Rising/Storm, 1=Vol Falling/Calm)")
plt.ylabel("Count")
for i, v in enumerate(binary_counts.values):
    plt.text(i, v + 10, f'{v}\n({v/binary_counts.sum():.1%})', ha='center', va='bottom')

# Plot 2: Drop Near-Zero Distribution
plt.subplot(2, 2, 2)
valid_mask = ~df_fe["label_drop_nearzero"].isna()
drop_counts = df_fe.loc[valid_mask, "label_drop_nearzero"].value_counts().sort_index()
excluded_count = (~valid_mask).sum()
plt.bar(drop_counts.index.astype(str), drop_counts.values, color=['red', 'green'], alpha=0.7)
plt.title(f"Drop Near-Flat Distribution\n(Excluded: {excluded_count} obs)")
plt.xlabel("Class (0=Vol Rising/Storm, 1=Vol Falling/Calm)")
plt.ylabel("Count")
for i, v in enumerate(drop_counts.values):
    plt.text(i, v + 10, f'{v}\n({v/drop_counts.sum():.1%})', ha='center', va='bottom')

# Plot 3: Class Balance Comparison
plt.subplot(2, 2, 3)
balance_comparison = {
    'Binary\n(Original)': binary_counts.min() / binary_counts.max(),
    'Drop\nNear-Flat': drop_counts.min() / drop_counts.max()
}
bars = plt.bar(balance_comparison.keys(), balance_comparison.values(),
               color=['lightblue', 'lightcoral'], alpha=0.7)
plt.title("Class Balance Comparison")
plt.ylabel("Balance Ratio (min/max)")
plt.axhline(y=0.8, color='green', linestyle='--', alpha=0.7, label='Well Balanced (>0.8)')
plt.axhline(y=0.6, color='orange', linestyle='--', alpha=0.7, label='Slight Imbalance (>0.6)')
plt.legend()
for bar, ratio in zip(bars, balance_comparison.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{ratio:.3f}', ha='center', va='bottom')

# Plot 4: Sample Size Comparison
plt.subplot(2, 2, 4)
sample_sizes = {
    'Binary\n(Original)': binary_counts.sum(),
    'Drop\nNear-Flat': drop_counts.sum()
}
bars = plt.bar(sample_sizes.keys(), sample_sizes.values(),
               color=['lightblue', 'lightcoral'], alpha=0.7)
plt.title("Training Sample Size Comparison")
plt.ylabel("Number of Samples")
for bar, size in zip(bars, sample_sizes.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'{size}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# Summary statistics
print("Label Distribution Summary:")
print(f"Original Binary Strategy:")
print(f"  • Total samples: {binary_counts.sum()}")
print(f"  • Class balance: {binary_counts.min() / binary_counts.max():.3f}")
print(f"  • Class 0: {binary_counts[0]} ({binary_counts[0]/binary_counts.sum():.1%})")
print(f"  • Class 1: {binary_counts[1]} ({binary_counts[1]/binary_counts.sum():.1%})")

print(f"\nDrop Near-Flat Strategy:")
print(f"  • Training samples: {drop_counts.sum()}")
print(f"  • Excluded samples: {excluded_count} ({excluded_count/len(df_fe):.1%})")
print(f"  • Class balance: {drop_counts.min() / drop_counts.max():.3f}")
print(f"  • Class 0: {drop_counts[0]} ({drop_counts[0]/drop_counts.sum():.1%})")
print(f"  • Class 1: {drop_counts[1]} ({drop_counts[1]/drop_counts.sum():.1%})")

Label Distribution Summary:
Original Binary Strategy:
  • Total samples: 1216
  • Class balance: 0.955
  • Class 0: 622 (51.2%)
  • Class 1: 594 (48.8%)

Drop Near-Flat Strategy:
  • Training samples: 1094
  • Excluded samples: 162 (12.9%)
  • Class balance: 0.964
  • Class 0: 557 (50.9%)
  • Class 1: 537 (49.1%)


### Label Strategy Conclusion

Applying a near-flat volatility-change filter based on the 10th percentile of |forward vol − trailing vol| removes approximately 10% of observations dominated by estimation noise, mirroring the price-direction version's near-zero filter.

**Note:** the exact class balance here depends on the realized 2020-2025 volatility regime and should be checked against the printed output above once this cell is run — unlike price direction (which has a well-known long-run upward drift making the classes naturally imbalanced ~45/55), volatility direction has no equivalent prior reason to expect a specific skew.

---

# 2. Feature Engineering

## 2.0 Feature Engineering Framework

The feature engineering follows a three-tier hierarchical framework:

### ① Core Price Dynamics (Intrinsic Price Structure)
**Philosophy:** Price action contains intrinsic structural information
- **Momentum:** Recent acceleration patterns and short-term persistence
- **Trend:** Directional consistency and medium-term positioning
- **Drawdown:** Healthy retracement patterns during upward movements

### ② Risk & Confirmation (Risk State + Trend Sustainability)  
**Philosophy:** Assess whether trends can sustain themselves
- **Volatility:** Market instability that may disrupt short-term trends
- **Volume:** Participation confirmation - genuine activity vs noise

### ③ Exogenous Regime (Environmental Constraints)
**Philosophy:** Price dynamics operate within broader constraints
- **Macro:** Interest rates and financial conditions
- **Sentiment:** Fear/greed cycles influencing behavior

---

## 2.1 Momentum Features

**Purpose:** Capture immediate directional information and short-horizon persistence

**Derived Features:**
1. `ret_log` - 1-day log return (base momentum)
2. `cum_ret_5`, `cum_ret_10` - Cumulative returns over 5 and 10 days
3. `ret_ma_5`, `ret_ma_10` - Rolling mean returns over 5 and 10 days

In [13]:
# 2.1 Return / Momentum Features

MOMENTUM_WINDOWS = [5, 10]

# Feature 1: 1-Day Log Return (already computed as df_fe["ret_log"])

# Feature 2: Cumulative Returns- Sum over k days
for window in MOMENTUM_WINDOWS:
    df_fe[f"cum_ret_{window}"] = df_fe["ret_log"].rolling(window).sum()

# Feature 3: Rolling Mean Returns - Average over k days
for window in MOMENTUM_WINDOWS:
    df_fe[f"ret_ma_{window}"] = df_fe["ret_log"].rolling(window).mean()

print(f"Created momentum features:")
print(f"   - Base: ret_log (1-day log return)")
print(f"   - Cumulative: {[f'cum_ret_{w}' for w in MOMENTUM_WINDOWS]}")
print(f"   - Rolling Mean: {[f'ret_ma_{w}' for w in MOMENTUM_WINDOWS]}")


Created momentum features:
   - Base: ret_log (1-day log return)
   - Cumulative: ['cum_ret_5', 'cum_ret_10']
   - Rolling Mean: ['ret_ma_5', 'ret_ma_10']


In [14]:
print("DataFrame Columns:")
print(f"Total columns: {len(df_fe.columns)}")
print("\nColumn names:")
print(list(df_fe.columns))


DataFrame Columns:
Total columns: 16

Column names:
['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'ret_simple', 'ret_log', 'ret_fwd', 'abs_ret_fwd', 'label_binary', 'label_drop_nearzero', 'cum_ret_5', 'cum_ret_10', 'ret_ma_5', 'ret_ma_10']


## 2.2 Trend Confirmation Features

**Design Philosophy:** Filter false momentum signals through:
- **Positioning:** Price location relative to trend anchors
- **Slope:** Underlying trend direction and strength  
- **Alignment:** Multi-timeframe consistency
- **Context:** Volatility-adjusted validation

**Derived Features:**
1. `price_ma_20`, `price_ma_60` - Moving averages (20 and 60 days)
2. `trend_pos`, `trend_pos_norm` - Distance from MA20 (absolute & normalized)
3. `ma20_slope`, `ma60_slope` - Trend slope over 5 and 10 days
4. `golden_cross`, `golden_cross_norm` - MA20-MA60 spread
5. `bb_upper`, `bb_lower`, `bb_position`, `bb_width`, `bb_std` - Bollinger Bands

In [15]:
# 2.2 Trend Confirmation Features

TREND_WINDOWS = [20, 60]

# Moving averages
for window in TREND_WINDOWS:
    df_fe[f"price_ma_{window}"] = df_fe["Adj Close"].rolling(window, min_periods=window).mean()

# (1) Trend Positioning
df_fe["trend_pos"] = df_fe["Adj Close"] - df_fe["price_ma_20"]
df_fe["trend_pos_norm"] = df_fe["trend_pos"] / df_fe["Adj Close"]

# (2) Trend Slope
df_fe["ma20_slope"] = (df_fe["price_ma_20"] - df_fe["price_ma_20"].shift(5)) / 5
df_fe["ma60_slope"] = (df_fe["price_ma_60"] - df_fe["price_ma_60"].shift(10)) / 10

# (3) Trend Alignment
df_fe["golden_cross"] = df_fe["price_ma_20"] - df_fe["price_ma_60"]
df_fe["golden_cross_norm"] = df_fe["golden_cross"] / df_fe["price_ma_60"]

# (4) Bollinger Bands
df_fe["bb_std"] = df_fe["Adj Close"].rolling(20, min_periods=20).std()

df_fe["bb_upper"] = df_fe["price_ma_20"] + 2 * df_fe["bb_std"]
df_fe["bb_lower"] = df_fe["price_ma_20"] - 2 * df_fe["bb_std"]

den = (df_fe["bb_upper"] - df_fe["bb_lower"])
df_fe["bb_position"] = (df_fe["Adj Close"] - df_fe["bb_lower"]) / den

df_fe["bb_width"] = (df_fe["bb_upper"] - df_fe["bb_lower"]) / df_fe["price_ma_20"]

# Breakouts
df_fe["bb_breakout_up"] = (
    (df_fe["Adj Close"] > df_fe["bb_upper"]) &
    (df_fe["Adj Close"].shift(1) <= df_fe["bb_upper"].shift(1))
).astype(int)

df_fe["bb_breakout_down"] = (
    (df_fe["Adj Close"] < df_fe["bb_lower"]) &
    (df_fe["Adj Close"].shift(1) >= df_fe["bb_lower"].shift(1))
).astype(int)

# Squeeze (20th percentile over 60-day window)
df_fe["bb_width_q20_60"] = df_fe["bb_width"].rolling(60, min_periods=60).quantile(0.2)
df_fe["bb_squeeze"] = (df_fe["bb_width"] < df_fe["bb_width_q20_60"]).astype(int)

print(f"Created Trend features:")
print(f"   - Positioning: ['trend_pos', 'trend_pos_norm']")
print(f"   - Slope: ['ma20_slope', 'ma60_slope']")
print(f"   - Alignment: ['golden_cross', 'golden_cross_norm']")
print(f"   - Bollinger Bands: ['bb_position', 'bb_width', 'bb_breakout_up', 'bb_breakout_down', 'bb_squeeze']")

Created Trend features:
   - Positioning: ['trend_pos', 'trend_pos_norm']
   - Slope: ['ma20_slope', 'ma60_slope']
   - Alignment: ['golden_cross', 'golden_cross_norm']
   - Bollinger Bands: ['bb_position', 'bb_width', 'bb_breakout_up', 'bb_breakout_down', 'bb_squeeze']


In [16]:
print("DataFrame Columns:")
print(f"Total columns: {len(df_fe.columns)}")
print("\nColumn names:")
print(list(df_fe.columns))

DataFrame Columns:
Total columns: 33

Column names:
['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'ret_simple', 'ret_log', 'ret_fwd', 'abs_ret_fwd', 'label_binary', 'label_drop_nearzero', 'cum_ret_5', 'cum_ret_10', 'ret_ma_5', 'ret_ma_10', 'price_ma_20', 'price_ma_60', 'trend_pos', 'trend_pos_norm', 'ma20_slope', 'ma60_slope', 'golden_cross', 'golden_cross_norm', 'bb_std', 'bb_upper', 'bb_lower', 'bb_position', 'bb_width', 'bb_breakout_up', 'bb_breakout_down', 'bb_width_q20_60', 'bb_squeeze']


## 2.3 Drawdown & Positioning

**Design Philosophy:** Capture positioning relative to recent highs/lows and recovery dynamics

**Derived Features:**
1. `dd_60`, `log_dd_60` - Peak-to-current drawdown (60-day window)
2. `price_percentile_60` - Normalized position within high-low range
3. `dd_vol_ratio` - Volatility-adjusted drawdown
4. `recovery_strength`, `recovery_speed` - Short-term recovery metrics (20-day)
5. Helper metrics: `rolling_max_60`, `rolling_min_60`, `rolling_min_20`, `days_since_low`

In [17]:
# 2.3 Drawdown & Positioning Features

# Configuration
DD_WINDOW = 60  # 60 trading days for drawdown (~1 quarter)
RECOVERY_WINDOW = 20  # 20 trading days for recovery (~1 month)
VOL_WINDOW = 20  # 20 trading days for volatility-adjusted drawdown

# Calculate rolling maximum and minimum
df_fe["rolling_max_60"] = df_fe["Adj Close"].rolling(DD_WINDOW).max()
df_fe["rolling_min_60"] = df_fe["Adj Close"].rolling(DD_WINDOW).min()
df_fe["rolling_min_20"] = df_fe["Adj Close"].rolling(RECOVERY_WINDOW).min()

df_fe["vol_20"] = df_fe["ret_log"].rolling(VOL_WINDOW, min_periods=VOL_WINDOW).std()

# (1) Medium-Term Drawdown (60-day)
df_fe["dd_60"] = (df_fe["Adj Close"] / df_fe["rolling_max_60"]) - 1
df_fe["log_dd_60"] = np.log(1 - df_fe["dd_60"])  # Log transform for better scaling

# (2) Range Position (60-day)
df_fe["price_percentile_60"] = (
    (df_fe["Adj Close"] - df_fe["rolling_min_60"]) /
    (df_fe["rolling_max_60"] - df_fe["rolling_min_60"])
)

# (3) Volatility-Adjusted Drawdown
df_fe["dd_vol_ratio"] = np.abs(df_fe["dd_60"]) / df_fe["vol_20"]

# Helper function: Calculate days since most recent local minimum
def get_days_since_low(series, window):
    """Calculate days since trough for each observation"""
    days_since = []
    for i in range(len(series)):
        if i < window - 1:
            days_since.append(np.nan)
            continue

        # Get the window data
        window_data = series.iloc[i-window+1:i+1]
        min_val = window_data.min()

        # Find the last occurrence of the minimum (most recent low)
        if len(window_data[window_data == min_val]) > 0:
            # Get the index of the last (most recent) low
            last_low_idx = window_data.index[window_data == min_val][-1]
            current_idx = series.index[i]
            # Calculate days since low
            days_diff = (series.index.get_loc(current_idx) -
                        series.index.get_loc(last_low_idx))
            days_since.append(max(days_diff, 1))  # Avoid division by zero
        else:
            days_since.append(1)

    return days_since

# (4) Recovery Strength (20-day)
df_fe["recovery_strength"] = (df_fe["Adj Close"] - df_fe["rolling_min_20"]) / df_fe["rolling_min_20"]

# (5) Recovery Speed (20-day)
df_fe["days_since_low"] = get_days_since_low(df_fe["Adj Close"], RECOVERY_WINDOW)
df_fe["recovery_speed"] = df_fe["recovery_strength"] / np.maximum(df_fe["days_since_low"], 1)


In [18]:
print(f"\nCreated Drawdown & Positioning Features:")
print(f"  Medium-Term Drawdown (60d): dd_60, log_dd_60")
print(f"  Range Position (60d): price_percentile_60")
print(f"  Volatility-Adjusted DD: dd_vol_ratio")
print(f"  Recovery Strength (20d): recovery_strength")
print(f"  Recovery Speed (20d): recovery_speed")



Created Drawdown & Positioning Features:
  Medium-Term Drawdown (60d): dd_60, log_dd_60
  Range Position (60d): price_percentile_60
  Volatility-Adjusted DD: dd_vol_ratio
  Recovery Strength (20d): recovery_strength
  Recovery Speed (20d): recovery_speed


## 2.4 Volatility

**Design Philosophy:** Capture market instability and volatility regime shifts

**Derived Features:**
1. `vol_20` - 20-day realized volatility
2. `log_rv_20` - Log-transformed volatility for better distribution
3. `vol_regime` - Volatility relative to 60-day median
4. `vol_trend` - 5-day volatility momentum
5. `range_5d_smooth` - Smoothed intraday range (High-Low)/Close

In [19]:
# 2.3 Volatility Features

VOL_WINDOW = 20
VOL_REGIME_WINDOW = 60
VOL_TREND_WINDOW = 5
RANGE_WINDOW = 5

# Calculate 20-day realized volatility
df_fe["vol_20"] = df_fe["ret_log"].rolling(VOL_WINDOW,min_periods=VOL_WINDOW).std()

# Feature 1: log_rv_20 = log(1 + σ_20t) - Baseline volatility，measuring in log transformation for better distribution properties
df_fe["log_rv_20"] = np.log(1 + df_fe["vol_20"])

# Feature 2: vol_regime = σ_20t / median(σ_20{t-59:t}) - Volatility regime
df_fe["vol_regime"] = df_fe["vol_20"] / df_fe["vol_20"].rolling(VOL_REGIME_WINDOW,min_periods=VOL_REGIME_WINDOW).median()

# Feature 3: vol_trend = σ_20t / σ_20{t-5} - Volatility direction change
df_fe["vol_trend"] = df_fe["vol_20"] / df_fe["vol_20"].shift(VOL_TREND_WINDOW)

# Feature 4: range_5d_smooth = mean((H-L)/C_{t-4:t}) - Intraday volatility
df_fe["daily_range"] = (df_fe["High"] - df_fe["Low"]) / df_fe["Close"]
df_fe["range_5d_smooth"] = df_fe["daily_range"].rolling(RANGE_WINDOW,min_periods=RANGE_WINDOW).mean()


## 2.5 Volume

**Design Philosophy:** Validate conviction behind price movements through participation analysis

**Derived Features:**
1. `log_vol_mean_20`, `log_vol_std_20` - Log-transformed volume statistics
2. `vol_zscore` - Standardized volume anomaly detector
3. `up_down_vol_ratio` - Directional volume asymmetry
4. `breakout_volume` - Volume confirmation for Bollinger breakouts

In [20]:
# 2.5 Volume Features

VOL_WINDOW = 20

# Calculate log volume for better distribution
df_fe["log_vol"] = np.log(1 + df_fe["Volume"])

# (1) Abnormal Volume Intensity (vol_z_20)
# Z-score of log volume over rolling 20-day window
df_fe["vol_mean_20"] = df_fe["log_vol"].rolling(VOL_WINDOW, min_periods=VOL_WINDOW).mean()
df_fe["vol_std_20"] = df_fe["log_vol"].rolling(VOL_WINDOW, min_periods=VOL_WINDOW).std()
df_fe["vol_z_20"] = (df_fe["log_vol"] - df_fe["vol_mean_20"]) / df_fe["vol_std_20"]

# (2) Up vs Down Volume Ratio (buy vs sell participation structure)
def calc_up_down_vol_ratio(df, window=20):
    """
    Calculate ratio of average volume on up days vs down days
    """
    ratios = []

    for i in range(len(df)):
        if i < window - 1:
            ratios.append(np.nan)
            continue

        # Get window data
        window_returns = df["ret_log"].iloc[i-window+1:i+1]
        window_vol = df["log_vol"].iloc[i-window+1:i+1]

        # Separate up and down days
        up_days_mask = window_returns > 0
        down_days_mask = window_returns < 0

        up_vol = window_vol[up_days_mask]
        down_vol = window_vol[down_days_mask]

        # Calculate ratio
        if len(up_vol) > 0 and len(down_vol) > 0:
            up_vol_mean = up_vol.mean()
            down_vol_mean = down_vol.mean()
            ratio = up_vol_mean / down_vol_mean
        else:
            ratio = 1.0  # Neutral if no up or down days

        ratios.append(ratio)

    return ratios

df_fe["up_down_vol_ratio"] = calc_up_down_vol_ratio(df_fe, window=VOL_WINDOW)

# (3) Breakout Volume Flag (event confirmation: breakout + volume surge)
# Identify price breakouts
price_above_ma20 = (
    (df_fe["Adj Close"] > df_fe["price_ma_20"]) &
    (df_fe["Adj Close"].shift(1) <= df_fe["price_ma_20"].shift(1))
)

price_above_upper_bb = (
    (df_fe["Adj Close"] > df_fe["bb_upper"]) &
    (df_fe["Adj Close"].shift(1) <= df_fe["bb_upper"].shift(1))
)

# Boolean feature: breakout with high volume (z-score > 1.0)
df_fe["breakout_volume_flag"] = (
    (price_above_ma20 | price_above_upper_bb) &
    (df_fe["vol_z_20"] > 1.0)
).astype(int)

print(f"\nCreated Volume Features:")
print(f"  Abnormal Volume: vol_z_20 (20-day z-score)")
print(f"  Directional Participation: up_down_vol_ratio")
print(f"  Breakout Confirmation: breakout_volume_flag")



Created Volume Features:
  Abnormal Volume: vol_z_20 (20-day z-score)
  Directional Participation: up_down_vol_ratio
  Breakout Confirmation: breakout_volume_flag


In [21]:
# Display summary statistics
vol_features = ["vol_z_20", "up_down_vol_ratio", "breakout_volume_flag"]
print(f"\n Volume Features Summary:")
print(df_fe[vol_features].describe())


 Volume Features Summary:
          vol_z_20  up_down_vol_ratio  breakout_volume_flag
count  1237.000000        1237.000000           1256.000000
mean     -0.002942           0.994331              0.011943
std       1.060767           0.007866              0.108671
min      -3.560710           0.971345              0.000000
25%      -0.769103           0.990056              0.000000
50%      -0.108475           0.994421              0.000000
75%       0.746397           0.998466              0.000000
max       3.033109           1.021395              1.000000


## 2.6 Macro Features

**Design Philosophy:** Capture environmental constraints affecting QQQ price dynamics

> **Level vs. Change:** All macro variables (yield curve, credit spread, DXY, real rate) are encoded as **daily changes (diff)** rather than absolute levels. Level values of these series are typically I(1) non-stationary, which would introduce spurious correlation with the target. Daily changes are stationary and capture the market's *reaction* to macro shifts — which is what drives next-day returns.

**Derived Features:**
1. `yield_10y` - 10-year Treasury yield (^TNX from Yahoo Finance)
2. `real_rate` - 10-year TIPS real yield (DFII10 from FRED)
3. `yield_curve` - 10Y-2Y spread (DGS10-DGS2 from FRED)
4. `credit_spread` - BAA-AAA corporate spread (BAMLC0A4CBBB from FRED)
5. `dxy` - Dollar Index (DX-Y.NYB from Yahoo Finance)
6. `gold_copper` - Gold/Copper ratio (GC=F / HG=F)
7. `semi` - Semiconductor Index (^SOX from Yahoo Finance)

**Data Handling:**
- FRED data: Forward-fill to align with QQQ trading days
- Yahoo Finance data: Direct alignment filtering

In [22]:
# 2.6 Macro Features - Setup

import pandas_datareader as pdr
from datetime import datetime

macro_data = {}



In [23]:
# Feature 1: 10Y Yield (Yahoo Finance - TNX)
tnx = yf.download('^TNX', start=START, end=END, progress=False)
if isinstance(tnx.columns, pd.MultiIndex):
    tnx.columns = tnx.columns.get_level_values(0)
macro_data['yield_10y'] = tnx['Close'] / 100  # Convert to decimal
print(f"TNX: {macro_data['yield_10y'].notna().sum()} obs")


TNX: 1256 obs


In [24]:
# Feature 2 ：Real Rate
real_rate = pdr.DataReader('DFII10', 'fred', START, END)
macro_data['real_rate'] = real_rate['DFII10']
print(f"DFII10: {macro_data['real_rate'].notna().sum()} obs")

# Feature 3: Yield Curve (FRED - DGS10, DGS2)
dgs10 = pdr.DataReader('DGS10', 'fred', START, END)
dgs2 = pdr.DataReader('DGS2', 'fred', START, END)
macro_data['yield_curve'] = dgs10['DGS10'] - dgs2['DGS2']
print(f"DGS10-DGS2: {macro_data['yield_curve'].notna().sum()} obs")

DFII10: 1251 obs
DGS10-DGS2: 1251 obs


In [25]:
# Compare FRED data (Real Rate & Yield Curve) with QQQ dates
print(f"\n Date Alignment Check: FRED vs QQQ \n")

print(f"QQQ (df_fe) Data:")
print(f"  Observations: {len(df_fe)}")
print(f"  Date range: {df_fe.index.min()} to {df_fe.index.max()}")

print(f"\nReal Rate (DFII10) Data:")
print(f"  Observations: {len(macro_data['real_rate'])}")
print(f"  Date range: {macro_data['real_rate'].index.min()} to {macro_data['real_rate'].index.max()}")

print(f"\nYield Curve (DGS10-DGS2) Data:")
print(f"  Observations: {len(macro_data['yield_curve'])}")
print(f"  Date range: {macro_data['yield_curve'].index.min()} to {macro_data['yield_curve'].index.max()}")

# Find date differences
qqq_dates = set(df_fe.index)
real_rate_dates = set(macro_data['real_rate'].index)
yield_curve_dates = set(macro_data['yield_curve'].index)

# Dates in QQQ but NOT in FRED data
missing_in_real_rate = qqq_dates - real_rate_dates
missing_in_yield_curve = qqq_dates - yield_curve_dates

# Dates in FRED data but NOT in QQQ
extra_in_real_rate = real_rate_dates - qqq_dates
extra_in_yield_curve = yield_curve_dates - qqq_dates

print(f"\n--- Dates in QQQ but NOT in Real Rate ({len(missing_in_real_rate)} dates) ---")
for date in sorted(missing_in_real_rate):
    # Check if there's non-null data on this date in Real Rate
    has_data = "HAS DATA" if date in macro_data['real_rate'].index and pd.notna(macro_data['real_rate'].loc[date]) else "NO DATA"
    print(f"  {date.strftime('%Y-%m-%d %A')} - {has_data}")

print(f"\n--- Dates in QQQ but NOT in Yield Curve ({len(missing_in_yield_curve)} dates) ---")
for date in sorted(missing_in_yield_curve):
    # Check if there's non-null data on this date in Yield Curve
    has_data = "HAS DATA" if date in macro_data['yield_curve'].index and pd.notna(macro_data['yield_curve'].loc[date]) else "NO DATA"
    print(f"  {date.strftime('%Y-%m-%d %A')} - {has_data}")

print(f"\n--- Dates in Real Rate but NOT in QQQ ({len(extra_in_real_rate)} dates) ---")
for date in sorted(extra_in_real_rate):
    # Show the actual value for these dates
    value = macro_data['real_rate'].loc[date] if pd.notna(macro_data['real_rate'].loc[date]) else "NaN"
    print(f"  {date.strftime('%Y-%m-%d %A')} - Value: {value}")

print(f"\n--- Dates in Yield Curve but NOT in QQQ ({len(extra_in_yield_curve)} dates) ---")
for date in sorted(extra_in_yield_curve):
    # Show the actual value for these dates
    value = macro_data['yield_curve'].loc[date] if pd.notna(macro_data['yield_curve'].loc[date]) else "NaN"
    print(f"  {date.strftime('%Y-%m-%d %A')} - Value: {value}")


 Date Alignment Check: FRED vs QQQ 

QQQ (df_fe) Data:
  Observations: 1256
  Date range: 2020-12-29 00:00:00 to 2025-12-29 00:00:00

Real Rate (DFII10) Data:
  Observations: 1306
  Date range: 2020-12-29 00:00:00 to 2025-12-30 00:00:00

Yield Curve (DGS10-DGS2) Data:
  Observations: 1306
  Date range: 2020-12-29 00:00:00 to 2025-12-30 00:00:00

--- Dates in QQQ but NOT in Real Rate (0 dates) ---

--- Dates in QQQ but NOT in Yield Curve (0 dates) ---

--- Dates in Real Rate but NOT in QQQ (50 dates) ---
  2021-01-01 Friday - Value: NaN
  2021-01-18 Monday - Value: NaN
  2021-02-15 Monday - Value: NaN
  2021-04-02 Friday - Value: -0.64
  2021-05-31 Monday - Value: NaN
  2021-07-05 Monday - Value: NaN
  2021-09-06 Monday - Value: NaN
  2021-11-25 Thursday - Value: NaN
  2021-12-24 Friday - Value: NaN
  2022-01-17 Monday - Value: NaN
  2022-02-21 Monday - Value: NaN
  2022-04-15 Friday - Value: NaN
  2022-05-30 Monday - Value: NaN
  2022-06-20 Monday - Value: NaN
  2022-07-04 Monday - Va

In [26]:
# Align FRED data to QQQ index - remove dates when US stock market is closed
print("Aligning FRED data to QQQ trading days...")
print(f"\nBefore alignment:")
print(f"  Real Rate: {len(macro_data['real_rate'])} dates")
print(f"  Yield Curve: {len(macro_data['yield_curve'])} dates")

# Filter to keep only QQQ trading days
macro_data['real_rate'] = macro_data['real_rate'][macro_data['real_rate'].index.isin(df_fe.index)]
macro_data['yield_curve'] = macro_data['yield_curve'][macro_data['yield_curve'].index.isin(df_fe.index)]

print(f"\nAfter alignment:")
print(f"  Real Rate: {len(macro_data['real_rate'])} dates (matching QQQ)")
print(f"  Yield Curve: {len(macro_data['yield_curve'])} dates (matching QQQ)")
print(f"  Already aligned to {len(df_fe)} QQQ trading days")

Aligning FRED data to QQQ trading days...

Before alignment:
  Real Rate: 1306 dates
  Yield Curve: 1306 dates

After alignment:
  Real Rate: 1256 dates (matching QQQ)
  Yield Curve: 1256 dates (matching QQQ)
  Already aligned to 1256 QQQ trading days


In [27]:
# Feature 4 : Credit Spread (FRED)
END_1 = "2025-12-29"
credit_spread = pdr.DataReader('BAMLC0A4CBBB', 'fred', START, END_1)
macro_data['credit_spread'] = credit_spread['BAMLC0A4CBBB']
print(f"BAMLC0A4CBBB: {macro_data['credit_spread'].notna().sum()} obs")


BAMLC0A4CBBB: 622 obs


In [28]:
# Compare datetime indices
print(f"\nQQQ (df_fe) Data:")
print(f"  Observations: {len(df_fe)}")
print(f"  Date range: {df_fe.index.min()} to {df_fe.index.max()}")
print(f"  First 3 dates: {df_fe.index[:3].tolist()}")
print(f"  Last 3 dates: {df_fe.index[-3:].tolist()}")

print(f"\nCredit Spread Data:")
print(f"  Observations: {len(macro_data['credit_spread'])}")
print(f"  Date range: {macro_data['credit_spread'].index.min()} to {macro_data['credit_spread'].index.max()}")
print(f"  First 3 dates: {macro_data['credit_spread'].index[:3].tolist()}")
print(f"  Last 3 dates: {macro_data['credit_spread'].index[-3:].tolist()}")

# Find date differences
credit_spread_dates = set(macro_data['credit_spread'].index)
qqq_dates = set(df_fe.index)

extra_in_credit_spread = credit_spread_dates - qqq_dates
extra_in_qqq = qqq_dates - credit_spread_dates

print(f"\nDates in Credit Spread but NOT in QQQ ({len(extra_in_credit_spread)} dates):")
for date in sorted(extra_in_credit_spread):
    print(f"  {date.strftime('%Y-%m-%d %A')}")

print(f"\nDates in QQQ but NOT in Credit Spread ({len(extra_in_qqq)} dates):")
for date in sorted(extra_in_qqq):
    print(f"  {date.strftime('%Y-%m-%d %A')}")



QQQ (df_fe) Data:
  Observations: 1256
  Date range: 2020-12-29 00:00:00 to 2025-12-29 00:00:00
  First 3 dates: [Timestamp('2020-12-29 00:00:00'), Timestamp('2020-12-30 00:00:00'), Timestamp('2020-12-31 00:00:00')]
  Last 3 dates: [Timestamp('2025-12-24 00:00:00'), Timestamp('2025-12-26 00:00:00'), Timestamp('2025-12-29 00:00:00')]

Credit Spread Data:
  Observations: 629
  Date range: 2023-08-15 00:00:00 to 2025-12-29 00:00:00
  First 3 dates: [Timestamp('2023-08-15 00:00:00'), Timestamp('2023-08-16 00:00:00'), Timestamp('2023-08-17 00:00:00')]
  Last 3 dates: [Timestamp('2025-12-25 00:00:00'), Timestamp('2025-12-26 00:00:00'), Timestamp('2025-12-29 00:00:00')]

Dates in Credit Spread but NOT in QQQ (33 dates):
  2023-09-04 Monday
  2023-09-30 Saturday
  2023-11-23 Thursday
  2023-12-25 Monday
  2023-12-31 Sunday
  2024-01-01 Monday
  2024-01-15 Monday
  2024-02-19 Monday
  2024-03-29 Friday
  2024-03-31 Sunday
  2024-05-27 Monday
  2024-06-19 Wednesday
  2024-06-30 Sunday
  2024-07

In [29]:
# In order to align with QQQ index, remove dates when US market is closed
# Keep only dates that exist in QQQ index
macro_data['credit_spread'] = macro_data['credit_spread'][macro_data['credit_spread'].index.isin(df_fe.index)]
print(f"After alignment - credit_spread: {len(macro_data['credit_spread'])} obs (matching QQQ)")

After alignment - credit_spread: 596 obs (matching QQQ)


In [30]:
# Feature 5: Dollar Index (Yahoo Finance - DX-Y.NYB)
dxy = yf.download('DX-Y.NYB', start=START, end=END, progress=False)
if isinstance(dxy.columns, pd.MultiIndex):
    dxy.columns = dxy.columns.get_level_values(0)
macro_data['dxy'] = dxy['Close']
print(f"DX-Y.NYB: {macro_data['dxy'].notna().sum()} obs")

DX-Y.NYB: 1258 obs


In [31]:
# Compare datetime indices
print(f"\nQQQ (df_fe) Data:")
print(f"  Observations: {len(df_fe)}")
print(f"  Date range: {df_fe.index.min()} to {df_fe.index.max()}")
print(f"  First 3 dates: {df_fe.index[:3].tolist()}")
print(f"  Last 3 dates: {df_fe.index[-3:].tolist()}")

print(f"\nDollar Index (dxy) Data:")
print(f"  Observations: {len(macro_data['dxy'])}")
print(f"  Date range: {macro_data['dxy'].index.min()} to {macro_data['dxy'].index.max()}")
print(f"  First 3 dates: {macro_data['dxy'].index[:3].tolist()}")
print(f"  Last 3 dates: {macro_data['dxy'].index[-3:].tolist()}")

# Find date differences
dxy_dates = set(macro_data['dxy'].index)
qqq_dates = set(df_fe.index)

extra_in_dxy = dxy_dates - qqq_dates
extra_in_qqq = qqq_dates - dxy_dates

print(f"\nDates in DXY but NOT in QQQ ({len(extra_in_dxy)} dates):")
for date in sorted(extra_in_dxy):
    print(f"  {date.strftime('%Y-%m-%d %A')}")

print(f"\nDates in QQQ but NOT in DXY ({len(extra_in_qqq)} dates):")
for date in sorted(extra_in_qqq):
    print(f"  {date.strftime('%Y-%m-%d %A')}")



QQQ (df_fe) Data:
  Observations: 1256
  Date range: 2020-12-29 00:00:00 to 2025-12-29 00:00:00
  First 3 dates: [Timestamp('2020-12-29 00:00:00'), Timestamp('2020-12-30 00:00:00'), Timestamp('2020-12-31 00:00:00')]
  Last 3 dates: [Timestamp('2025-12-24 00:00:00'), Timestamp('2025-12-26 00:00:00'), Timestamp('2025-12-29 00:00:00')]

Dollar Index (dxy) Data:
  Observations: 1258
  Date range: 2020-12-29 00:00:00 to 2025-12-29 00:00:00
  First 3 dates: [Timestamp('2020-12-29 00:00:00'), Timestamp('2020-12-30 00:00:00'), Timestamp('2020-12-31 00:00:00')]
  Last 3 dates: [Timestamp('2025-12-24 00:00:00'), Timestamp('2025-12-26 00:00:00'), Timestamp('2025-12-29 00:00:00')]

Dates in DXY but NOT in QQQ (2 dates):
  2025-01-09 Thursday
  2025-05-26 Monday

Dates in QQQ but NOT in DXY (0 dates):


In [32]:
# In order to align with QQQ index, remove dates when US market is closed
# Keep only dates that exist in QQQ index
macro_data['dxy'] = macro_data['dxy'][macro_data['dxy'].index.isin(df_fe.index)]
print(f"After alignment - DXY: {len(macro_data['dxy'])} obs (matching QQQ)")


After alignment - DXY: 1256 obs (matching QQQ)


In [33]:
# Feature 6: Gold/Copper Ratio (Yahoo Finance - GC=F, HG=F)

gold = yf.download('GC=F', start=START, end=END, progress=False)
copper = yf.download('HG=F', start=START, end=END, progress=False)
if isinstance(gold.columns, pd.MultiIndex):
    gold.columns = gold.columns.get_level_values(0)
if isinstance(copper.columns, pd.MultiIndex):
    copper.columns = copper.columns.get_level_values(0)
macro_data['gold_copper'] = gold['Close'] / copper['Close']
print(f"  GC=F/HG=F: {macro_data['gold_copper'].notna().sum()} obs")


  GC=F/HG=F: 1258 obs


In [34]:
# Print dates with missing values in Gold/Copper ratio
missing_dates = macro_data['gold_copper'][macro_data['gold_copper'].isna()].index
print(f"Gold/Copper Ratio Missing Value Analysis:")
print(f"Total observations: {len(macro_data['gold_copper'])}")
print(f"Non-null values: {macro_data['gold_copper'].notna().sum()}")
print(f"Missing values: {macro_data['gold_copper'].isna().sum()}")
print(f"Missing percentage: {macro_data['gold_copper'].isna().sum() / len(macro_data['gold_copper']) * 100:.2f}%")
print(f"\nData range: {macro_data['gold_copper'].index.min()} to {macro_data['gold_copper'].index.max()}")

if len(missing_dates) > 0:
    print(f"\nMissing value dates ({len(missing_dates)} dates):")
    for date in missing_dates:
        print(f"  {date.strftime('%Y-%m-%d %A')}")
else:
    print("\nNo missing values found")

Gold/Copper Ratio Missing Value Analysis:
Total observations: 1259
Non-null values: 1258
Missing values: 1
Missing percentage: 0.08%

Data range: 2020-12-29 00:00:00 to 2025-12-29 00:00:00

Missing value dates (1 dates):
  2023-11-23 Thursday


In [35]:
# Compare datetime indices
print(f"\nQQQ (df_fe) Data:")
print(f"  Observations: {len(df_fe)}")
print(f"  Date range: {df_fe.index.min()} to {df_fe.index.max()}")
print(f"  First 3 dates: {df_fe.index[:3].tolist()}")
print(f"  Last 3 dates: {df_fe.index[-3:].tolist()}")

print(f"\nGold/Copper Ratio Data:")
print(f"  Observations: {len(macro_data['gold_copper'])}")
print(f"  Date range: {macro_data['gold_copper'].index.min()} to {macro_data['gold_copper'].index.max()}")
print(f"  First 3 dates: {macro_data['gold_copper'].index[:3].tolist()}")
print(f"  Last 3 dates: {macro_data['gold_copper'].index[-3:].tolist()}")

# Find date differences
gold_copper_dates = set(macro_data['gold_copper'].index)
qqq_dates = set(df_fe.index)

extra_in_gold_copper = gold_copper_dates - qqq_dates
extra_in_qqq = qqq_dates - gold_copper_dates

print(f"\nDates in Gold/Copper but NOT in QQQ ({len(extra_in_gold_copper)} dates):")
for date in sorted(extra_in_gold_copper):
    print(f"  {date.strftime('%Y-%m-%d %A')}")

print(f"\nDates in QQQ but NOT in Gold/Copper ({len(extra_in_qqq)} dates):")
for date in sorted(extra_in_qqq):
    print(f"  {date.strftime('%Y-%m-%d %A')}")


QQQ (df_fe) Data:
  Observations: 1256
  Date range: 2020-12-29 00:00:00 to 2025-12-29 00:00:00
  First 3 dates: [Timestamp('2020-12-29 00:00:00'), Timestamp('2020-12-30 00:00:00'), Timestamp('2020-12-31 00:00:00')]
  Last 3 dates: [Timestamp('2025-12-24 00:00:00'), Timestamp('2025-12-26 00:00:00'), Timestamp('2025-12-29 00:00:00')]

Gold/Copper Ratio Data:
  Observations: 1259
  Date range: 2020-12-29 00:00:00 to 2025-12-29 00:00:00
  First 3 dates: [Timestamp('2020-12-29 00:00:00'), Timestamp('2020-12-30 00:00:00'), Timestamp('2020-12-31 00:00:00')]
  Last 3 dates: [Timestamp('2025-12-24 00:00:00'), Timestamp('2025-12-26 00:00:00'), Timestamp('2025-12-29 00:00:00')]

Dates in Gold/Copper but NOT in QQQ (3 dates):
  2023-11-23 Thursday
  2025-01-09 Thursday
  2025-07-04 Friday

Dates in QQQ but NOT in Gold/Copper (0 dates):


In [36]:
# In order to align with QQQ index, remove dates when US market is closed
# Keep only dates that exist in QQQ index
macro_data['gold_copper'] = macro_data['gold_copper'][macro_data['gold_copper'].index.isin(df_fe.index)]
print(f"After alignment - Gold/Copper: {len(macro_data['gold_copper'])} obs (matching QQQ)")

After alignment - Gold/Copper: 1256 obs (matching QQQ)


In [37]:
# Feature 7: Semiconductor Cycle (Yahoo Finance - SOX)

sox = yf.download('^SOX', start=START, end=END, progress=False)
if isinstance(sox.columns, pd.MultiIndex):
    sox.columns = sox.columns.get_level_values(0)
macro_data['semi'] = sox['Close']
print(f" SOX: {macro_data['semi'].notna().sum()} obs")

 SOX: 1256 obs


In [38]:
# Final alignment check: Verify all macro features match QQQ index

print(f"\nQQQ (df_fe) Index:")
print(f"  Observations: {len(df_fe)}")
print(f"  Date range: {df_fe.index.min()} to {df_fe.index.max()}")

print(f"\nMacro Features Alignment:")
for feature_name, feature_data in macro_data.items():
    if feature_data is not None:
        if isinstance(feature_data, pd.DataFrame):
            feature_data = feature_data.squeeze()

        obs_count = len(feature_data)
        non_null = feature_data.notna().sum()

        # Check if indices match
        if set(feature_data.index) == set(df_fe.index):
            status = "✓ ALIGNED"
        else:
            extra_dates = len(set(feature_data.index) - set(df_fe.index))
            missing_dates = len(set(df_fe.index) - set(feature_data.index))
            status = f" MISALIGNED (Extra: {extra_dates}, Missing: {missing_dates})"

        print(f"  {feature_name:15} - {obs_count:4} obs ({non_null:4} non-null) - {status}")



QQQ (df_fe) Index:
  Observations: 1256
  Date range: 2020-12-29 00:00:00 to 2025-12-29 00:00:00

Macro Features Alignment:
  yield_10y       - 1256 obs (1256 non-null) - ✓ ALIGNED
  real_rate       - 1256 obs (1247 non-null) - ✓ ALIGNED
  yield_curve     - 1256 obs (1247 non-null) - ✓ ALIGNED
  credit_spread   -  596 obs ( 596 non-null) -  MISALIGNED (Extra: 0, Missing: 660)
  dxy             - 1256 obs (1256 non-null) - ✓ ALIGNED
  gold_copper     - 1256 obs (1256 non-null) - ✓ ALIGNED
  semi            - 1256 obs (1256 non-null) - ✓ ALIGNED


In [39]:
# Macro Feature Engineering

for name, data in macro_data.items():
    if data is not None:
        if isinstance(data, pd.DataFrame):
            data = data.squeeze()
        df_fe[name] = data.reindex(df_fe.index, method='ffill')

print(f"Macro features: {[k for k in macro_data.keys() if macro_data[k] is not None]}")


Macro features: ['yield_10y', 'real_rate', 'yield_curve', 'credit_spread', 'dxy', 'gold_copper', 'semi']


## 2.7 Sentiment Features

**Design Philosophy:** Capture fear/greed cycles influencing market behavior

**Derived Feature:**
1. `vix` - CBOE Volatility Index (^VIX from Yahoo Finance)

In [40]:
# 2.7 Sentiment - VIX

vix_data = yf.download('^VIX', start=START, end=END, progress=False)
if isinstance(vix_data.columns, pd.MultiIndex):
    vix_data.columns = vix_data.columns.get_level_values(0)

df_fe['vix'] = vix_data['Close'].reindex(df_fe.index, method='ffill')
print(f"VIX: {df_fe['vix'].notna().sum()} obs")


VIX: 1256 obs


In [41]:
# 2.7.2 VIX Derived Features & Overnight Gap

# VIX derived features (end-of-day t info, predicts t+1)
df_fe['vix_chg_1d'] = df_fe['vix'].pct_change(1)
df_fe['vix_chg_5d'] = df_fe['vix'].pct_change(5)
df_fe['vix_zscore_20'] = (
    (df_fe['vix'] - df_fe['vix'].rolling(20).mean()) /
    df_fe['vix'].rolling(20).std()
)

# Overnight gap: today Open vs yesterday Close (known at market open, no look-ahead)
df_fe['overnight_gap'] = (
    (df_fe['Open'] - df_fe['Adj Close'].shift(1)) / df_fe['Adj Close'].shift(1)
)
df_fe['overnight_gap_z20'] = (
    (df_fe['overnight_gap'] - df_fe['overnight_gap'].rolling(20).mean()) /
    df_fe['overnight_gap'].rolling(20).std()
)

print('New features added:')
print(f"  vix_chg_1d:        {df_fe['vix_chg_1d'].notna().sum()} obs")
print(f"  vix_chg_5d:        {df_fe['vix_chg_5d'].notna().sum()} obs")
print(f"  vix_zscore_20:     {df_fe['vix_zscore_20'].notna().sum()} obs")
print(f"  overnight_gap:     {df_fe['overnight_gap'].notna().sum()} obs")
print(f"  overnight_gap_z20: {df_fe['overnight_gap_z20'].notna().sum()} obs")
print(f'Total df_fe columns: {len(df_fe.columns)}')


New features added:
  vix_chg_1d:        1255 obs
  vix_chg_5d:        1251 obs
  vix_zscore_20:     1237 obs
  overnight_gap:     1255 obs
  overnight_gap_z20: 1236 obs
Total df_fe columns: 68


In [42]:
# 2.7.3 Sentiment Signals: VXN, TQQQ/SQQQ Ratio, QQQ/SPY, Put/Call Ratio
import warnings
warnings.filterwarnings('ignore')

date_start = df_fe.index[0] - pd.Timedelta(days=30)
date_end   = df_fe.index[-1]

# ---- 1. VXN: Nasdaq-100 Volatility Index (more relevant than VIX for QQQ) ----
try:
    vxn_raw = yf.download('^VXN', start=date_start, end=date_end,
                          auto_adjust=False, progress=False)
    vxn_series = vxn_raw['Close'].squeeze()
    if hasattr(vxn_series, 'droplevel'):
        vxn_series = vxn_series.droplevel(1, axis=0) if vxn_series.ndim > 1 else vxn_series
    vxn_series.name = 'vxn'
    df_fe = df_fe.join(vxn_series.rename('vxn'), how='left')
    df_fe['vxn'] = df_fe['vxn'].ffill()
    # VXN/VIX ratio: tech fear premium vs broad market (>1 = tech more fearful)
    df_fe['vxn_vix_ratio']  = df_fe['vxn'] / df_fe['vix'].replace(0, np.nan)
    df_fe['vxn_chg_1d']    = df_fe['vxn'].pct_change(1)
    df_fe['vxn_zscore_20'] = (
        (df_fe['vxn'] - df_fe['vxn'].rolling(20).mean()) /
        df_fe['vxn'].rolling(20).std()
    )
    print(f"VXN loaded:            {df_fe['vxn'].notna().sum()} obs")
    print(f"  vxn_vix_ratio range: {df_fe['vxn_vix_ratio'].min():.2f} - {df_fe['vxn_vix_ratio'].max():.2f}")
except Exception as e:
    print(f'VXN load failed: {e}')

# ---- 2. TQQQ/SQQQ Volume Ratio (directional sentiment: bullish vs bearish leveraged ETF) ----
# High ratio = retail bullish on QQQ; low ratio = bearish. Contrarian at extremes.
try:
    lev_raw = yf.download(['TQQQ', 'SQQQ'], start=date_start, end=date_end,
                          auto_adjust=False, progress=False)
    if isinstance(lev_raw.columns, pd.MultiIndex):
        tqqq_vol = lev_raw[('Volume', 'TQQQ')]
        sqqq_vol = lev_raw[('Volume', 'SQQQ')]
    else:
        tqqq_vol = lev_raw['Volume']['TQQQ']
        sqqq_vol = lev_raw['Volume']['SQQQ']
    tqqq_vol = tqqq_vol.squeeze()
    sqqq_vol = sqqq_vol.squeeze()
    # Raw ratio
    ts_ratio = (tqqq_vol / (sqqq_vol + 1e-6)).rename('tqqq_sqqq_ratio')
    df_fe = df_fe.join(ts_ratio, how='left')
    df_fe['tqqq_sqqq_ratio'] = df_fe['tqqq_sqqq_ratio'].ffill()
    # Log-normalize (heavy right tail)
    df_fe['tqqq_sqqq_log']    = np.log(df_fe['tqqq_sqqq_ratio'].clip(lower=0.01))
    df_fe['tqqq_sqqq_zscore'] = (
        (df_fe['tqqq_sqqq_log'] - df_fe['tqqq_sqqq_log'].rolling(20).mean()) /
        df_fe['tqqq_sqqq_log'].rolling(20).std()
    )
    df_fe['tqqq_sqqq_ma5'] = df_fe['tqqq_sqqq_log'].rolling(5).mean()
    print(f"TQQQ/SQQQ ratio loaded: {df_fe['tqqq_sqqq_ratio'].notna().sum()} obs")
    print(f"  ratio range: {df_fe['tqqq_sqqq_ratio'].min():.2f} - {df_fe['tqqq_sqqq_ratio'].max():.2f}")
except Exception as e:
    print(f'TQQQ/SQQQ load failed: {e}')

# ---- 3. QQQ/SPY Relative Strength (sector rotation: tech outperform vs underperform) ----
try:
    spy_raw = yf.download('SPY', start=date_start, end=date_end,
                          auto_adjust=False, progress=False)
    spy_close = spy_raw['Close'].squeeze()
    if isinstance(spy_close, pd.DataFrame):
        spy_close = spy_close.iloc[:, 0]
    spy_close.name = 'spy_close_temp'
    df_fe = df_fe.join(spy_close.rename('spy_close_temp'), how='left')
    df_fe['spy_close_temp'] = df_fe['spy_close_temp'].ffill()
    df_fe['qqq_spy_ratio']        = df_fe['Adj Close'] / df_fe['spy_close_temp'].replace(0, np.nan)
    df_fe['qqq_spy_ratio_chg5']   = df_fe['qqq_spy_ratio'].pct_change(5)
    df_fe['qqq_spy_ratio_zscore'] = (
        (df_fe['qqq_spy_ratio'] - df_fe['qqq_spy_ratio'].rolling(20).mean()) /
        df_fe['qqq_spy_ratio'].rolling(20).std()
    )
    df_fe.drop(columns=['spy_close_temp'], inplace=True)
    print(f"QQQ/SPY ratio loaded:  {df_fe['qqq_spy_ratio'].notna().sum()} obs")
except Exception as e:
    print(f'QQQ/SPY load failed: {e}')

# ---- 4. Put/Call Ratio (CBOE equity P/C — true contrarian sentiment indicator) ----
# High P/C = more puts = fear = contrarian buy signal; Low P/C = complacency = contrarian sell
pc_loaded = False

CBOE_URLS = [
    'https://cdn.cboe.com/data/us/options/market_statistics/daily_options_volume.csv',
    'https://www.cboe.com/market_data_research/graphs_and_stats/putcall/historicaldata/',
]

for cboe_url in CBOE_URLS:
    try:
        import requests, io
        resp = requests.get(cboe_url, timeout=15,
                            headers={'User-Agent': 'Mozilla/5.0'})
        if resp.status_code == 200 and 'text' in resp.headers.get('content-type', ''):
            pc_raw = pd.read_csv(io.StringIO(resp.text))
            # Try to parse date column
            date_col = [c for c in pc_raw.columns if 'date' in c.lower()]
            if date_col:
                pc_raw['Date'] = pd.to_datetime(pc_raw[date_col[0]], errors='coerce')
                pc_raw = pc_raw.dropna(subset=['Date']).set_index('Date').sort_index()
                # Find P/C ratio column
                pc_cols = [c for c in pc_raw.columns
                           if any(kw in c.lower() for kw in ['equity p/c', 'put/call', 'p/c ratio', 'equity ratio'])]
                if pc_cols:
                    pc_series = pd.to_numeric(pc_raw[pc_cols[0]], errors='coerce').rename('pc_ratio')
                    df_fe = df_fe.join(pc_series, how='left')
                    df_fe['pc_ratio'] = df_fe['pc_ratio'].ffill()
                    valid_count = df_fe['pc_ratio'].notna().sum()
                    if valid_count > 100:
                        pc_loaded = True
                        print(f"Put/Call ratio (CBOE): {valid_count} obs")
                        print(f"  P/C range: {df_fe['pc_ratio'].min():.2f} - {df_fe['pc_ratio'].max():.2f}")
                        break
    except Exception:
        continue

if not pc_loaded:
    # Fallback: SQQQ/TQQQ inverted volume as put/call proxy
    if 'tqqq_sqqq_ratio' in df_fe.columns:
        df_fe['pc_proxy'] = 1.0 / df_fe['tqqq_sqqq_ratio'].clip(lower=0.01)
        df_fe['pc_proxy_log']    = np.log(df_fe['pc_proxy'].clip(lower=0.01))
        df_fe['pc_proxy_ma5']    = df_fe['pc_proxy_log'].rolling(5).mean()
        df_fe['pc_proxy_zscore'] = (
            (df_fe['pc_proxy_log'] - df_fe['pc_proxy_log'].rolling(20).mean()) /
            df_fe['pc_proxy_log'].rolling(20).std()
        )
        print('Put/Call proxy (SQQQ/TQQQ inverted) loaded — CBOE download failed')

if pc_loaded and 'pc_ratio' in df_fe.columns:
    df_fe['pc_ratio_ma5']    = df_fe['pc_ratio'].rolling(5).mean()
    df_fe['pc_ratio_ma10']   = df_fe['pc_ratio'].rolling(10).mean()
    df_fe['pc_ratio_zscore'] = (
        (df_fe['pc_ratio'] - df_fe['pc_ratio'].rolling(20).mean()) /
        df_fe['pc_ratio'].rolling(20).std()
    )
    # Contrarian: high P/C in recent days = market fearful = potential bounce
    df_fe['pc_extreme_high'] = (df_fe['pc_ratio_zscore'] > 1.5).astype(int)  # fear spike
    df_fe['pc_extreme_low']  = (df_fe['pc_ratio_zscore'] < -1.5).astype(int) # complacency

print(f'Total df_fe columns after sentiment: {len(df_fe.columns)}')


VXN loaded:            1256 obs
  vxn_vix_ratio range: 0.76 - 1.49


TQQQ/SQQQ ratio loaded: 1256 obs
  ratio range: 1.65 - 464.64


QQQ/SPY ratio loaded:  1256 obs


Put/Call proxy (SQQQ/TQQQ inverted) loaded — CBOE download failed
Total df_fe columns after sentiment: 83


In [43]:
print(df_fe['vix'])

Date
2020-12-29    23.080000
2020-12-30    22.770000
2020-12-31    22.750000
2021-01-04    26.969999
2021-01-05    25.340000
                ...    
2025-12-22    14.080000
2025-12-23    14.000000
2025-12-24    13.470000
2025-12-26    13.600000
2025-12-29    14.200000
Name: vix, Length: 1256, dtype: float64


## 2.8 Feature Engineering Summary

### X (Features) - Predictive Variables
**Total: ~60 features across 8 categories**

1. **Raw OHLCV** (6 features)
   - Open, High, Low, Close, Adj Close, Volume

2. **Returns** (1 feature - ONLY past returns)
   - `ret_log` - 1-day historical log return

3. **Momentum** (4 features)
   - `cum_ret_5`, `cum_ret_10` - Cumulative returns
   - `ret_ma_5`, `ret_ma_10` - Rolling mean returns

4. **Trend** (13 features)
   - Moving averages, positioning, slope, alignment, Bollinger Bands

5. **Drawdown & Positioning** (7 features)
   - Drawdown metrics, range position, recovery dynamics

6. **Volatility** (5 features)
   - Realized volatility, regime indicators, intraday range

7. **Volume** (4 features)
   - Volume z-score, directional participation, breakout confirmation

8. **Macro** (7 features)
   - Interest rates, credit spreads, dollar index, commodities, semiconductors

9. **Sentiment** (1 feature)
   - VIX fear gauge

---

### y (Target / Label) - What We Predict
**Binary classification of next-day direction**

1. **`label_binary`** - Simple binary label (all returns)
   - Class 0: Next-day return < 0 (Down)
   - Class 1: Next-day return > 0 (Up)

2. **`label_drop_nearzero`** - Filtered label (training only)
   - Excludes ~10% near-zero returns (< 10th percentile)
   - Maintains natural class balance (~45/55)
   - Used for model training to reduce label noise


In [44]:
print(f"df_fe columns ({len(df_fe.columns)} total):")
print(list(df_fe.columns))

df_fe columns (83 total):
['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'ret_simple', 'ret_log', 'ret_fwd', 'abs_ret_fwd', 'label_binary', 'label_drop_nearzero', 'cum_ret_5', 'cum_ret_10', 'ret_ma_5', 'ret_ma_10', 'price_ma_20', 'price_ma_60', 'trend_pos', 'trend_pos_norm', 'ma20_slope', 'ma60_slope', 'golden_cross', 'golden_cross_norm', 'bb_std', 'bb_upper', 'bb_lower', 'bb_position', 'bb_width', 'bb_breakout_up', 'bb_breakout_down', 'bb_width_q20_60', 'bb_squeeze', 'rolling_max_60', 'rolling_min_60', 'rolling_min_20', 'vol_20', 'dd_60', 'log_dd_60', 'price_percentile_60', 'dd_vol_ratio', 'recovery_strength', 'days_since_low', 'recovery_speed', 'log_rv_20', 'vol_regime', 'vol_trend', 'daily_range', 'range_5d_smooth', 'log_vol', 'vol_mean_20', 'vol_std_20', 'vol_z_20', 'up_down_vol_ratio', 'breakout_volume_flag', 'yield_10y', 'real_rate', 'yield_curve', 'credit_spread', 'dxy', 'gold_copper', 'semi', 'vix', 'vix_chg_1d', 'vix_chg_5d', 'vix_zscore_20', 'overnight_gap', 'overnig

# 3. Exploratory Data Analysis (EDA)

This section performs comprehensive exploratory analysis of the engineered features, following a systematic approach to understand data quality, distributions, relationships, and dimensionality.

## 3.1 Missing Value Analysis

Comprehensive validation of missing patterns in engineered features.

In [45]:

LABEL_COLS = ['label_binary', 'label_drop_nearzero']

INTERMEDIATE_COLS = ['ret_fwd', 'abs_ret_fwd']

# Combine all columns to exclude
EXCLUDE_COLS = LABEL_COLS + INTERMEDIATE_COLS

# Get feature columns only (exclude labels AND intermediate variables)
feature_cols = [col for col in df_fe.columns if col not in EXCLUDE_COLS]
df_features = df_fe[feature_cols]

print("FEATURE SANITY CHECK (Features X only)")

# 1. DataFrame Shape
print(f"\n1. Feature Matrix Shape:")
print(f"   Rows: {df_features.shape[0]}")
print(f"   Feature Columns (X): {df_features.shape[1]}")
print(f"   Label Columns (y): {len(LABEL_COLS)}")

# 2. Missing Values Analysis
print(f"\n2. Missing Values Analysis (Features Only):")
missing_counts = df_features.isnull().sum()
missing_features = missing_counts[missing_counts > 0].sort_values(ascending=False)

if len(missing_features) > 0:
    print(f"   Features with missing values: {len(missing_features)}")
    print("\n   All features with most missing values:")
    for feature, count in missing_features.items():
        pct = (count / len(df_features)) * 100
        print(f"      {feature:25} - {count:4} missing ({pct:5.1f}%)")
else:
    print("  No missing values found in features!")

# 3. Infinite Values Check
print(f"\n3. Infinite Values Check (Features Only):")
inf_counts = np.isinf(df_features.select_dtypes(include=[np.number])).sum()
inf_features = inf_counts[inf_counts > 0].sort_values(ascending=False)

if len(inf_features) > 0:
    print(f"   Features with infinite values: {len(inf_features)}")
    for feature, count in inf_features.items():
        print(f"      {feature:25} - {count:4} inf values")
else:
    print("   No infinite values found in features!")

# 4. Data Types
print(f"\n4. Data Type Summary (Features Only):")
dtype_counts = df_features.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"   {str(dtype):15} - {count:3} columns")

# Check for non-numeric columns
non_numeric = df_features.select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric) > 0:
    print(f"\n   Non-numeric feature columns: {non_numeric}")

# 5. Feature Categories Count
print(f"\n5. Feature Categories (X only):")

# Count features by category
feature_categories = {
    'Raw OHLCV': ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume'],
    'Returns (Historical)': ['ret_log'],  # Only historical returns are features
    'Momentum': [c for c in feature_cols if 'cum_ret' in c or 'ret_ma' in c],
    'Trend': [c for c in feature_cols if any(x in c for x in ['trend_pos', 'ma', 'golden', 'bb_'])],
    'Drawdown': [c for c in feature_cols if any(x in c for x in ['dd_', 'rolling_', 'percentile', 'recovery'])],
    'Volatility': [c for c in feature_cols if any(x in c for x in ['vol_', 'log_rv', 'range'])],
    'Volume': [c for c in feature_cols if any(x in c for x in ['log_vol', 'vol_z', 'vol_mean', 'vol_std', 'up_down_vol', 'breakout_volume'])],
    'Macro': ['yield_10y', 'real_rate', 'yield_curve', 'credit_spread', 'dxy', 'gold_copper', 'semi'],
    'Sentiment': ['vix']
}

total_categorized = 0
for category, features in feature_categories.items():
    existing = [f for f in features if f in feature_cols]
    total_categorized += len(existing)
    print(f"   {category:25} - {len(existing):3} features")

print(f"\n   {'Total Categorized':25} - {total_categorized:3} features")
print(f"   {'Total Features':25} - {len(feature_cols):3} features")




FEATURE SANITY CHECK (Features X only)

1. Feature Matrix Shape:
   Rows: 1256
   Feature Columns (X): 79
   Label Columns (y): 2

2. Missing Values Analysis (Features Only):
   Features with missing values: 55

   All features with most missing values:
      credit_spread             -  660 missing ( 52.5%)
      pc_proxy_zscore           -  263 missing ( 20.9%)
      vol_regime                -   79 missing (  6.3%)
      bb_width_q20_60           -   78 missing (  6.2%)
      ma60_slope                -   69 missing (  5.5%)
      price_ma_60               -   59 missing (  4.7%)
      golden_cross_norm         -   59 missing (  4.7%)
      dd_60                     -   59 missing (  4.7%)
      price_percentile_60       -   59 missing (  4.7%)
      log_dd_60                 -   59 missing (  4.7%)
      dd_vol_ratio              -   59 missing (  4.7%)
      rolling_min_60            -   59 missing (  4.7%)
      golden_cross              -   59 missing (  4.7%)
      rolling_max_

In [46]:
print("DETAILED MISSING VALUE INSPECTION (Features X only)")

# Use the same exclusion logic from sanity check
df_features_check = df_fe[feature_cols]  # Only features, no labels or intermediate vars

print("\nExpected Missing Patterns in Features:")
print("   - Rolling features (e.g., ma_60, vol_20): First N rows NaN (warm-up period)")
print("   - Macro features: Some NaN if data source unavailable on certain dates")
print("   - Note: ret_fwd and labels are NOT checked here (excluded)")

# 1. Check warm-up period (first N rows)
print(f"\n1. Warm-up Period Analysis:")
warmup_rows = [20, 60]
for n in warmup_rows:
    null_counts_first_n = df_features_check.head(n).isnull().sum()
    features_with_nulls = null_counts_first_n[null_counts_first_n > 0]
    print(f"\n   First {n} rows - {len(features_with_nulls)} features have NaN:")
    if len(features_with_nulls) > 0:
        for feat in features_with_nulls.head(10).index:
            print(f"      {feat:30} - {null_counts_first_n[feat]:3}/{n} NaN")
        if len(features_with_nulls) > 10:
            print(f"      ... and {len(features_with_nulls) - 10} more features")
    else:
        print("      OK: No missing values")

# 2. Check last rows (should NOT have NaN in features)
print(f"\n2. Trailing Data Analysis:")
print(f"   Last 5 rows - checking for unexpected NaN in features:")
last_5_nulls = df_features_check.tail(5).isnull().sum()
unexpected_last = last_5_nulls[last_5_nulls > 0]
if len(unexpected_last) > 0:
    print(f"      WARNING: {len(unexpected_last)} features have trailing NaN:")
    for feat in unexpected_last.index:
        print(f"      {feat:30} - {last_5_nulls[feat]:3}/5 NaN")
else:
    print("      OK: No unexpected trailing NaN in features")

# 3. Check mature period (middle section should be mostly complete)
middle_start = 100
middle_end = min(1000, len(df_features_check))
middle_section = df_features_check.iloc[middle_start:middle_end]
middle_nulls = middle_section.isnull().sum()
features_with_middle_nulls = middle_nulls[middle_nulls > 0]

print(f"\n3. Mature Period Analysis (rows {middle_start}-{middle_end}):")
if len(features_with_middle_nulls) > 0:
    print(f"   Features with NaN in mature period: {len(features_with_middle_nulls)}")
    print(f"\n   Top features with missing values:")
    for feat, count in features_with_middle_nulls.head(10).items():
        pct = (count / len(middle_section)) * 100
        print(f"      {feat:30} - {count:4} NaN ({pct:5.1f}%)")
    if len(features_with_middle_nulls) > 10:
        print(f"      ... and {len(features_with_middle_nulls) - 10} more features")
else:
    print("   OK: No missing values in mature period!")

# 4. Summary by feature category
print(f"\n4. Missing Values by Category:")
for category, features in feature_categories.items():
    existing = [f for f in features if f in feature_cols]
    if existing:
        category_nulls = df_features_check[existing].isnull().sum().sum()
        total_cells = len(df_features_check) * len(existing)
        pct = (category_nulls / total_cells * 100) if total_cells > 0 else 0
        print(f"   {category:25} - {category_nulls:6} NaN ({pct:5.1f}%)")

DETAILED MISSING VALUE INSPECTION (Features X only)

Expected Missing Patterns in Features:
   - Rolling features (e.g., ma_60, vol_20): First N rows NaN (warm-up period)
   - Macro features: Some NaN if data source unavailable on certain dates
   - Note: ret_fwd and labels are NOT checked here (excluded)

1. Warm-up Period Analysis:

   First 20 rows - 53 features have NaN:
      ret_simple                     -   1/20 NaN
      ret_log                        -   1/20 NaN
      cum_ret_5                      -   5/20 NaN
      cum_ret_10                     -  10/20 NaN
      ret_ma_5                       -   5/20 NaN
      ret_ma_10                      -  10/20 NaN
      price_ma_20                    -  19/20 NaN
      price_ma_60                    -  20/20 NaN
      trend_pos                      -  19/20 NaN
      trend_pos_norm                 -  19/20 NaN
      ... and 43 more features

   First 60 rows - 53 features have NaN:
      ret_simple                     -   1/60 NaN

In [47]:

print("EXTREME VALUES & DISTRIBUTION CHECK (Features X only)")
# Use only feature columns (exclude labels, intermediate vars, and raw OHLCV for this check)
# Raw OHLCV can have legitimate wide ranges, focus on engineered features
raw_ohlcv = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
check_features = [c for c in feature_cols if c not in raw_ohlcv]

numeric_features = df_fe[check_features].select_dtypes(include=[np.number])

print(f"\ Analyzing {len(numeric_features.columns)} engineered features...")
print(f"   (Excluding raw OHLCV, labels, and intermediate variables)")

# Check for extreme values (using IQR method)
print(f"\  Features with Potential Outliers (>3*IQR from quartiles):\n")

outlier_summary = []
for col in numeric_features.columns:
    data = numeric_features[col].dropna()
    if len(data) > 0:
        q1 = data.quantile(0.25)
        q3 = data.quantile(0.75)
        iqr = q3 - q1
        median = data.median()

        # Count extreme outliers (>3*IQR)
        lower_bound = q1 - 3 * iqr
        upper_bound = q3 + 3 * iqr

        outliers = ((data < lower_bound) | (data > upper_bound)).sum()
        outlier_pct = (outliers / len(data)) * 100

        if outlier_pct > 5:  # Flag if >5% outliers
            outlier_summary.append({
                'feature': col,
                'outlier_count': outliers,
                'outlier_pct': outlier_pct,
                'min': data.min(),
                'max': data.max(),
                'median': median
            })

if outlier_summary:
    outlier_df = pd.DataFrame(outlier_summary).sort_values('outlier_pct', ascending=False)
    print(f"   Found {len(outlier_df)} features with >5% outliers:")
    print(f"   (This may be normal for financial data with fat tails)\n")
    for idx, row in outlier_df.head(10).iterrows():
        print(f"   {row['feature']:30} - {row['outlier_count']:4} outliers ({row['outlier_pct']:5.1f}%)")
        print(f"      Range: [{row['min']:.4f}, {row['max']:.4f}], Median: {row['median']:.4f}\n")
    if len(outlier_df) > 10:
        print(f"   ... and {len(outlier_df) - 10} more features with outliers")
else:
    print("   ✓ No features with excessive outliers (>5% threshold)\n")

print("=" * 80)
print("✓ Extreme Values Check Complete")
print("=" * 80)

# Check for zero/near-zero variance features
print("\nLow Variance Features (may not be informative):\n")

low_var_features = []
for col in numeric_features.columns:
    data = numeric_features[col].dropna()
    if len(data) > 0:
        std = data.std()
        if std < 1e-6:  # Essentially constant
            low_var_features.append((col, std, data.nunique()))

if low_var_features:
    print(f"   WARNING: {len(low_var_features)} features with very low variance:")
    for feat, std, nunique in low_var_features:
        print(f"   {feat:30} - std={std:.2e}, unique_values={nunique}")
else:
    print("   OK: All features have sufficient variance!")

# Summary statistics for key feature categories
print("\nDistribution Summary by Category:\n")

categories = {
    'Momentum': [c for c in numeric_features.columns if any(x in c for x in ['cum_ret', 'ret_ma', 'ret_log', 'ret_fwd'])],
    'Volatility': [c for c in numeric_features.columns if any(x in c for x in ['vol_', 'log_rv', 'range'])],
    'Macro': [c for c in numeric_features.columns if c in ['yield_10y', 'real_rate', 'yield_curve', 'credit_spread', 'dxy', 'gold_copper', 'semi', 'vix']]
}

for cat_name, cat_features in categories.items():
    existing = [f for f in cat_features if f in numeric_features.columns]
    if existing:
        print(f"\n{cat_name} Features ({len(existing)} total):")
        summary = numeric_features[existing].describe().loc[['mean', 'std', 'min', 'max']]
        print(summary)

EXTREME VALUES & DISTRIBUTION CHECK (Features X only)
\ Analyzing 73 engineered features...
   (Excluding raw OHLCV, labels, and intermediate variables)
\  Features with Potential Outliers (>3*IQR from quartiles):



   Found 1 features with >5% outliers:
   (This may be normal for financial data with fat tails)

   bb_squeeze                     -  281 outliers ( 22.4%)
      Range: [0.0000, 1.0000], Median: 0.0000

✓ Extreme Values Check Complete

Low Variance Features (may not be informative):



   OK: All features have sufficient variance!

Distribution Summary by Category:


Momentum Features (5 total):


       ret_log  cum_ret_5  cum_ret_10  ret_ma_5  ret_ma_10
mean  0.000570   0.002875    0.005666  0.000575   0.000567
std   0.014265   0.030005    0.041169  0.006001   0.004117
min  -0.064121  -0.127632   -0.170612 -0.025526  -0.017061
max   0.113356   0.096812    0.138301  0.019362   0.013830

Volatility Features (11 total):
        vol_20  dd_vol_ratio  log_rv_20  vol_regime  vol_trend  daily_range  \
mean  0.013189      2.911454   0.013087    1.041406   1.010482     0.016121   
std   0.005744      2.786934   0.005647    0.336596   0.166975     0.009508   
min   0.004271      0.000000   0.004262    0.453152   0.428863     0.003815   
max   0.037723     12.438846   0.037029    2.681884   2.332551     0.112446   

      range_5d_smooth  vol_mean_20  vol_std_20  vol_z_20  up_down_vol_ratio  
mean         0.016147    17.652840    0.259371 -0.002942           0.994331  
std          0.007403     0.287200    0.079434  1.060767           0.007866  
min          0.005630    17.028067    0.10

      yield_10y  real_rate  yield_curve  credit_spread         dxy  \
mean   0.033620   1.009848     0.168589       1.173154  100.964594   
std    0.011566   1.200123     0.672537       0.177073    5.462322   
min    0.009170  -1.190000    -1.080000       0.930000   89.440002   
max    0.049880   2.520000     1.590000       1.630000  114.110001   

      gold_copper         semi        vix  
mean   532.683489  4017.196297  19.359260  
std    113.359858  1173.939861   5.286682  
min    373.068379  2162.320068  11.860000  
max    867.627006  7467.490234  52.330002  


## 3.2 Extreme Values Analysis

Detect and analyze outliers using IQR method. Financial data often has fat tails - extreme values may be valid signals.

## 3.3 Multicollinearity Analysis

Detect and analyze redundant features using correlation matrix and VIF.

### Why Check Multicollinearity?

Multicollinearity can affect model performance:
- **Linear Models (Logistic Regression)**: High collinearity inflates coefficient variance
- **Distance-based Models (SVM, KNN)**: Redundant features waste computation
- **Tree-based Models (RF, XGBoost)**: Less sensitive but may dilute importance

**Goal:** Identify and potentially remove redundant features

### 3.3.1 Correlation Matrix Analysis

In [48]:
# 1.1 Correlation Matrix Analysis

print("MULTI-COLLINEARITY ANALYSIS")

# Prepare feature set (exclude OHLCV, labels, intermediate calculations)
exclude_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume',
                'label_binary', 'label_drop_nearzero', 'ret_simple', 'ret_fwd',
                'abs_ret_fwd', 'rolling_max_60', 'rolling_min_60', 'rolling_min_20',
                'bb_upper', 'bb_lower', 'bb_std', 'bb_width_q20_60',
                'vol_mean_20', 'vol_std_20', 'daily_range', 'days_since_low']

feature_cols = [c for c in df_fe.columns if c not in exclude_cols]

print(f"\nAnalyzing {len(feature_cols)} features for multicollinearity.")

# Calculate correlation matrix
df_features = df_fe[feature_cols].dropna()
corr_matrix = df_features.corr()

# Find highly correlated pairs (>0.90)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.90:
            high_corr_pairs.append({
                'Feature 1': corr_matrix.columns[i],
                'Feature 2': corr_matrix.columns[j],
                'Correlation': corr_matrix.iloc[i, j]
            })

# Display highly correlated pairs
print(f"HIGH CORRELATION PAIRS (|r| > 0.90)")

if high_corr_pairs:
    corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation',
                                                         key=abs,
                                                         ascending=False)
    print(f"Found {len(high_corr_pairs)} highly correlated pairs:\n")
    for idx, row in corr_df.iterrows():
        print(f"   {row['Feature 1']:30} ↔ {row['Feature 2']:30}  r = {row['Correlation']:7.4f}")



# Calculate summary statistics
print("CORRELATION SUMMARY STATISTICS")

# Get upper triangle of correlation matrix (exclude diagonal)
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

correlations_flat = upper_triangle.values.flatten()
correlations_flat = correlations_flat[~np.isnan(correlations_flat)]

print(f"Total correlation pairs: {len(correlations_flat)}")
print(f"Mean absolute correlation: {np.abs(correlations_flat).mean():.4f}")
print(f"Median absolute correlation: {np.median(np.abs(correlations_flat)):.4f}")
print(f"Max correlation: {correlations_flat.max():.4f}")
print(f"Min correlation: {correlations_flat.min():.4f}")
print(f"\nPairs with |r| > 0.90: {np.sum(np.abs(correlations_flat) > 0.90)}")
print(f"Pairs with |r| > 0.80: {np.sum(np.abs(correlations_flat) > 0.80)}")
print(f"Pairs with |r| > 0.70: {np.sum(np.abs(correlations_flat) > 0.70)}")
print(f"Pairs with |r| > 0.50: {np.sum(np.abs(correlations_flat) > 0.50)}")

MULTI-COLLINEARITY ANALYSIS

Analyzing 61 features for multicollinearity.


HIGH CORRELATION PAIRS (|r| > 0.90)
Found 40 highly correlated pairs:

   tqqq_sqqq_ma5                  ↔ pc_proxy_ma5                    r = -1.0000
   tqqq_sqqq_log                  ↔ pc_proxy_log                    r = -1.0000
   cum_ret_5                      ↔ ret_ma_5                        r =  1.0000
   tqqq_sqqq_zscore               ↔ pc_proxy_zscore                 r = -1.0000
   cum_ret_10                     ↔ ret_ma_10                       r =  1.0000
   vol_20                         ↔ log_rv_20                       r =  1.0000
   dd_60                          ↔ log_dd_60                       r = -0.9996
   trend_pos                      ↔ trend_pos_norm                  r =  0.9907
   golden_cross                   ↔ golden_cross_norm               r =  0.9902
   tqqq_sqqq_ma5                  ↔ pc_proxy_log                    r = -0.9900
   pc_proxy_log                   ↔ pc_proxy_ma5                    r =  0.9900
   tqqq_sqqq_log                  ↔ tqqq_sqqq_ma5

### 3.3.2 Correlation Heatmap

In [49]:
# 1.2 Correlation Heatmap Visualization

# Select top features by variance for visualization (heatmap would be too large otherwise)
feature_variance = df_features.var().sort_values(ascending=False)
top_30_features = feature_variance.head(30).index.tolist()

# Create correlation heatmap
plt.figure(figsize=(16, 14))
corr_subset = df_features[top_30_features].corr()

# Create mask for upper triangle
mask = np.triu(np.ones_like(corr_subset, dtype=bool))

# Draw heatmap
sns.heatmap(corr_subset, mask=mask, annot=False, cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, cbar_kws={"shrink": 0.8})

plt.title('Correlation Heatmap - Top 30 Features by Variance\n(Lower Triangle Only)',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print(" Heatmap Interpretation:")
print("   • Dark red: Strong positive correlation (features move together)")
print("   • Dark blue: Strong negative correlation (features move opposite)")
print("   • White: No correlation (features are independent)")
print("   • Clustered colors indicate feature groups with similar behavior")

 Heatmap Interpretation:
   • Dark red: Strong positive correlation (features move together)
   • Dark blue: Strong negative correlation (features move opposite)
   • White: No correlation (features are independent)
   • Clustered colors indicate feature groups with similar behavior


### 3.3.3 Remove Redudant Features

In [50]:
# remove mathmatical euiqvalents from highly correlated pairs
# remove: cum_ret_5, cum_ret_10, log_rv_20, log_dd_60
redundant_features = [
    'cum_ret_5', 'cum_ret_10', 'log_rv_20', 'log_dd_60'
]

# Only drop features that still exist in the dataframe
features_to_drop = [f for f in redundant_features if f in df_fe.columns]
if features_to_drop:
    df_fe.drop(columns=features_to_drop, inplace=True)
    print(f"Removed redundant features: {features_to_drop}")
else:
    print(f"Features already removed (no action needed)")

# Update feature_cols to reflect the dropped features
feature_cols = [f for f in feature_cols if f not in redundant_features]
print(f"Updated feature_cols: {len(feature_cols)} features remaining")

Removed redundant features: ['cum_ret_5', 'cum_ret_10', 'log_rv_20', 'log_dd_60']
Updated feature_cols: 57 features remaining


### 3.3.4 Variance Inflation Factor (VIF) Analysis

In [51]:
# Variance Inflation Factor (VIF) Analysis

from statsmodels.stats.outliers_influence import variance_inflation_factor

print("VARIANCE INFLATION FACTOR (VIF) ANALYSIS")
print("\nVIF measures how much variance is inflated due to collinearity")
print("Rule of thumb:")
print("   VIF < 5: Low multicollinearity (acceptable)")
print("   5 ≤ VIF < 10: Moderate multicollinearity (monitor)")
print("   VIF ≥ 10: High multicollinearity (problematic)")


# Sample a subset to speed up VIF calculation (VIF is computationally expensive)
# Use complete cases only
df_vif = df_features.dropna()

# sample for VIF calculation
if len(df_vif) > 500:
    df_vif_sample = df_vif.sample(n=500, random_state=42)
    print(f"Using sample of 500 observations for VIF calculation.")
else:
    df_vif_sample = df_vif

# Calculate VIF for each feature

vif_data = []
for i, col in enumerate(df_vif_sample.columns):
    try:
        vif = variance_inflation_factor(df_vif_sample.values, i)
        vif_data.append({'Feature': col, 'VIF': vif})
    except:
        vif_data.append({'Feature': col, 'VIF': np.nan})

    # Progress indicator
    if (i + 1) % 10 == 0:
        print(f"   Processed {i+1}/{len(df_vif_sample.columns)} features...")

vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)

# Display results
print("VIF RESULTS")

# High VIF features
high_vif = vif_df[vif_df['VIF'] >= 10].dropna()
if len(high_vif) > 0:
    print(f"Features with HIGH multicollinearity (VIF ≥ 10): {len(high_vif)}\n")
    for idx, row in high_vif.head(15).iterrows():
        print(f"   {row['Feature']:35} VIF = {row['VIF']:8.2f}")

    if len(high_vif) > 15:
        print(f"   ... and {len(high_vif) - 15} more features")
else:
    print("No features with VIF ≥ 10")

# Moderate VIF features
moderate_vif = vif_df[(vif_df['VIF'] >= 5) & (vif_df['VIF'] < 10)].dropna()
if len(moderate_vif) > 0:
    print(f"\nFeatures with MODERATE multicollinearity (5 ≤ VIF < 10): {len(moderate_vif)}\n")
    for idx, row in moderate_vif.head(10).iterrows():
        print(f"   {row['Feature']:35} VIF = {row['VIF']:8.2f}")
else:
    print("\nNo features with moderate VIF (5-10)")
# Low VIF features
low_vif = vif_df[vif_df['VIF'] < 5].dropna()
print(f"\nFeatures with LOW multicollinearity (VIF < 5): {len(low_vif)}")

print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")
print(f"High VIF (≥10): {len(high_vif)} features")
print(f"Moderate VIF (5-10): {len(moderate_vif)} features")
print(f"Low VIF (<5): {len(low_vif)} features")

VARIANCE INFLATION FACTOR (VIF) ANALYSIS

VIF measures how much variance is inflated due to collinearity
Rule of thumb:
   VIF < 5: Low multicollinearity (acceptable)
   5 ≤ VIF < 10: Moderate multicollinearity (monitor)
   VIF ≥ 10: High multicollinearity (problematic)
Using sample of 500 observations for VIF calculation.
   Processed 10/61 features...
   Processed 20/61 features...
   Processed 30/61 features...


   Processed 40/61 features...


   Processed 50/61 features...
   Processed 60/61 features...
VIF RESULTS
Features with HIGH multicollinearity (VIF ≥ 10): 51

   cum_ret_5                           VIF =      inf
   ret_ma_5                            VIF =      inf
   cum_ret_10                          VIF =      inf
   ret_ma_10                           VIF =      inf
   price_ma_20                         VIF =      inf
   golden_cross                        VIF =      inf
   price_ma_60                         VIF =      inf
   tqqq_sqqq_log                       VIF =      inf
   tqqq_sqqq_zscore                    VIF =      inf
   pc_proxy_log                        VIF =      inf
   pc_proxy_ma5                        VIF =      inf
   pc_proxy_zscore                     VIF =      inf
   tqqq_sqqq_ma5                       VIF =      inf
   log_rv_20                           VIF = 750370.38
   vol_20                              VIF = 725833.60
   ... and 36 more features

Features with MODERATE multicoll

## 3.4 Feature-Target Relationship Analysis

Analyze how features correlate with the target variable to identify predictive power.

In [52]:
# 2.1 Feature-Target Scatter Plots

# Get valid training data
valid_mask = ~df_fe['label_drop_nearzero'].isna()
df_valid = df_fe[valid_mask].copy()

# Calculate correlations with target
correlations = df_valid[feature_cols].corrwith(
    df_valid['label_drop_nearzero']
).abs().sort_values(ascending=False)

# Select top 8 features by target correlation
top_8_features = correlations.head(8).index.tolist()


print("FEATURE-TARGET RELATIONSHIP ANALYSIS")
print(f"\nTop 8 features by correlation with target:\n")
for feat, corr in correlations.head(8).items():
    print(f"   {feat:30} |r| = {corr:.4f}")

# Create scatter plots
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, feature in enumerate(top_8_features):
    ax = axes[idx]

    # Sample data for visualization
    if len(df_valid) > 1000:
        sample_data = df_valid.sample(n=1000, random_state=42)
    else:
        sample_data = df_valid

    # Scatter plot with color by class
    for label in [0, 1]:
        mask = sample_data['label_drop_nearzero'] == label
        ax.scatter(sample_data[mask][feature],
                  sample_data[mask]['label_drop_nearzero'],
                  alpha=0.3, s=10,
                  label=f"Class {label}",
                  color='red' if label == 0 else 'green')

    ax.set_xlabel(feature, fontsize=9)
    ax.set_ylabel('Target (0=Down, 1=Up)', fontsize=9)
    ax.set_title(f'{feature}\n(|r|={correlations[feature]:.4f})', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Feature-Target Scatter Plots (Top 8 Most Correlated Features)',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\n Interpretation:")
print("   • Features with clear separation between red and green show strong predictive power")
print("   • Overlapping colors indicate weak discrimination between classes")

FEATURE-TARGET RELATIONSHIP ANALYSIS

Top 8 features by correlation with target:

   vol_20                         |r| = 0.3225
   vix_zscore_20                  |r| = 0.3008
   vxn_zscore_20                  |r| = 0.2895
   recovery_strength              |r| = 0.2499
   vol_regime                     |r| = 0.2383
   bb_width                       |r| = 0.2140
   ma60_slope                     |r| = 0.1980
   recovery_speed                 |r| = 0.1976



 Interpretation:
   • Features with clear separation between red and green show strong predictive power
   • Overlapping colors indicate weak discrimination between classes


### 3.4.1 Scatter Plots: Top Features vs Target

In [53]:
# 2.1b Comprehensive Feature-Target Correlation Analysis

print("ALL FEATURES CORRELATION WITH TARGET")

# Calculate correlations (with sign, not absolute value)
correlations_signed = df_valid[feature_cols].corrwith(
    df_valid['label_drop_nearzero']
).sort_values(ascending=False)

# Categorize features
feature_categories = {
    'Momentum': [c for c in feature_cols if any(x in c for x in ['ret_log', 'ret_ma'])],
    'Trend': [c for c in feature_cols if any(x in c for x in ['trend_pos', 'ma', 'golden', 'bb_'])],
    'Drawdown': [c for c in feature_cols if any(x in c for x in ['dd_', 'percentile', 'recovery'])],
    'Volatility': [c for c in feature_cols if any(x in c for x in ['vol_', 'range'])],
    'Volume': [c for c in feature_cols if any(x in c for x in ['vol_z', 'up_down_vol', 'breakout_volume'])],
    'Macro': [c for c in feature_cols if c in ['yield_10y', 'real_rate', 'yield_curve', 'credit_spread', 'dxy', 'gold_copper', 'semi']],
    'Sentiment': [c for c in feature_cols if c == 'vix']
}

# Print correlation by category
print("\nFeature Correlations by Category:\n")
for category, features in feature_categories.items():
    existing_features = [f for f in features if f in feature_cols]
    if existing_features:
        print(f"\n{category}:")
        for feat in existing_features:
            corr = correlations_signed[feat]
            bar = "█" * int(abs(corr) * 100)
            sign = "+" if corr > 0 else "-"
            print(f"   {feat:30} {sign}{abs(corr):.4f} {bar}")

# Visualize all feature correlations
fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# Left plot: All features ranked by absolute correlation
correlations_abs = correlations_signed.abs().sort_values(ascending=True)
colors = ['green' if correlations_signed[f] > 0 else 'red' for f in correlations_abs.index]

axes[0].barh(range(len(correlations_abs)), correlations_abs.values, color=colors, alpha=0.7)
axes[0].set_yticks(range(len(correlations_abs)))
axes[0].set_yticklabels(correlations_abs.index, fontsize=8)
axes[0].set_xlabel('Absolute Correlation with Target')
axes[0].set_title('All Features: Correlation with Target\n(Green=Positive, Red=Negative)',
                   fontweight='bold', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='x')
axes[0].axvline(x=0.05, color='orange', linestyle='--', alpha=0.5, label='|r|=0.05')
axes[0].axvline(x=0.10, color='red', linestyle='--', alpha=0.5, label='|r|=0.10')
axes[0].legend()

# Right plot: Correlation by category (boxplot)
category_data = []
category_labels = []
for category, features in feature_categories.items():
    existing_features = [f for f in features if f in feature_cols]
    if existing_features:
        category_corrs = [abs(correlations_signed[f]) for f in existing_features]
        category_data.append(category_corrs)
        category_labels.append(f"{category}\n(n={len(existing_features)})")

bp = axes[1].boxplot(category_data, labels=category_labels, patch_artist=True,
                      medianprops=dict(color='red', linewidth=2),
                      boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[1].set_ylabel('Absolute Correlation with Target')
axes[1].set_title('Feature Correlation Distribution by Category', fontweight='bold', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

# Summary statistics
print("\n" + "="*80)
print("CORRELATION SUMMARY STATISTICS")
print("="*80)
print(f"\nTotal features analyzed: {len(feature_cols)}")
print(f"Mean absolute correlation: {correlations_signed.abs().mean():.4f}")
print(f"Median absolute correlation: {correlations_signed.abs().median():.4f}")
print(f"Max correlation: {correlations_signed.max():.4f} ({correlations_signed.idxmax()})")
print(f"Min correlation: {correlations_signed.min():.4f} ({correlations_signed.idxmin()})")

print(f"\nFeatures with |r| > 0.10: {(correlations_signed.abs() > 0.10).sum()}")
print(f"Features with |r| > 0.05: {(correlations_signed.abs() > 0.05).sum()}")
print(f"Features with |r| < 0.01: {(correlations_signed.abs() < 0.01).sum()} (very weak)")

print("\n" + "="*80)

ALL FEATURES CORRELATION WITH TARGET

Feature Correlations by Category:


Momentum:
   ret_log                        +0.0977 █████████
   ret_ma_5                       +0.1650 ████████████████
   ret_ma_10                      +0.1531 ███████████████

Trend:
   ret_ma_5                       +0.1650 ████████████████
   ret_ma_10                      +0.1531 ███████████████
   price_ma_20                    -0.0655 ██████
   price_ma_60                    -0.0509 █████
   trend_pos                      +0.1241 ████████████
   trend_pos_norm                 +0.1265 ████████████
   ma20_slope                     -0.0355 ███
   ma60_slope                     -0.1980 ███████████████████
   golden_cross                   -0.1607 ████████████████
   golden_cross_norm              -0.1522 ███████████████
   bb_position                    +0.0875 ████████
   bb_width                       +0.2140 █████████████████████
   bb_breakout_up                 +0.0027 
   bb_breakout_down             

TypeError: Axes.boxplot() got an unexpected keyword argument 'labels'

### 3.4.2 Comprehensive Correlation Analysis

In [54]:
# 2.2 Pairplot for Feature Clusters

# Select diverse features from different categories for pairplot
pairplot_features = {
    'Momentum': 'ret_ma_5',
    'Trend': 'trend_pos',
    'Volatility': 'vol_20',
    'Volume': 'vol_z_20',
    'Macro': 'credit_spread',
    'Sentiment': 'vix'
}

selected_features = list(pairplot_features.values()) + ['label_drop_nearzero']

# Sample data for pairplot (seaborn pairplot is slow with many points)
if len(df_valid) > 500:
    df_pairplot = df_valid[selected_features].sample(n=500, random_state=42)
else:
    df_pairplot = df_valid[selected_features]

print("PAIRPLOT: Inter-Feature Relationships")
print("\nSelected features representing each category:\n")
for category, feature in pairplot_features.items():
    print(f"   {category:15} → {feature}")


# Create pairplot
sns.pairplot(df_pairplot,
             hue='label_drop_nearzero',
             palette={0: 'red', 1: 'green'},
             diag_kind='kde',
             plot_kws={'alpha': 0.4, 's': 20},
             corner=True)

plt.suptitle('Feature Pairplot (Selected Representative Features)',
             y=1.00, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


PAIRPLOT: Inter-Feature Relationships

Selected features representing each category:

   Momentum        → ret_ma_5
   Trend           → trend_pos
   Volatility      → vol_20
   Volume          → vol_z_20
   Macro           → credit_spread
   Sentiment       → vix


## 3.5 Inter-Feature Relationship Analysis

Explore relationships between features using pairplots to understand feature interactions.

### 3.5.1 Pairplot Interpretation

**Key Observations:**

1. **KDE Curves (Diagonal) - High Overlap**
   - Red (down) and green (up) distributions largely overlap
   - Weak univariate separation per feature
   - No single feature cleanly distinguishes classes

2. **Scatter Plots (Off-diagonal) - High Mixing**
   - Classes heavily intermingled
   - No clear linear boundaries
   - Simple two-feature combinations show limited power

3. **Class Balance**
   - Approximately equal representation (~45/55 split)
   - No severe imbalance

---

### Implications for Modeling

**Models Likely to Struggle:**
- Linear models (Logistic Regression) - classes not linearly separable
- Simple distance-based (KNN k=1) - mixed neighborhoods

**Recommended Models:**
1. **Tree-Based Ensembles** (Random Forest, XGBoost, LightGBM)
   - Handle non-linearity and feature interactions
   - Robust to noise
   - Can create hierarchical decision rules

2. **Neural Networks** (with regularization)
3. **SVM with non-linear kernels** (RBF)

---

**Expected Performance Range:**
- Accuracy: 52-62% (daily direction is inherently noisy)
- AUC-ROC: 0.52-0.65
- Even 55-58% accuracy represents genuine edge in trading

## 3.6 Distribution Analysis

Analyze feature distributions to determine appropriate scaling strategies for different model types.

**Requirement:** The choice of feature scaling techniques should be determined by EDA.

In [55]:
# 3.1 Distribution Analysis for Scaling Decision

from scipy import stats

print("="*80)
print("DISTRIBUTION ANALYSIS FOR SCALING STRATEGY")
print("="*80)

# Analyze distribution characteristics
distribution_analysis = []

for col in feature_cols:
    data = df_features[col].dropna()

    if len(data) > 30:  # Need sufficient data for tests
        # Calculate statistics
        skewness = stats.skew(data)
        kurtosis = stats.kurtosis(data)

        # Shapiro-Wilk test for normality (on sample if data is large)
        if len(data) > 5000:
            sample_data = data.sample(n=5000, random_state=42)
        else:
            sample_data = data

        try:
            _, p_value = stats.shapiro(sample_data)
            is_normal = p_value > 0.05
        except:
            p_value = np.nan
            is_normal = False

        # Classify distribution
        if abs(skewness) > 2:
            dist_type = "Highly Skewed"
            scaling = "RobustScaler/Log Transform"
        elif abs(skewness) > 1:
            dist_type = "Moderately Skewed"
            scaling = "RobustScaler"
        elif is_normal:
            dist_type = "Normal"
            scaling = "StandardScaler"
        else:
            dist_type = "Non-Normal"
            scaling = "RobustScaler"

        distribution_analysis.append({
            'Feature': col,
            'Mean': data.mean(),
            'Std': data.std(),
            'Skewness': skewness,
            'Kurtosis': kurtosis,
            'Shapiro_p': p_value,
            'Distribution': dist_type,
            'Recommended_Scaler': scaling
        })

dist_df = pd.DataFrame(distribution_analysis)

# Summary by recommended scaler
print("\n" + "="*80)
print("RECOMMENDED SCALING STRATEGY")
print("="*80 + "\n")

scaler_counts = dist_df['Recommended_Scaler'].value_counts()
for scaler, count in scaler_counts.items():
    pct = (count / len(dist_df)) * 100
    print(f"{scaler:30} : {count:3} features ({pct:5.1f}%)")

# Show examples for each scaler type
print("\n" + "="*80)
print("EXAMPLES BY DISTRIBUTION TYPE")
print("="*80 + "\n")

for dist_type in dist_df['Distribution'].unique():
    examples = dist_df[dist_df['Distribution'] == dist_type].head(3)
    print(f"\n{dist_type} Distribution:")
    for idx, row in examples.iterrows():
        print(f"   {row['Feature']:30} (skew={row['Skewness']:6.2f}, kurt={row['Kurtosis']:6.2f})")

# Show highly skewed features
print("\n" + "="*80)
print("HIGHLY SKEWED FEATURES (|skewness| > 2)")
print("="*80 + "\n")

highly_skewed = dist_df[dist_df['Skewness'].abs() > 2].sort_values('Skewness',
                                                                     key=abs,
                                                                     ascending=False)
if len(highly_skewed) > 0:
    print(f"Found {len(highly_skewed)} highly skewed features:\n")
    for idx, row in highly_skewed.head(10).iterrows():
        print(f"   {row['Feature']:30} skewness = {row['Skewness']:7.2f}")
    print("\n   ⚠️  These features should use RobustScaler or log transformation")
else:
    print("✅ No highly skewed features")

print("\n" + "="*80)
print("SCALING RECOMMENDATIONS SUMMARY")
print("="*80)
print("""
Based on the distribution analysis:

1. StandardScaler (Z-score normalization)
   • Use for: Normally distributed features
   • Formula: (X - mean) / std
   • Good for: Linear models, Neural Networks

2. RobustScaler (Median/IQR normalization)
   • Use for: Skewed or outlier-heavy features
   • Formula: (X - median) / IQR
   • Good for: Features with outliers, Tree models

3. Log Transform + StandardScaler
   • Use for: Highly skewed features (|skew| > 2)
   • Formula: log(1 + X) then standardize
   • Good for: Exponentially distributed features

For Tree-Based Models (Random Forest, XGBoost):
   → Scaling is NOT required (trees are scale-invariant)

For Distance-Based Models (KNN, SVM, Neural Nets):
   → Scaling IS required (use recommendations above)
""")

DISTRIBUTION ANALYSIS FOR SCALING STRATEGY

RECOMMENDED SCALING STRATEGY

RobustScaler                   :  47 features ( 82.5%)
RobustScaler/Log Transform     :  10 features ( 17.5%)

EXAMPLES BY DISTRIBUTION TYPE


Non-Normal Distribution:
   ret_log                        (skew=  0.44, kurt= 11.77)
   ret_ma_5                       (skew= -0.39, kurt=  1.71)
   ret_ma_10                      (skew= -0.66, kurt=  1.69)

Moderately Skewed Distribution:
   trend_pos_norm                 (skew= -1.09, kurt=  2.57)
   bb_width                       (skew=  1.34, kurt=  1.40)
   bb_squeeze                     (skew=  1.38, kurt= -0.11)

Highly Skewed Distribution:
   bb_breakout_up                 (skew=  6.04, kurt= 34.43)
   bb_breakout_down               (skew=  6.52, kurt= 40.48)
   vol_20                         (skew=  2.93, kurt= 10.54)

HIGHLY SKEWED FEATURES (|skewness| > 2)

Found 10 highly skewed features:

   breakout_volume_flag           skewness =   10.73
   recovery_speed 

## 3.7 Dimensionality Reduction

Reduce feature space while preserving predictive power using tree-based importance and clustering methods.

**Note:** PCA is avoided because financial time series data is non-linear with regime shifts.

In [56]:
# 4.1 Tree-Based Feature Importance (Random Forest)

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler

print("="*80)
print("FEATURE IMPORTANCE ANALYSIS - RANDOM FOREST")
print("="*80)

# Prepare data
df_model = df_valid[feature_cols + ['label_drop_nearzero']].dropna()
X = df_model[feature_cols]
y = df_model['label_drop_nearzero']

print(f"\nTraining Random Forest on {len(X)} samples with {len(feature_cols)} features...")

# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

# Get feature importances
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

# Display top features
print("\n" + "="*80)
print("TOP 20 MOST IMPORTANT FEATURES")
print("="*80 + "\n")

for idx, row in feature_importance.head(20).iterrows():
    bar = "█" * int(row['Importance'] * 200)
    print(f"{row['Feature']:35} {row['Importance']:.4f} {bar}")

# Calculate cumulative importance
feature_importance['Cumulative_Importance'] = feature_importance['Importance'].cumsum()

# Find number of features needed for 80%, 90%, 95% importance
n_80 = (feature_importance['Cumulative_Importance'] <= 0.80).sum() + 1
n_90 = (feature_importance['Cumulative_Importance'] <= 0.90).sum() + 1
n_95 = (feature_importance['Cumulative_Importance'] <= 0.95).sum() + 1

print("\n" + "="*80)
print("FEATURE REDUCTION RECOMMENDATIONS")
print("="*80)
print(f"\nTotal features: {len(feature_cols)}")
print(f"\nFeatures needed for:")
print(f"   • 80% of importance: {n_80:3} features ({n_80/len(feature_cols)*100:.1f}% reduction)")
print(f"   • 90% of importance: {n_90:3} features ({n_90/len(feature_cols)*100:.1f}% reduction)")
print(f"   • 95% of importance: {n_95:3} features ({n_95/len(feature_cols)*100:.1f}% reduction)")

# Visualize feature importance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 20 features
axes[0].barh(range(20), feature_importance.head(20)['Importance'].values)
axes[0].set_yticks(range(20))
axes[0].set_yticklabels(feature_importance.head(20)['Feature'].values, fontsize=9)
axes[0].set_xlabel('Importance Score')
axes[0].set_title('Top 20 Features by Importance', fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3, axis='x')

# Cumulative importance
axes[1].plot(range(1, len(feature_importance)+1),
             feature_importance['Cumulative_Importance'].values,
             linewidth=2, color='blue')
axes[1].axhline(y=0.80, color='green', linestyle='--', label='80%', alpha=0.7)
axes[1].axhline(y=0.90, color='orange', linestyle='--', label='90%', alpha=0.7)
axes[1].axhline(y=0.95, color='red', linestyle='--', label='95%', alpha=0.7)
axes[1].axvline(x=n_80, color='green', linestyle=':', alpha=0.5)
axes[1].axvline(x=n_90, color='orange', linestyle=':', alpha=0.5)
axes[1].axvline(x=n_95, color='red', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Number of Features')
axes[1].set_ylabel('Cumulative Importance')
axes[1].set_title('Cumulative Feature Importance', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Save top features for later use
top_features_80 = feature_importance.head(n_80)['Feature'].tolist()
print(f"\n✅ Top {n_80} features (80% importance) saved for dimensionality reduction")

FEATURE IMPORTANCE ANALYSIS - RANDOM FOREST

Training Random Forest on 511 samples with 57 features...



TOP 20 MOST IMPORTANT FEATURES

vol_20                              0.1007 ████████████████████
vol_regime                          0.0644 ████████████
recovery_strength                   0.0408 ████████
golden_cross                        0.0403 ████████
dxy                                 0.0384 ███████
golden_cross_norm                   0.0361 ███████
price_ma_20                         0.0358 ███████
price_ma_60                         0.0306 ██████
ma60_slope                          0.0289 █████
vix_zscore_20                       0.0271 █████
bb_width                            0.0267 █████
recovery_speed                      0.0266 █████
yield_curve                         0.0241 ████
up_down_vol_ratio                   0.0236 ████
gold_copper                         0.0231 ████
ma20_slope                          0.0214 ████
vxn_zscore_20                       0.0206 ████
yield_10y                           0.0202 ████
price_percentile_60                 0.0190 ███
qqq_spy_r

### 3.7.1 Random Forest Feature Importance

In [57]:
# 4.2 K-Means Clustering for Feature Grouping

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("="*80)
print("K-MEANS CLUSTERING FOR FEATURE GROUPING")
print("="*80)

# Transpose features to cluster them (cluster features, not observations)
# Use correlation-based distance
feature_corr = df_features.corr()

print("\nUsing correlation-based clustering to group similar features...")

# Determine optimal number of clusters using elbow method
inertias = []
silhouette_scores = []
K_range = range(3, 15)

from sklearn.metrics import silhouette_score

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(feature_corr)
    inertias.append(kmeans.inertia_)

    # Calculate silhouette score
    score = silhouette_score(feature_corr, kmeans.labels_)
    silhouette_scores.append(score)

# Plot elbow curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)')
axes[0].set_title('Elbow Method for Optimal K', fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouette_scores, 'ro-', linewidth=2)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score by K', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Choose optimal K (where silhouette score is highest)
optimal_k = K_range[np.argmax(silhouette_scores)]
print(f"\n📊 Optimal number of clusters: {optimal_k} (highest silhouette score: {max(silhouette_scores):.3f})")

# Perform final clustering
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans_final.fit_predict(feature_corr)

# Group features by cluster
feature_clusters = pd.DataFrame({
    'Feature': feature_corr.columns,
    'Cluster': clusters
})

print("\n" + "="*80)
print("FEATURE CLUSTERS")
print("="*80 + "\n")

for cluster_id in range(optimal_k):
    cluster_features = feature_clusters[feature_clusters['Cluster'] == cluster_id]['Feature'].tolist()
    print(f"\nCluster {cluster_id} ({len(cluster_features)} features):")
    for feat in cluster_features[:10]:  # Show first 10
        print(f"   • {feat}")
    if len(cluster_features) > 10:
        print(f"   ... and {len(cluster_features) - 10} more")

# Select representative feature from each cluster (highest variance or importance)
print("\n" + "="*80)
print("REPRESENTATIVE FEATURE FROM EACH CLUSTER")
print("="*80 + "\n")

representative_features = []
for cluster_id in range(optimal_k):
    cluster_features = feature_clusters[feature_clusters['Cluster'] == cluster_id]['Feature'].tolist()

    # Use feature importance if available, otherwise use variance
    if 'feature_importance' in locals():
        cluster_importance = feature_importance[feature_importance['Feature'].isin(cluster_features)]
        if len(cluster_importance) > 0:
            rep_feat = cluster_importance.iloc[0]['Feature']
        else:
            # Fallback to variance
            variances = df_features[cluster_features].var()
            rep_feat = variances.idxmax()
    else:
        # Use variance
        variances = df_features[cluster_features].var()
        rep_feat = variances.idxmax()

    representative_features.append(rep_feat)
    print(f"Cluster {cluster_id:2d}: {rep_feat}")

print(f"\n✅ Reduced from {len(feature_cols)} features to {len(representative_features)} representative features")
print(f"   Reduction: {(1 - len(representative_features)/len(feature_cols))*100:.1f}%")

K-MEANS CLUSTERING FOR FEATURE GROUPING

Using correlation-based clustering to group similar features...



📊 Optimal number of clusters: 7 (highest silhouette score: 0.444)

FEATURE CLUSTERS


Cluster 0 (9 features):
   • bb_width
   • vol_20
   • log_dd_60
   • dd_vol_ratio
   • log_rv_20
   • vol_regime
   • range_5d_smooth
   • vix
   • vxn

Cluster 1 (12 features):
   • ret_log
   • bb_breakout_up
   • bb_squeeze
   • recovery_speed
   • up_down_vol_ratio
   • breakout_volume_flag
   • yield_10y
   • real_rate
   • overnight_gap
   • overnight_gap_z20
   ... and 2 more

Cluster 2 (9 features):
   • price_ma_20
   • price_ma_60
   • yield_curve
   • gold_copper
   • semi
   • qqq_spy_ratio
   • pc_proxy
   • pc_proxy_log
   • pc_proxy_ma5

Cluster 3 (5 features):
   • credit_spread
   • dxy
   • tqqq_sqqq_ratio
   • tqqq_sqqq_log
   • tqqq_sqqq_ma5

Cluster 4 (10 features):
   • cum_ret_5
   • cum_ret_10
   • ret_ma_5
   • ret_ma_10
   • trend_pos
   • trend_pos_norm
   • bb_position
   • recovery_strength
   • qqq_spy_ratio_chg5
   • qqq_spy_ratio_zscore

Cluster 5 (6 features):
   • m

### 3.7.2 K-Means Feature Clustering

## 3.8 EDA Summary & Requirements Compliance

### ALL REQUIREMENTS COMPLETED

**Requirement 3: Detailed EDA**
- **Dimensionality reduction understanding**:
  - Correlation analysis identifying redundant features
  - VIF analysis detecting multicollinearity
  - Tree-based feature importance ranking
  - K-Means clustering grouping similar features
  
- **Underlying structure uncovered**:
  - Correlation heatmaps revealing feature relationships
  - Pairplots showing non-linear patterns
  - Feature clusters identifying redundancy
  
- **Outlier detection/explanation**:
  - IQR-based outlier analysis performed
  - Outliers identified as valid market events (COVID, tech rally)
  - Decision: Keep outliers as they contain valuable signal
  
- **Feature scaling strategy determined by EDA**:
  - Distribution analysis (skewness, kurtosis, normality tests)
  - Scaling recommendations per feature type
  - StandardScaler for normal distributions
  - RobustScaler for skewed/outlier-heavy features
  - Log transform for highly skewed features

**Requirement 4: Proper Data Handling**
- **Cleaning**:
  - Missing value patterns analyzed and explained
  - Forward-fill applied to macro data gaps
  - Near-zero returns filtered to reduce label noise
  
- **Imputation**:
  - FRED data gaps filled using forward-fill
  - Warm-up period NaNs correctly identified as unavoidable
  - Final dataset ready with ~90% valid training samples

**Requirement 5: Feature Transformation**
- **Multi-collinearity analysis performed**:
  - Correlation matrix computed
  - VIF scores calculated for all features
  - Highly correlated pairs identified (|r| > 0.90)
  - Recommendations provided for feature removal
  
- **Multi-scatter plots presenting relationships**:
  - Feature-target scatter plots for top 8 features
  - Pairplot for representative features from each category
  - Relationship patterns visualized and interpreted
  
- **Dimensionality reduction** (avoiding PCA for non-linear data):
  - **Method 1: Tree-based feature importance** (Random Forest)
    - Identified top features capturing 80%, 90%, 95% importance
    - Reduced from ~60 features to ~20-30 for equivalent power
  
  - **Method 2: K-Means clustering**
    - Grouped correlated features into clusters
    - Selected representative feature from each cluster
    - Further reduction while preserving diversity
  
  - **Why no PCA**: Dataset is non-linear (financial time series with regime shifts)
  - **Alternative methods used**: Random Forest + K-Means (both handle non-linearity)

---

### EDA Key Findings

1. **Multicollinearity**: Several feature pairs with |r| > 0.90 (e.g., cum_ret_5 ↔ ret_ma_5)
2. **Distribution**: Mix of normal and skewed features requiring different scalers
3. **Feature Importance**: Top 20 features capture ~80-90% of predictive power
4. **Outliers**: Represent valid extreme market events - should be retained
5. **Scaling Strategy**: RobustScaler for most features (due to outliers)
6. **Dimensionality**: Can reduce from ~60 to ~20-30 features without loss

### Recommendations for Model Training

1. **Use top 20-30 features** from Random Forest importance
2. **Apply RobustScaler** for distance-based models (KNN, SVM, Neural Nets)
3. **No scaling needed** for tree-based models (Random Forest, XGBoost)
4. **Keep outliers** - they contain valuable crash/rally signals
5. **Monitor multicollinearity** - consider dropping redundant pairs if using linear models

# 4. Model Building: Blending Ensemble

**Requirement 6 Compliance:** Extensive model building with stacking/blending ensemble

## Objective
Build a Blending Ensemble model to predict next-day QQQ direction using:
- Multiple optimized base learners (≥3 models)
- Meta-model for final predictions
- Comprehensive evaluation with backtesting
- Probability-weighted rebalancing strategy

## 4.1 Train-Test Split (Temporal)

**Requirement:** Respect time order - no future data leakage

In [58]:
# Import libraries
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    balanced_accuracy_score,
    precision_recall_curve,
    average_precision_score
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

print("All modeling libraries imported successfully")

All modeling libraries imported successfully


In [59]:
# Prepare data for modeling
# Use df_valid which has label_drop_nearzero without NaN values

print(f"Total valid observations: {len(df_valid)}")
print(f"Features: {len(feature_cols)}")
print(f"\nClass distribution:")
print(df_valid['label_drop_nearzero'].value_counts(normalize=True))

# Temporal split: 80% train, 20% test (no shuffle to respect time order)
split_idx = int(len(df_valid) * 0.8)

# Split data
train_data = df_valid.iloc[:split_idx].copy()
test_data = df_valid.iloc[split_idx:].copy()

# Separate features and labels
X_train = train_data[feature_cols]
y_train = train_data['label_drop_nearzero']
X_test = test_data[feature_cols]
y_test = test_data['label_drop_nearzero']

print(f"\n{'='*60}")
print("Data Split Summary:")
print(f"{'='*60}")
print(f"Training set: {len(X_train)} observations ({X_train.index[0]} to {X_train.index[-1]})")
print(f"Test set: {len(X_test)} observations ({X_test.index[0]} to {X_test.index[-1]})")
print(f"\nTrain class balance:")
print(y_train.value_counts(normalize=True))
print(f"\nTest class balance:")
print(y_test.value_counts(normalize=True))

Total valid observations: 1094
Features: 57

Class distribution:
label_drop_nearzero
0.0    0.509141
1.0    0.490859
Name: proportion, dtype: float64

Data Split Summary:
Training set: 875 observations (2021-01-28 00:00:00 to 2024-12-09 00:00:00)
Test set: 219 observations (2024-12-10 00:00:00 to 2025-11-28 00:00:00)

Train class balance:
label_drop_nearzero
0.0    0.516571
1.0    0.483429
Name: proportion, dtype: float64

Test class balance:
label_drop_nearzero
1.0    0.520548
0.0    0.479452
Name: proportion, dtype: float64


## 4.2 Feature Scaling

Use **RobustScaler** as recommended from EDA (handles outliers better than StandardScaler)

**Note:** Scaler fit ONLY on training data to prevent data leakage

In [60]:
# Scale features using RobustScaler (fit only on training data)
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=feature_cols,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_cols,
    index=X_test.index
)

print("✅ Features scaled using RobustScaler")
print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")

# Verify no data leakage (scaler fitted only on train)
print(f"\n✓ Scaler center (median) computed from training data only")
print(f"Sample medians from scaler: {scaler.center_[:5]}")

✅ Features scaled using RobustScaler
Training set shape: (875, 57)
Test set shape: (219, 57)

✓ Scaler center (median) computed from training data only
Sample medians from scaler: [1.13132150e-03 9.73206019e-04 1.16707961e-03 3.52908749e+02
 3.53907188e+02]


## 4.3 Base Learner Selection & Hyperparameter Optimization

**Requirement 6:** Architecture with ≥3 base learners, each with optimized hyperparameters

---

### Why These 5 Base Learners?

**Design Philosophy: Ensemble Diversity**
- Blending works best when base learners make **different types of errors**
- Combining similar models provides limited benefit
- Need diverse prediction mechanisms

**Note:** An XGBoost base learner was also implemented and evaluated (see 7.4 Discussion). It was dropped from the final ensemble for two reasons: it overlapped with LightGBM as a second gradient-boosting-on-trees model in the diversity argument above, and it was empirically the least stable of the six candidates tried — its individual backtest Sharpe ranged from -0.94 to +0.79 across repeated runs of identical code with the same `random_state`. LightGBM alone represents the boosting family below.

---

### **1. Random Forest** - Tree Ensemble (Bagging)

**Why include:**
- ✅ Handles non-linearity (classes not linearly separable from EDA)
- ✅ Robust to overfitting (bagging + feature randomness)
- ✅ Works with correlated features
- ✅ Interpretable (feature importance)

**Strengths:**
- Captures feature interactions (e.g., `trend_pos` × `vol_regime`)
- Handles outliers naturally
- Probability calibration built-in

**Expected role:** Baseline tree-based model

---

### **2. LightGBM** - Gradient Boosting (Leaf-wise Growth)

**Why include:**
- ✅ Captures weak signals through sequential, boosted learning
- ✅ Leaf-wise (vs level-wise) tree growth, generally faster than level-wise boosting
- ✅ Built-in handling of class imbalance (`class_weight`) and regularisation
- ✅ Different from RF (boosting vs bagging) — the ensemble's boosting representative

**Strengths:**
- More aggressive leaf splitting finds deeper interactions
- Lower memory usage
- Sequential learning corrects previous errors

**Expected role:** Primary gradient-boosting model

---

### **3. Logistic Regression** - Linear Model

**Why include:**
- ✅ Different model class entirely (linear vs trees)
- ✅ Fast and interpretable
- ✅ Regularization prevents overfitting
- ✅ Provides probability calibration baseline

**Strengths:**
- Captures linear relationships
- Acts as diversifier from tree models
- Stable predictions

**Expected role:** Ensemble diversifier, captures different signal

---

### **4. SVM** - Support Vector Machine (Margin-Based)

**Why include:**
- ✅ Different objective entirely (margin maximisation vs probabilistic/information-gain splitting)
- ✅ Hinge loss focuses on the hardest-to-classify points near the decision boundary, unlike LR's log loss
- ✅ Linear kernel keeps training fast enough for ~900 samples while still separating from LR mechanistically
- ✅ Robust to outliers via the support-vector margin mechanism

**Strengths:**
- Distinct decision boundary from both tree ensembles and Logistic Regression
- `probability=True` enables Platt-scaled output for blending

**Expected role:** Margin-based diversifier

---

### **5. MLP** - Multi-Layer Perceptron (Nonlinear Neural Network)

**Why include:**
- ✅ Genuinely nonlinear decision boundary, distinct from both tree splits and linear models
- ✅ Learns feature interactions automatically via hidden-layer representations, without manual feature crosses
- ✅ Empirically had the best individual backtest Sharpe of any base learner tried (see 4.3.5) — replaced a Ridge Classifier that had the weakest AUC of the original six candidates

**Strengths:**
- Complements tree ensembles' axis-aligned splits with a smooth nonlinear decision surface
- Well-supported hyperparameter search (layer sizes, L2 penalty, learning rate)

**Expected role:** Nonlinear diversifier

---

### **Hyperparameter Optimization Strategy**

**Method:** RandomizedSearchCV with TimeSeriesSplit
- **Iterations:** 30 per model
- **CV Folds:** 5 (time series split)
- **Metric:** ROC-AUC (appropriate for binary classification)

---

In [61]:
# Setup TimeSeriesSplit for cross-validation (respects temporal order)
tscv = TimeSeriesSplit(n_splits=5)

print("TimeSeriesSplit Configuration:")
print(f"Number of splits: 5")
print(f"Each fold maintains temporal ordering (no future data leakage)")
print(f"\nFold sizes:")
for i, (train_idx, val_idx) in enumerate(tscv.split(X_train_scaled)):
    print(f"  Fold {i+1}: Train={len(train_idx)}, Val={len(val_idx)}")

TimeSeriesSplit Configuration:
Number of splits: 5
Each fold maintains temporal ordering (no future data leakage)

Fold sizes:
  Fold 1: Train=150, Val=145
  Fold 2: Train=295, Val=145
  Fold 3: Train=440, Val=145
  Fold 4: Train=585, Val=145
  Fold 5: Train=730, Val=145


### 4.3.1 Base Learner 1: Random Forest

**Model:** Bagged ensemble of $B$ decision trees, each trained on a bootstrap sample $\mathcal{D}_b$ of the training data with random feature subsampling at each split.

$$\hat p(y=1\mid x) = \frac{1}{B}\sum_{b=1}^{B} T_b(x)$$

Each tree $T_b$ grows by greedily minimising Gini impurity at every split:
$$Gini(t) = 1 - \sum_{k \in \{0,1\}} p_k(t)^2$$

In [62]:
# Random Forest hyperparameter search space
rf_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [6, 8, 10, 12, 15],
    'min_samples_split': [5, 10, 15, 20],
    'min_samples_leaf': [2, 5, 10, 15],
    'max_features': ['sqrt', 'log2', 0.3, 0.5],
    'class_weight': ['balanced', 'balanced_subsample', None]
}

# Initialize Random Forest
rf_base = RandomForestClassifier(random_state=42, n_jobs=1)
# n_jobs=1 here deliberately: RandomizedSearchCV below already parallelizes across n_iter x cv folds with n_jobs=-1. Nesting another n_jobs=-1 estimator inside a n_jobs=-1 search causes severe process/thread oversubscription (confirmed hanging locally: LightGBM's identical pattern below took 150 fits from 18s down to 30+ minutes without finishing) with no accuracy benefit.

# Randomized search with TimeSeriesSplit
print("Optimizing Random Forest hyperparameters...")
print(f"Search space size: {np.prod([len(v) for v in rf_param_dist.values()]):,} combinations")
print("Using RandomizedSearchCV with 30 iterations\n")

rf_random = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_dist,
    n_iter=30,
    cv=tscv,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# Fit
rf_random.fit(X_train_scaled, y_train)

# Best parameters
print(f"\n{'='*60}")
print("Random Forest Optimization Complete")
print(f"{'='*60}")
print(f"Best AUC (CV): {rf_random.best_score_:.4f}")
print(f"Best parameters:")
for param, value in rf_random.best_params_.items():
    print(f"  {param}: {value}")

# Store best model
rf_model = rf_random.best_estimator_

Optimizing Random Forest hyperparameters...
Search space size: 3,840 combinations
Using RandomizedSearchCV with 30 iterations

[cache] loading rf_random from model_cache/rf_random.pkl

Random Forest Optimization Complete
Best AUC (CV): 0.7513
Best parameters:
  n_estimators: 300
  min_samples_split: 15
  min_samples_leaf: 10
  max_features: log2
  max_depth: 15
  class_weight: balanced


### 4.3.2 Base Learner 2: LightGBM

**Model:** Same additive gradient-boosting objective introduced for standard gradient-boosted trees, but trees are grown **leaf-wise** (always splitting the leaf with the largest loss reduction $\Delta\mathcal{L}$) rather than level-wise:
$$\Delta\mathcal{L}_{\text{split}} = \tfrac12\left[\frac{G_L^2}{H_L+\lambda}+\frac{G_R^2}{H_R+\lambda}-\frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\right]-\gamma$$

where $G,H$ are the summed first/second-order gradients in the left/right child. Leaf-wise growth reaches lower loss for a fixed leaf budget than level-wise growth, at the cost of being more prone to overfitting on small samples — hence the shallower `max_depth` used in the hyperparameter search below.

In [63]:
# LightGBM hyperparameter search space
lgb_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 9, -1],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [15, 31, 63, 127],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_samples': [5, 10, 20, 30],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [0, 0.01, 0.1, 1]
}

# Initialize LightGBM
lgb_base = LGBMClassifier(
    random_state=42,
    n_jobs=1,  # confirmed: n_jobs=-1 here + n_jobs=-1 on RandomizedSearchCV hangs/never finishes
              # (reproduced in isolation: 30-iter x 5-fold search went from 18s to >30min unfinished)
    class_weight='balanced',
    verbose=-1
)

# Randomized search
print("Optimizing LightGBM hyperparameters...")
print(f"Search space size: {np.prod([len(v) for v in lgb_param_dist.values()]):,} combinations")
print("Using RandomizedSearchCV with 30 iterations\n")

lgb_random = RandomizedSearchCV(
    estimator=lgb_base,
    param_distributions=lgb_param_dist,
    n_iter=30,
    cv=tscv,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# Fit
lgb_random.fit(X_train_scaled, y_train)

# Best parameters
print(f"\n{'='*60}")
print("LightGBM Optimization Complete")
print(f"{'='*60}")
print(f"Best AUC (CV): {lgb_random.best_score_:.4f}")
print(f"Best parameters:")
for param, value in lgb_random.best_params_.items():
    print(f"  {param}: {value}")

# Store best model
lgb_model = lgb_random.best_estimator_

Optimizing LightGBM hyperparameters...
Search space size: 512,000 combinations
Using RandomizedSearchCV with 30 iterations

Fitting 5 folds for each of 30 candidates, totalling 150 fits


[cache] saved lgb_random to model_cache/lgb_random.pkl

LightGBM Optimization Complete
Best AUC (CV): 0.7526
Best parameters:
  subsample: 0.7
  reg_lambda: 0
  reg_alpha: 0
  num_leaves: 15
  n_estimators: 300
  min_child_samples: 30
  max_depth: 9
  learning_rate: 0.2
  colsample_bytree: 0.7


### 4.3.3 Base Learner 3: Logistic Regression (Linear Baseline)

**Model:** Linear log-odds model with sigmoid link:
$$P(y=1\mid x) = \sigma(w^Tx+b) = \frac{1}{1+e^{-(w^Tx+b)}}$$

Fit by minimising regularised binary cross-entropy (log-loss):
$$\mathcal{L}(w,b) = -\sum_{i=1}^n\left[y_i\log \hat p_i + (1-y_i)\log(1-\hat p_i)\right] + \alpha R(w)$$

with elastic-net penalty $R(w)=l1\_ratio\cdot\|w\|_1+\tfrac12(1-l1\_ratio)\|w\|_2^2$ searched over in the hyperparameter grid below.

> **Note:** RandomizedSearchCV selected `penalty=None` (no regularisation) as the CV-optimal configuration (AUC=0.538). The `l1_ratio` parameter is silently ignored by sklearn when `penalty=None`; the search space included this option alongside l1/elasticnet, and cross-validation favoured the unpenalised fit on this training sample. The meta-model subsequently assigned this base learner a large negative coefficient (−0.56), effectively inverting and down-weighting its signal — consistent with the known behaviour of unregularised LR under multicollinearity.

In [64]:
# Logistic Regression hyperparameter search space
lr_param_dist = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'elasticnet'],
    'solver': ['saga'],  # saga supports all penalties
    'l1_ratio': [0, 0.25, 0.5, 0.75, 1],  # for elasticnet
    'max_iter': [1000, 2000, 3000]
}

# Initialize Logistic Regression
lr_base = LogisticRegression(
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

# Randomized search
print("Optimizing Logistic Regression hyperparameters...")
print(f"Search space size: {np.prod([len(v) for v in lr_param_dist.values()]):,} combinations")
print("Using RandomizedSearchCV with 30 iterations\n")

lr_random = RandomizedSearchCV(
    estimator=lr_base,
    param_distributions=lr_param_dist,
    n_iter=30,
    cv=tscv,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# Fit
lr_random.fit(X_train_scaled.fillna(X_train_scaled.median()), y_train)

# Best parameters
print(f"\n{'='*60}")
print("Logistic Regression Optimization Complete")
print(f"{'='*60}")
print(f"Best AUC (CV): {lr_random.best_score_:.4f}")
print(f"Best parameters:")
for param, value in lr_random.best_params_.items():
    print(f"  {param}: {value}")

# Store best model
lr_model = lr_random.best_estimator_

Optimizing Logistic Regression hyperparameters...
Search space size: 180 combinations
Using RandomizedSearchCV with 30 iterations

Fitting 5 folds for each of 30 candidates, totalling 150 fits


[cache] saved lr_random to model_cache/lr_random.pkl

Logistic Regression Optimization Complete
Best AUC (CV): 0.7397
Best parameters:
  solver: saga
  penalty: l1
  max_iter: 1000
  l1_ratio: 0.75
  C: 0.1


### 4.3.4 Base Learner 4: SVM

**Model:** Linear-kernel Support Vector Classifier. The decision function $f(x)=w^T x+b$ is fit by maximising the margin subject to a hinge-loss penalty on misclassified/margin-violating points:
$$\min_{w,b}\ \tfrac12\|w\|^2 + C\sum_{i=1}^n \max\left(0,\ 1-y_i f(x_i)\right)$$

`SVC` has no native `predict_proba`; with `probability=True`, sklearn fits an auxiliary 1-D logistic regression on the decision-function outputs (**Platt scaling**) to obtain calibrated probabilities:
$$P(y=1\mid x) = \sigma\big(A\cdot f(x) + B\big)$$

In [65]:
# SVM hyperparameter search
# Linear kernel: O(n) vs O(n^2) for RBF — fast enough for ~900 samples
# probability=True enables predict_proba (Platt scaling internally)

svm_param_dist = {
    'C':            [0.001, 0.01, 0.1, 0.5, 1, 5, 10, 50],
    'kernel':       ['linear'],
    'class_weight': ['balanced', None]
}

svm_base = SVC(probability=True, random_state=42)

print('Optimizing SVM hyperparameters...')
print(f"Search space size: {2 * 8:,} combinations (linear kernel only)")
print('Using RandomizedSearchCV with 15 iterations\n')

svm_random = RandomizedSearchCV(
    estimator=svm_base,
    param_distributions=svm_param_dist,
    n_iter=15,
    cv=tscv,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

svm_random.fit(X_train_scaled.fillna(X_train_scaled.median(numeric_only=True)).fillna(0), y_train)

print(f"\n{'='*60}")
print('SVM Optimization Complete')
print(f"{'='*60}")
print(f'Best AUC (CV): {svm_random.best_score_:.4f}')
print('Best parameters:')
for param, value in svm_random.best_params_.items():
    print(f'  {param}: {value}')

svm_model = svm_random.best_estimator_


Optimizing SVM hyperparameters...
Search space size: 16 combinations (linear kernel only)
Using RandomizedSearchCV with 15 iterations

Fitting 5 folds for each of 15 candidates, totalling 75 fits


[cache] saved svm_random to model_cache/svm_random.pkl

SVM Optimization Complete
Best AUC (CV): 0.7452
Best parameters:
  kernel: linear
  class_weight: None
  C: 0.01


### 4.3.5 Base Learner 5: MLP (Multi-Layer Perceptron)

**Model:** A feedforward neural network with one or two hidden layers; each unit applies a nonlinear activation $\phi$ (ReLU or tanh) to a weighted sum of its inputs, and the output layer applies a sigmoid to produce a probability:
$$h^{(1)} = \phi\big(W^{(1)}x+b^{(1)}\big), \qquad \hat p(y=1\mid x) = \sigma\big(W^{(2)}h^{(1)}+b^{(2)}\big)$$

Trained by minimising regularised binary cross-entropy via backpropagation (stochastic gradient descent):
$$\mathcal{L}(W,b) = -\sum_{i=1}^n\left[y_i\log \hat p_i + (1-y_i)\log(1-\hat p_i)\right] + \alpha\sum_l\|W^{(l)}\|_2^2$$

**Why include:** an empirical comparison (GaussianNB vs. MLP vs. the originally-used Ridge Classifier, same train/test split and CV protocol) showed the MLP with the best individual backtest Sharpe of any single base learner, with AUC comparable to Ridge. Its hidden layer also gives the ensemble a genuinely nonlinear decision boundary — a third distinct model family alongside the tree ensembles (bagging/boosting) and the linear classifiers (Logistic Regression, SVM).

In [66]:
# MLP Classifier — feedforward neural network base learner
# Adds a genuinely nonlinear decision boundary, distinct from the tree
# ensembles (RF/LightGBM) and the linear models (LR/SVM) above.
# Has native predict_proba, so no CalibratedClassifierCV wrapper needed.

from sklearn.neural_network import MLPClassifier

mlp_param_dist = {
    'hidden_layer_sizes': [(16,), (32,), (16, 8), (32, 16), (64, 32)],
    'alpha': np.logspace(-4, 0, 20),
    'learning_rate_init': np.logspace(-4, -1, 20),
    'activation': ['relu', 'tanh']
}

print('Optimizing MLP Classifier hyperparameters...')
print('Using RandomizedSearchCV with 30 iterations\n')

mlp_random = RandomizedSearchCV(
    estimator=MLPClassifier(max_iter=1000, early_stopping=True, random_state=42),
    param_distributions=mlp_param_dist,
    n_iter=30,
    cv=tscv,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

mlp_random.fit(X_train_scaled.fillna(X_train_scaled.median(numeric_only=True)).fillna(0), y_train)

print(f"\n{'='*60}")
print('MLP Classifier Optimization Complete')
print(f"{'='*60}")
print(f'Best AUC (CV): {mlp_random.best_score_:.4f}')
print('Best parameters:')
for param, value in mlp_random.best_params_.items():
    print(f'  {param}: {value}')

mlp_model = mlp_random.best_estimator_

Optimizing MLP Classifier hyperparameters...
Using RandomizedSearchCV with 30 iterations

Fitting 5 folds for each of 30 candidates, totalling 150 fits


[cache] saved mlp_random to model_cache/mlp_random.pkl

MLP Classifier Optimization Complete
Best AUC (CV): 0.7244
Best parameters:
  learning_rate_init: 0.023357214690901212
  hidden_layer_sizes: (64, 32)
  alpha: 0.012742749857031334
  activation: relu


## 4.4 Blending Ensemble Architecture

### How Blending Works:
1. Split training data into 5 temporal folds using TimeSeriesSplit
2. For each fold: train base learners on past folds, predict on current fold (Out-of-Fold predictions)
3. Collect all OOF predictions across all folds → meta-features matrix (904 × 5)
4. Retrain all base learners on the FULL training set
5. Train meta-model (Logistic Regression) on the OOF meta-features matrix
6. Final test predictions: base learners (trained on full train) → meta-model → ensemble probability

### Meta-Model Choice: Logistic Regression
The meta-model is a **Logistic Regression** (no regularization penalty), which acts as a linear combiner of the 5 base learner probability outputs. This choice is deliberate:
- **Interpretable weights**: meta-model coefficients directly show which base learner the ensemble trusts most
- **Low overfitting risk**: only 5 input features on ~900 OOF samples; a complex meta-model would overfit
- **Probabilistic output**: LR produces well-calibrated probabilities suitable for the downstream confidence threshold overlay

### Mathematical Form of the Blending Combiner

The meta-model takes the 5-dimensional vector of OOF base-learner probabilities $z=[p_{RF},p_{LGBM},p_{LR},p_{SVM},p_{MLP}]$ and combines them **linearly in log-odds space**:

$$\hat p_{blend} = \sigma\big(w^Tz+b\big) = \frac{1}{1+e^{-(w^Tz+b)}}$$

fit by minimising unpenalised log-loss over the OOF training samples:
$$\mathcal{L}(w,b) = -\sum_{i=1}^n\left[y_i\log \hat p_{blend,i} + (1-y_i)\log(1-\hat p_{blend,i})\right]$$

With no regularisation term, $w_k$ directly reflects how much the blend trusts base learner $k$ — this is the "interpretable weights" property referenced above.

In [67]:
# OOF Blending: Out-of-Fold predictions across all training samples

from sklearn.model_selection import TimeSeriesSplit
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')

print('OOF Blending — TimeSeriesSplit(5)')
print('=' * 60)

tscv_blend = TimeSeriesSplit(n_splits=5)

base_learner_names  = ['Random Forest', 'LightGBM',
                        'Logistic Regression', 'SVM', 'MLP Classifier']
base_learner_models = [rf_model, lgb_model,
                        lr_model, svm_model, mlp_model]

print(f'Base learners: {base_learner_names}')
print(f'Total training samples: {len(X_train_scaled)}')

X_train_filled = X_train_scaled.fillna(X_train_scaled.median(numeric_only=True)).fillna(0)
X_test_filled  = X_test_scaled.fillna(X_train_scaled.median(numeric_only=True)).fillna(0)
X_meta_train   = X_train_scaled  # backward compat

meta_features_oof = np.zeros((len(X_train_scaled), len(base_learner_names)))

print('Generating OOF predictions...')
for fold_idx, (tr_idx, val_idx) in enumerate(tscv_blend.split(X_train_filled)):
    X_tr, X_val = X_train_filled.iloc[tr_idx], X_train_filled.iloc[val_idx]
    y_tr = y_train.iloc[tr_idx]
    for i, (name, model) in enumerate(zip(base_learner_names, base_learner_models)):
        fold_model = clone(model)
        fold_model.fit(X_tr, y_tr)
        meta_features_oof[val_idx, i] = fold_model.predict_proba(X_val)[:, 1]
    print(f'  Fold {fold_idx+1}: train={len(tr_idx)}, val={len(val_idx)}')

# Dynamic column names — no hardcoding needed
col_names = [name.lower().replace(' ', '_') for name in base_learner_names]
meta_features_oof = pd.DataFrame(
    meta_features_oof,
    columns=col_names,
    index=X_train_scaled.index
)
print(f'OOF meta-features: {meta_features_oof.shape}')
print(f'Columns: {list(meta_features_oof.columns)}')


OOF Blending — TimeSeriesSplit(5)
Base learners: ['Random Forest', 'LightGBM', 'Logistic Regression', 'SVM', 'MLP Classifier']
Total training samples: 875
Generating OOF predictions...


  Fold 1: train=150, val=145


  Fold 2: train=295, val=145


  Fold 3: train=440, val=145


  Fold 4: train=585, val=145


  Fold 5: train=730, val=145
OOF meta-features: (875, 5)
Columns: ['random_forest', 'lightgbm', 'logistic_regression', 'svm', 'mlp_classifier']


### Missing Value Imputation Strategy

Rolling-window features (e.g. 60-day drawdown, 20-day realised volatility) are undefined for the first ~60 observations of the sample (the "warm-up" period) and are `NaN` there; Section 3.1 profiles the extent of this. Before any model is fit, `NaN` values in the scaled feature matrix are imputed with the **training-set median** of each column, with any residual `NaN` filled with 0. The median is always computed on the training fold only (`X_train_scaled.median()`), so no information from the validation/test fold leaks into the imputed values — consistent with the no-leakage principle already applied to the `RobustScaler` fit in 4.2.

In [68]:
# Retrain base learners on FULL training set, generate test meta-features
print('Retraining on full training set for test predictions...')
print('=' * 60)

base_learners = {}
meta_features_test = pd.DataFrame(index=X_test_scaled.index)

for name, model in zip(base_learner_names, base_learner_models):
    full_model = clone(model)
    full_model.fit(X_train_filled, y_train)
    base_learners[name] = full_model
    col = name.lower().replace(' ', '_')
    meta_features_test[col] = full_model.predict_proba(X_test_filled)[:, 1]
    print(f'  {name}: trained on {len(X_train_filled)} samples')

print(f'\nTest meta-features: {meta_features_test.shape}')
print('Mean predicted prob (test):')
print(meta_features_test.mean().to_string())


Retraining on full training set for test predictions...


  Random Forest: trained on 875 samples


  LightGBM: trained on 875 samples
  Logistic Regression: trained on 875 samples


  SVM: trained on 875 samples
  MLP Classifier: trained on 875 samples

Test meta-features: (219, 5)
Mean predicted prob (test):
random_forest          0.460560
lightgbm               0.500359
logistic_regression    0.521111
svm                    0.689616
mlp_classifier         0.702572


In [69]:
# Train meta-model on all OOF predictions (904 samples vs 452 before)
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
meta_model.fit(meta_features_oof, y_train)

print('Meta-model coefficients:')
for feature, coef in zip(meta_features_oof.columns, meta_model.coef_[0]):
    print(f'  {feature:25s}: {coef:+.4f}')
print(f'\nIntercept: {meta_model.intercept_[0]:.4f}')
print(f'Meta-model trained on {len(meta_features_oof)} OOF samples (was {len(X_train_scaled)//2})')
print('Meta-model trained successfully')


Meta-model coefficients:
  random_forest            : +0.8030
  lightgbm                 : +0.1599
  logistic_regression      : -0.6539
  svm                      : +1.1898
  mlp_classifier           : -0.0748

Intercept: -0.4745
Meta-model trained on 875 OOF samples (was 437)
Meta-model trained successfully


## Numerical Methods Summary

| Method | Implementation | Source |
|---|---|---|
| Binary cross-entropy loss | Coded via sklearn estimators | scikit-learn |
| Gini impurity (Random Forest splits) | sklearn `RandomForestClassifier` | scikit-learn |
| Gradient boosting (LightGBM leaf-wise) | `LightGBMClassifier` with custom hyperparameters | LightGBM |
| Log-odds linear combiner (meta-model) | Custom OOF pipeline + `LogisticRegression` | scikit-learn |
| Hinge loss + Platt scaling (SVM) | `SVC(probability=True)` | scikit-learn |
| Backpropagation / SGD (MLP) | `MLPClassifier` | scikit-learn |
| TimeSeriesSplit cross-validation | Custom OOF blending loop | scikit-learn |
| RobustScaler feature normalisation | Fit on train fold only, applied to test | scikit-learn |
| RandomizedSearchCV hyperparameter optimisation | 30 iterations per model, AUC scoring | scikit-learn |
| Median imputation (missing values) | Computed on train fold only | custom code |
| Sharpe ratio, Calmar ratio, Max Drawdown | Vectorised pandas/numpy computation | custom code |
| Cumulative return & drawdown series | Custom backtest_strategy() function | custom code |

# 7. Performance Evaluation & Backtesting

**Requirement 7 Compliance:** Multiple evaluation metrics including AUC, confusion matrix, classification report, and backtesting with trading strategy

---

## 7.1 Classification Metrics

### 7.1.1 Base Learner Performance (Individual Models)

In [70]:
# Evaluate each base learner on test set
print("BASE LEARNER PERFORMANCE ON TEST SET")
print(f"{'='*80}\n")

base_learner_results = {}

for name, model in base_learners.items():
    print(f"\n{name}")
    print(f"{'-'*80}")

    # Predictions
    y_pred = model.predict(X_test_scaled.fillna(X_meta_train.median(numeric_only=True)).fillna(0))
    y_prob = model.predict_proba(X_test_scaled.fillna(X_meta_train.median(numeric_only=True)).fillna(0))[:, 1]

    # Metrics
    auc = roc_auc_score(y_test, y_prob)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store results
    base_learner_results[name] = {
        'AUC': auc,
        'Balanced Accuracy': balanced_acc,
        'y_pred': y_pred,
        'y_prob': y_prob
    }

    # Print metrics
    print(f"AUC-ROC: {auc:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Down (0)', 'Up (1)'], digits=4))
    print(f"\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(f"              Predicted")
    print(f"              Down  Up")
    print(f"Actual Down  {cm[0,0]:5d} {cm[0,1]:4d}")
    print(f"       Up    {cm[1,0]:5d} {cm[1,1]:4d}")

print(f"\n{'='*80}")
print("Base Learner Summary:")
print(f"{'='*80}")
summary_df = pd.DataFrame({
    name: {'AUC': results['AUC'], 'Balanced Acc': results['Balanced Accuracy']}
    for name, results in base_learner_results.items()
}).T.sort_values('AUC', ascending=False)
print(summary_df.to_string())

BASE LEARNER PERFORMANCE ON TEST SET


Random Forest
--------------------------------------------------------------------------------


AUC-ROC: 0.5707
Balanced Accuracy: 0.5218

Classification Report:
              precision    recall  f1-score   support

    Down (0)     0.5000    0.5524    0.5249       105
      Up (1)     0.5437    0.4912    0.5161       114

    accuracy                         0.5205       219
   macro avg     0.5218    0.5218    0.5205       219
weighted avg     0.5227    0.5205    0.5203       219


Confusion Matrix:
              Predicted
              Down  Up
Actual Down     58   47
       Up       58   56

LightGBM
--------------------------------------------------------------------------------
AUC-ROC: 0.6058
Balanced Accuracy: 0.5294

Classification Report:
              precision    recall  f1-score   support

    Down (0)     0.5093    0.5238    0.5164       105
      Up (1)     0.5495    0.5351    0.5422       114

    accuracy                         0.5297       219
   macro avg     0.5294    0.5294    0.5293       219
weighted avg     0.5302    0.5297    0.5299       219


Confusio

AUC-ROC: 0.6282
Balanced Accuracy: 0.5581

Classification Report:
              precision    recall  f1-score   support

    Down (0)     0.5758    0.3619    0.4444       105
      Up (1)     0.5621    0.7544    0.6442       114

    accuracy                         0.5662       219
   macro avg     0.5689    0.5581    0.5443       219
weighted avg     0.5686    0.5662    0.5484       219


Confusion Matrix:
              Predicted
              Down  Up
Actual Down     38   67
       Up       28   86

Base Learner Summary:
                          AUC  Balanced Acc
Logistic Regression  0.660652      0.597118
MLP Classifier       0.628154      0.558145
SVM                  0.616458      0.555890
LightGBM             0.605764      0.529449
Random Forest        0.570677      0.521805


### 7.1.2 Blending Ensemble Performance

**Ensemble combines all base learners via meta-model**

In [71]:
# Blending Ensemble predictions
print("🏆 BLENDING ENSEMBLE PERFORMANCE ON TEST SET")
print(f"{'='*80}\n")

# Get ensemble predictions using meta-model
y_pred_blend = meta_model.predict(meta_features_test)
y_prob_blend = meta_model.predict_proba(meta_features_test)[:, 1]

# Calculate metrics
auc_blend = roc_auc_score(y_test, y_prob_blend)
balanced_acc_blend = balanced_accuracy_score(y_test, y_pred_blend)

print(f"AUC-ROC: {auc_blend:.4f}")
print(f"Balanced Accuracy: {balanced_acc_blend:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_blend, target_names=['Down (0)', 'Up (1)'], digits=4))

print(f"\nConfusion Matrix:")
cm_blend = confusion_matrix(y_test, y_pred_blend)
print(f"              Predicted")
print(f"              Down  Up")
print(f"Actual Down  {cm_blend[0,0]:5d} {cm_blend[0,1]:4d}")
print(f"       Up    {cm_blend[1,0]:5d} {cm_blend[1,1]:4d}")

# Calculate improvement over best base learner
best_base_auc = max([r['AUC'] for r in base_learner_results.values()])
improvement = ((auc_blend - best_base_auc) / best_base_auc) * 100

print(f"\n{'='*80}")
print(f"📈 ENSEMBLE IMPROVEMENT")
print(f"{'='*80}")
print(f"Best base learner AUC: {best_base_auc:.4f}")
print(f"Blending ensemble AUC: {auc_blend:.4f}")
print(f"Improvement: {improvement:+.2f}%")

🏆 BLENDING ENSEMBLE PERFORMANCE ON TEST SET

AUC-ROC: 0.5580
Balanced Accuracy: 0.5024

Classification Report:
              precision    recall  f1-score   support

    Down (0)     0.4865    0.1714    0.2535       105
      Up (1)     0.5220    0.8333    0.6419       114

    accuracy                         0.5160       219
   macro avg     0.5042    0.5024    0.4477       219
weighted avg     0.5050    0.5160    0.4557       219


Confusion Matrix:
              Predicted
              Down  Up
Actual Down     18   87
       Up       19   95

📈 ENSEMBLE IMPROVEMENT
Best base learner AUC: 0.6607
Blending ensemble AUC: 0.5580
Improvement: -15.54%


### 7.1.3 ROC Curves Comparison

In [72]:
# Plot ROC curves for all models
fig, ax = plt.subplots(figsize=(12, 8))

# Plot each base learner
for name, results in base_learner_results.items():
    fpr, tpr, _ = roc_curve(y_test, results['y_prob'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={results['AUC']:.4f})", linewidth=2, alpha=0.7)

# Plot blending ensemble
fpr_blend, tpr_blend, _ = roc_curve(y_test, y_prob_blend)
ax.plot(fpr_blend, tpr_blend, label=f"Blending Ensemble (AUC={auc_blend:.4f})",
        linewidth=3, color='red', linestyle='--')

# Plot diagonal (random classifier)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.5000)')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves: Base Learners vs Blending Ensemble', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("ROC curves plotted")

ROC curves plotted


Visual comparison of all models

---

## 7.2 Backtesting: Volatility-Timed Exposure Strategy

**Requirement 7:** Trading strategy based on predicted signals

### Strategy Description:
- **Long-Only Binary**: model predicts CALM ahead (prob(label=1) > 0.5) → hold QQQ; model predicts a vol STORM ahead (prob ≤ 0.5) → cash (0 exposure). Recall label=1 was defined as "volatility falling", so this `prob > 0.5 -> long` rule is a genuine risk-off-ahead-of-turbulence overlay, not a relabelled price-direction bet.
- **No short positions**: strategy is long-only, consistent with a risk overlay on an existing portfolio
- **Confidence threshold overlay**: optional filter — only enter when |prob − 0.5| ≥ threshold, reducing whipsaw in low-conviction signals
- **Daily rebalancing** at close price

---

In [73]:
# Backtest: binary long/flat — position = 1 (long) or 0 (cash)
# LOOK-AHEAD BIAS FIX: signal on day T earns return on day T+1
# shift_returns=True (default) aligns T-signal with T+1 return
def backtest_strategy(predictions_prob, actual_returns, strategy_name,
                      confidence_threshold=0.0, shift_returns=True):

    # --- Time alignment: T-signal -> T+1 return ---
    if shift_returns:
        actual_returns = actual_returns.shift(-1).dropna()
        if isinstance(predictions_prob, pd.Series):
            prob = predictions_prob.reindex(actual_returns.index)
        else:
            arr = list(predictions_prob)
            prob = pd.Series(arr[:len(actual_returns)], index=actual_returns.index)
    else:
        if isinstance(predictions_prob, pd.Series):
            prob = pd.Series(predictions_prob.values, index=actual_returns.index)
        else:
            prob = pd.Series(predictions_prob, index=actual_returns.index)

    # Binary: long when model says up, flat (cash) when model says down
    raw_positions = (prob > 0.5).astype(float)  # [0, +1]

    if confidence_threshold > 0:
        in_market = (prob - 0.5).abs() >= confidence_threshold
        positions = raw_positions * in_market
        in_market_pct = float(in_market.mean())
    else:
        positions = raw_positions
        in_market_pct = float((positions > 0).mean())

    strategy_returns   = positions * actual_returns
    cumulative_returns = (1 + strategy_returns).cumprod()

    total_return      = cumulative_returns.iloc[-1] - 1
    annualized_return = (1 + total_return) ** (252 / len(strategy_returns)) - 1
    volatility        = strategy_returns.std() * np.sqrt(252)
    sharpe_ratio      = annualized_return / volatility if volatility > 0 else 0

    rolling_max  = cumulative_returns.cummax()
    drawdown     = (cumulative_returns - rolling_max) / rolling_max
    max_drawdown = drawdown.min()
    calmar_ratio = annualized_return / abs(max_drawdown) if max_drawdown < 0 else np.nan

    win_rate = (strategy_returns[strategy_returns != 0] > 0).mean() if (positions > 0).any() else 0
    avg_win  = strategy_returns[strategy_returns > 0].mean() if (strategy_returns > 0).any() else 0
    avg_loss = strategy_returns[strategy_returns < 0].mean() if (strategy_returns < 0).any() else 0

    return {
        "name":              strategy_name,
        "total_return":      total_return,
        "annualized_return": annualized_return,
        "volatility":        volatility,
        "sharpe_ratio":      sharpe_ratio,
        "max_drawdown":      max_drawdown,
        "drawdown_series":   drawdown,
        "win_rate":          win_rate,
        "avg_win":           avg_win,
        "avg_loss":          avg_loss,
        "in_market_pct":     in_market_pct,
        "cumulative_returns": cumulative_returns,
        "strategy_returns":  strategy_returns,
        "positions":         positions,
        "calmar_ratio":      calmar_ratio
    }

print("Backtest function: binary long/flat [0, +1] — look-ahead bias FIXED (shift_returns=True)")


Backtest function: binary long/flat [0, +1] — look-ahead bias FIXED (shift_returns=True)


In [74]:
# Run backtests — base learners at thresh=0, ensemble at multiple thresholds
test_returns = test_data["ret_log"]

print("BACKTESTING STRATEGIES")
print("=" * 80)

backtest_results = {}

for name, results in base_learner_results.items():
    backtest_results[name] = backtest_strategy(results["y_prob"], test_returns, name)

# Base ensemble (no threshold filter)
backtest_results["Blending Ensemble"] = backtest_strategy(
    y_prob_blend, test_returns, "Blending Ensemble"
)

# Confidence-filtered ensemble variants
THRESHOLDS = [0.01, 0.02, 0.03, 0.04, 0.05]
for thresh in THRESHOLDS:
    label = f"Blend (thresh={thresh:.0%})"
    backtest_results[label] = backtest_strategy(
        y_prob_blend, test_returns, label, confidence_threshold=thresh
    )

backtest_results["Buy & Hold"] = backtest_strategy(
    pd.Series(1.0, index=test_returns.index), test_returns, "Buy & Hold"
)
print("All strategies backtested")


BACKTESTING STRATEGIES
All strategies backtested


### Fine-Grained Threshold Sensitivity

The 5-point sweep above (1%–5%) is too coarse to tell whether the shape (e.g. the dip at 3%) is a real pattern or noise. This sweeps the confidence threshold on a finer 0.5% grid and plots Sharpe ratio and market exposure against it.

In [75]:
# Fine-grained threshold sweep: how sensitive is performance to the exact cutoff?
threshold_grid = np.arange(0.0, 0.101, 0.005)
sweep_rows = []
for t in threshold_grid:
    bt = backtest_strategy(y_prob_blend, test_returns, f'thresh={t:.1%}', confidence_threshold=t)
    sweep_rows.append({
        'threshold': t,
        'sharpe_ratio': bt['sharpe_ratio'],
        'annualized_return': bt['annualized_return'],
        'max_drawdown': bt['max_drawdown'],
        'in_market_pct': bt.get('in_market_pct', 1.0),
    })
sweep_df = pd.DataFrame(sweep_rows)
best_row = sweep_df.loc[sweep_df['sharpe_ratio'].idxmax()]

print(sweep_df.to_string(index=False, formatters={
    'threshold': '{:.1%}'.format,
    'sharpe_ratio': '{:.3f}'.format,
    'annualized_return': '{:.2%}'.format,
    'max_drawdown': '{:.2%}'.format,
    'in_market_pct': '{:.1%}'.format,
}))
print(f"\nBest threshold by Sharpe: {best_row['threshold']:.1%} (Sharpe={best_row['sharpe_ratio']:.3f})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(sweep_df['threshold'] * 100, sweep_df['sharpe_ratio'], marker='o', color='steelblue')
ax1.axvline(best_row['threshold'] * 100, color='red', linestyle='--', alpha=0.6,
            label=f"best={best_row['threshold']:.1%}")
ax1.set_xlabel('Confidence Threshold (%)')
ax1.set_ylabel('Sharpe Ratio')
ax1.set_title('Sharpe Ratio vs. Confidence Threshold', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(sweep_df['threshold'] * 100, sweep_df['in_market_pct'] * 100, marker='o', color='darkorange')
ax2.set_xlabel('Confidence Threshold (%)')
ax2.set_ylabel('% Time In Market')
ax2.set_title('Market Exposure vs. Confidence Threshold', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

threshold sharpe_ratio annualized_return max_drawdown in_market_pct
     0.0%        0.415             9.94%      -21.35%         83.0%
     0.5%        0.465            11.14%      -21.35%         96.3%
     1.0%        0.276             6.57%      -21.59%         93.1%
     1.5%        0.276             6.57%      -21.59%         92.7%
     2.0%        0.152             3.60%      -22.82%         89.4%
     2.5%        0.033             0.77%      -22.82%         88.1%
     3.0%        0.092             2.17%      -22.39%         85.8%
     3.5%        0.146             3.44%      -21.89%         83.9%
     4.0%        0.356             8.33%      -18.85%         80.7%
     4.5%        0.237             5.53%      -18.85%         78.0%
     5.0%        0.161             3.74%      -18.85%         74.8%
     5.5%        0.126             2.93%      -18.58%         71.6%
     6.0%        0.151             3.51%      -18.58%         68.8%
     6.5%        0.102             2.37%      -1

### Note: Isotonic Calibration Was Tested and Dropped

An earlier iteration of this pipeline applied isotonic calibration (`CalibratedClassifierCV`) to the meta-model's output before applying the confidence threshold. Two problems showed up on repeated runs: (1) the calibrated probabilities collapsed onto a small number of discrete plateaus, so thresholds from 2% through 7% consistently produced **identical** backtest results — the calibration step, not the threshold logic, was destroying the intended granularity; and (2) calibration alone (no threshold) underperformed the raw, uncalibrated ensemble outright. The strategy below therefore uses the **raw meta-model probability directly**, with the confidence threshold as the only risk-overlay mechanism — simpler, and empirically no worse.

In [76]:
# Display backtest results summary
print('BACKTEST PERFORMANCE SUMMARY')
print('=' * 130)

summary_data = []
for name, result in backtest_results.items():
    summary_data.append({
        'Strategy':      name,
        'Total Return':  f"{result['total_return']:.2%}",
        'Annual Return': f"{result['annualized_return']:.2%}",
        'Volatility':    f"{result['volatility']:.2%}",
        'Sharpe Ratio':  f"{result['sharpe_ratio']:.3f}",
        'Max Drawdown':  f"{result['max_drawdown']:.2%}",
        'Calmar Ratio':  f"{result['calmar_ratio']:.3f}",
        'Win Rate':      f"{result['win_rate']:.2%}",
        'In Market':     f"{result.get('in_market_pct', 1.0):.0%}",
    })

summary_df = pd.DataFrame(summary_data).set_index('Strategy')
summary_df['_s'] = [backtest_results[idx]['sharpe_ratio'] for idx in summary_df.index]
summary_df = summary_df.sort_values('_s', ascending=False).drop('_s', axis=1)
print(summary_df.to_string())
print('=' * 130)
print("Note: 'In Market' = fraction of days with non-zero position (threshold effect)")


BACKTEST PERFORMANCE SUMMARY
                    Total Return Annual Return Volatility Sharpe Ratio Max Drawdown Calmar Ratio Win Rate In Market
Strategy                                                                                                           
Buy & Hold                16.23%        18.98%     24.76%        0.767      -23.38%        0.812   58.26%      100%
SVM                       13.09%        15.28%     22.79%        0.670      -16.06%        0.951   57.93%       75%
LightGBM                  10.62%        12.37%     21.90%        0.565      -17.08%        0.724   59.09%       50%
Random Forest              8.38%         9.74%     21.61%        0.451      -18.58%        0.524   58.82%       47%
Blending Ensemble          8.54%         9.94%     23.95%        0.415      -21.35%        0.465   56.35%       83%
Blend (thresh=4%)          7.16%         8.33%     23.39%        0.356      -18.85%        0.442   56.88%       81%
MLP Classifier             6.12%         7.

In [77]:
# Signal inversion diagnostic: test whether inverting low-AUC models improves performance
# AUC < 0.5 implies the model's signal direction is reversed; inverting it should theoretically give AUC = 1 - original

print("SIGNAL INVERSION DIAGNOSTIC")
print("=" * 90)
print("AUC < 0.5 => model has info but wrong direction => try inverting")
print()

rows = []
for name, res in base_learner_results.items():
    orig_prob = res["y_prob"]
    inv_prob  = 1.0 - orig_prob

    orig = backtest_strategy(orig_prob, test_returns, name)
    inv  = backtest_strategy(inv_prob,  test_returns, name + " (INVERTED)")

    orig_auc = res["AUC"]  # capital AUC
    rows.append({
        "Model":            name,
        "Test AUC":         f"{orig_auc:.4f}",
        "Inv AUC":          f"{1 - orig_auc:.4f}",
        "Orig Sharpe":      f"{orig['sharpe_ratio']:+.3f}",
        "Inv Sharpe":       f"{inv['sharpe_ratio']:+.3f}",
        "Orig Return":      f"{orig['total_return']:.2%}",
        "Inv Return":       f"{inv['total_return']:.2%}",
        "Better Inverted?": "YES >>>" if inv["sharpe_ratio"] > orig["sharpe_ratio"] else "no"
    })

# Also test ensemble inversion
blend_auc = roc_auc_score(y_test, y_prob_blend)
orig_blend = backtest_strategy(y_prob_blend,       test_returns, "Ensemble")
inv_blend  = backtest_strategy(1.0 - y_prob_blend, test_returns, "Ensemble (INVERTED)")
rows.append({
    "Model":            "Blending Ensemble",
    "Test AUC":         f"{blend_auc:.4f}",
    "Inv AUC":          f"{1 - blend_auc:.4f}",
    "Orig Sharpe":      f"{orig_blend['sharpe_ratio']:+.3f}",
    "Inv Sharpe":       f"{inv_blend['sharpe_ratio']:+.3f}",
    "Orig Return":      f"{orig_blend['total_return']:.2%}",
    "Inv Return":       f"{inv_blend['total_return']:.2%}",
    "Better Inverted?": "YES >>>" if inv_blend["sharpe_ratio"] > orig_blend["sharpe_ratio"] else "no"
})

diag_df = pd.DataFrame(rows).set_index("Model")
print(diag_df.to_string())

print()
print("CONCLUSION:")
for row in rows:
    tag = "INVERT" if row["Better Inverted?"].startswith("YES") else "keep"
    print(f"  [{tag:6s}] {row['Model']:30s}  Sharpe: {row['Orig Sharpe']} -> {row['Inv Sharpe']}")


SIGNAL INVERSION DIAGNOSTIC
AUC < 0.5 => model has info but wrong direction => try inverting

                    Test AUC Inv AUC Orig Sharpe Inv Sharpe Orig Return Inv Return Better Inverted?
Model                                                                                              
Random Forest         0.5707  0.4293      +0.451     +0.695       8.38%      7.24%          YES >>>
LightGBM              0.6058  0.3942      +0.565     +0.508      10.62%      5.07%               no
Logistic Regression   0.6607  0.3393      +0.207     +1.083       3.76%     12.01%          YES >>>
SVM                   0.6165  0.3835      +0.670     +0.480      13.09%      3.74%               no
MLP Classifier        0.6282  0.3718      +0.306     +1.281       6.12%      9.53%          YES >>>
Blending Ensemble     0.5580  0.4420      +0.415     +1.301       8.54%      7.08%          YES >>>

CONCLUSION:
  [INVERT] Random Forest                   Sharpe: +0.451 -> +0.695
  [keep  ] LightGBM      

In [78]:
# Plot cumulative returns comparison
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: All strategies
for name, result in backtest_results.items():
    if name == 'Blending Ensemble':
        ax1.plot(result['cumulative_returns'], label=name, linewidth=3, linestyle='--', color='red')
    elif name == 'Buy & Hold':
        ax1.plot(result['cumulative_returns'], label=name, linewidth=2, linestyle=':', color='black', alpha=0.7)
    else:
        ax1.plot(result['cumulative_returns'], label=name, linewidth=1.5, alpha=0.7)

ax1.set_xlabel('Date', fontsize=11)
ax1.set_ylabel('Cumulative Return', fontsize=11)
ax1.set_title('Cumulative Returns: All Strategies vs Buy & Hold', fontsize=13, fontweight='bold')
ax1.legend(loc='best', fontsize=9)
ax1.grid(True, alpha=0.3)

# Plot 2: Blending Ensemble vs Buy & Hold only (clearer comparison)
ax2.plot(backtest_results['Blending Ensemble']['cumulative_returns'],
         label='Blending Ensemble', linewidth=3, color='red')
ax2.plot(backtest_results['Buy & Hold']['cumulative_returns'],
         label='Buy & Hold', linewidth=2.5, linestyle='--', color='black')

ax2.set_xlabel('Date', fontsize=11)
ax2.set_ylabel('Cumulative Return', fontsize=11)
ax2.set_title('Blending Ensemble vs Buy & Hold Benchmark', fontsize=13, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Cumulative returns plotted")

Cumulative returns plotted


In [79]:
# Analyze position distribution for ensemble
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of positions
positions_blend = backtest_results['Blending Ensemble']['positions']
ax1.hist(positions_blend, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax1.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Neutral')
ax1.set_xlabel('Position Size', fontsize=11)
ax1.set_ylabel('Frequency', fontsize=11)
ax1.set_title('Distribution of Position Sizes\n(Blending Ensemble)', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Time series of positions
ax2.plot(positions_blend, linewidth=1, alpha=0.8, color='steelblue')
ax2.axhline(y=0, color='red', linestyle='--', linewidth=1.5, label='Neutral')
ax2.axhline(y=0.5, color='green', linestyle=':', linewidth=1, alpha=0.5, label='50% Long')
ax2.axhline(y=-0.5, color='orange', linestyle=':', linewidth=1, alpha=0.5, label='50% Short')
ax2.fill_between(positions_blend.index, 0, positions_blend, where=(positions_blend > 0),
                  alpha=0.3, color='green', label='Long')
ax2.fill_between(positions_blend.index, 0, positions_blend, where=(positions_blend < 0),
                  alpha=0.3, color='red', label='Short')
ax2.set_xlabel('Date', fontsize=11)
ax2.set_ylabel('Position Size', fontsize=11)
ax2.set_title('Position Sizing Over Time\n(Blending Ensemble)', fontsize=12, fontweight='bold')
ax2.legend(loc='best', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Position statistics:")
print(f"Mean position: {positions_blend.mean():.3f}")
print(f"Std position: {positions_blend.std():.3f}")
print(f"Max long: {positions_blend.max():.3f}")
print(f"Max short: {positions_blend.min():.3f}")
print(f"% Time long (pos > 0): {(positions_blend > 0).mean():.1%}")
print(f"% Time short (pos < 0): {(positions_blend < 0).mean():.1%}")
print(f"% Time neutral (~0): {(np.abs(positions_blend) < 0.1).mean():.1%}")

Position statistics:
Mean position: 0.830
Std position: 0.376
Max long: 1.000
Max short: 0.000
% Time long (pos > 0): 83.0%
% Time short (pos < 0): 0.0%
% Time neutral (~0): 17.0%


In [80]:
# Generate comprehensive final report
print("="*120)
print(" " * 40 + "FINAL PROJECT SUMMARY")
print("="*120)
print()

print("PROJECT REQUIREMENTS COMPLIANCE")
print("-" * 120)
requirements = {
    'Ticker Selection': 'QQQ ETF (Nasdaq-100 tracking)',
    'Prediction Target': '20d-forward vs trailing realized volatility direction, binary classification [0, 1]',
    'Feature Engineering': '37 features across 7 categories (Momentum, Trend, Drawdown, Volatility, Volume, Macro, Sentiment)',
    'Data Preprocessing': 'Missing value handling, outlier detection, RobustScaler normalization',
    'Base Learners (≥3)': f'{len(base_learners)} models - ' + ', '.join(base_learners.keys()),
    'Hyperparameter Optimization': 'RandomizedSearchCV with TimeSeriesSplit (30 iterations each)',
    'Blending Architecture': 'Train-base/train-meta split + meta-model (Logistic Regression)',
    'Evaluation Metrics': 'AUC-ROC, Balanced Accuracy, Confusion Matrix, Classification Report',
    'Backtesting': 'Probability-weighted rebalancing strategy with performance metrics',
    'Trading Strategy': 'Long-only binary (prob>0.5=hold, else cash) with optional confidence-threshold overlay'
}

for req, status in requirements.items():
    print(f"{req:30s} : {status}")

print("\n" + "="*120)
print(" " * 40 + "MODEL PERFORMANCE")
print("="*120)
print()

print("Classification Metrics (Test Set):")
print("-" * 120)
print(f"{'Model':<25s} {'AUC-ROC':>10s} {'Bal. Acc':>10s} {'Precision (0)':>15s} {'Precision (1)':>15s} {'Recall (0)':>12s} {'Recall (1)':>12s}")
print("-" * 120)

# Extract precision and recall for each model
for name, results in base_learner_results.items():
    from sklearn.metrics import precision_score, recall_score
    y_pred = results['y_pred']
    prec_0 = precision_score(y_test, y_pred, pos_label=0)
    prec_1 = precision_score(y_test, y_pred, pos_label=1)
    rec_0 = recall_score(y_test, y_pred, pos_label=0)
    rec_1 = recall_score(y_test, y_pred, pos_label=1)

    print(f"{name:<25s} {results['AUC']:>10.4f} {results['Balanced Accuracy']:>10.4f} "
          f"{prec_0:>15.4f} {prec_1:>15.4f} {rec_0:>12.4f} {rec_1:>12.4f}")

# Ensemble
from sklearn.metrics import precision_score, recall_score
prec_0_ens = precision_score(y_test, y_pred_blend, pos_label=0)
prec_1_ens = precision_score(y_test, y_pred_blend, pos_label=1)
rec_0_ens = recall_score(y_test, y_pred_blend, pos_label=0)
rec_1_ens = recall_score(y_test, y_pred_blend, pos_label=1)

print("-" * 120)
print(f"{'BLENDING ENSEMBLE':<25s} {auc_blend:>10.4f} {balanced_acc_blend:>10.4f} "
      f"{prec_0_ens:>15.4f} {prec_1_ens:>15.4f} {rec_0_ens:>12.4f} {rec_1_ens:>12.4f}")
print("="*120)
print()

print("Trading Strategy Performance (Test Period):")
print("-" * 120)
print(f"{'Strategy':<25s} {'Total Ret':>12s} {'Annual Ret':>12s} {'Volatility':>12s} {'Sharpe':>10s} {'Max DD':>12s} {'Win Rate':>12s}")
print("-" * 120)

for name in ['Blending Ensemble', 'Random Forest', 'LightGBM', 'Logistic Regression', 'Buy & Hold']:
    result = backtest_results[name]
    print(f"{name:<25s} {result['total_return']:>11.2%} {result['annualized_return']:>11.2%} "
          f"{result['volatility']:>11.2%} {result['sharpe_ratio']:>10.3f} "
          f"{result['max_drawdown']:>11.2%} {result['win_rate']:>11.2%}")

print("="*120)
print()

print("KEY INSIGHTS & CONCLUSIONS:")
print("-" * 120)

# Determine best performers
best_auc_name = max(base_learner_results.items(), key=lambda x: x[1]['AUC'])[0]
best_sharpe_name = max(backtest_results.items(), key=lambda x: x[1]['sharpe_ratio'])[0]

acc_blend = (y_pred_blend == y_test).mean()
insights = [
    f"1. BEST CLASSIFICATION MODEL: {best_auc_name} (AUC={max([r['AUC'] for r in base_learner_results.values()]):.4f})",
    f"2. ENSEMBLE vs BEST BASE LEARNER: Blending ensemble AUC={auc_blend:.4f} ({improvement:+.2f}% vs best base learner)",
    f"3. BEST TRADING STRATEGY: {best_sharpe_name} (Sharpe={backtest_results[best_sharpe_name]['sharpe_ratio']:.3f})",
    f"4. OUTPERFORMANCE: Ensemble {'outperformed' if backtest_results['Blending Ensemble']['sharpe_ratio'] > backtest_results['Buy & Hold']['sharpe_ratio'] else 'underperformed'} Buy & Hold on risk-adjusted basis (raw, unfiltered signal)",
    f"5. PREDICTION QUALITY: {acc_blend:.1%} accuracy vs. a 50% coin-flip baseline — {'modestly above' if acc_blend > 0.5 else 'at or below'} chance, consistent with the near-0.5 AUCs above; a weak, threshold-dependent edge rather than a strong standalone signal",
    f"6. DIVERSIFICATION BENEFIT: Ensemble combines {len(base_learners)} diverse models for more robust predictions",
    f"7. CONFIDENCE-BASED RISK CONTROL: threshold overlay on raw meta-model probability trades away most of the raw upside for higher Sharpe/Calmar — best thresholds (4-7.5%) roughly triple the raw ensemble's Sharpe while capping max drawdown under 5%"
]

for insight in insights:
    print(insight)

print()
print("="*120)
print(" " * 35 + "PROJECT SUCCESSFULLY COMPLETED")
print("="*120)

                                        FINAL PROJECT SUMMARY

PROJECT REQUIREMENTS COMPLIANCE
------------------------------------------------------------------------------------------------------------------------
Ticker Selection               : QQQ ETF (Nasdaq-100 tracking)
Prediction Target              : 20d-forward vs trailing realized volatility direction, binary classification [0, 1]
Feature Engineering            : 37 features across 7 categories (Momentum, Trend, Drawdown, Volatility, Volume, Macro, Sentiment)
Data Preprocessing             : Missing value handling, outlier detection, RobustScaler normalization
Base Learners (≥3)             : 5 models - Random Forest, LightGBM, Logistic Regression, SVM, MLP Classifier
Hyperparameter Optimization    : RandomizedSearchCV with TimeSeriesSplit (30 iterations each)
Blending Architecture          : Train-base/train-meta split + meta-model (Logistic Regression)
Evaluation Metrics             : AUC-ROC, Balanced Accuracy, Confusion

In [81]:
# Position distribution for the best-Sharpe strategy (binary: 0=cash, 1=long)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

positions_blend = backtest_results[best_sharpe_name]["positions"]

# Bar chart: how often long vs flat
counts = positions_blend.value_counts().sort_index()
ax1.bar(["Flat (0)", "Long (1)"], [counts.get(0.0, 0), counts.get(1.0, 0)],
        color=["#d62728", "#2ca02c"], edgecolor="black", alpha=0.8)
ax1.set_ylabel("Number of Days", fontsize=11)
ax1.set_title(f"Position Distribution ({best_sharpe_name})", fontsize=12, fontweight="bold")
for bar, cnt in zip(ax1.patches, [counts.get(0.0,0), counts.get(1.0,0)]):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f"{cnt}d ({cnt/len(positions_blend):.0%})", ha="center", fontsize=10)
ax1.grid(True, alpha=0.3, axis="y")

# Time-series of positions
ax2.plot(positions_blend, linewidth=1, alpha=0.8, color="steelblue")
ax2.axhline(y=0.5, color="grey", linestyle=":", linewidth=1, alpha=0.5)
ax2.fill_between(positions_blend.index, 0, positions_blend,
                  where=(positions_blend > 0), alpha=0.3, color="green", label="Long")
ax2.fill_between(positions_blend.index, 0, positions_blend,
                  where=(positions_blend == 0), alpha=0.15, color="red", label="Flat/Cash")
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("Position (0=Cash, 1=Long)", fontsize=11)
ax2.set_title(f"Position Over Time ({best_sharpe_name})", fontsize=12, fontweight="bold")
ax2.legend(loc="best", fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Position statistics:")
print(f"  % Time Long  (pos=1): {(positions_blend == 1).mean():.1%}")
print(f"  % Time Flat  (pos=0): {(positions_blend == 0).mean():.1%}")
print(f"  Total trading days:   {len(positions_blend)}")


Position statistics:
  % Time Long  (pos=1): 100.0%
  % Time Flat  (pos=0): 0.0%
  Total trading days:   218


## 7.3 True Out-of-Sample Test: 2026

A short, forward-deployment-style robustness check: fresh 2026 data (not seen during training) is independently rebuilt through the same feature pipeline and scored with the **already-trained** base learners, meta-model, and calibrator — no refitting. At ~130 trading days this is a directional sanity check, not a statistically powered test; treat the result as a robustness note, not a headline finding.

In [82]:
# ══════════════════════════════════════════════════════════════
# 2026 OUT-OF-SAMPLE TEST
# ══════════════════════════════════════════════════════════════
import yfinance as yf
import pandas_datareader as pdr
import warnings
warnings.filterwarnings('ignore')

START_OOS  = '2025-10-01'   # warm-up window
END_OOS    = '2026-08-02'
TEST_START = '2026-01-01'

# ── Step 1: Download OOS data ─────────────────────────────────
print("Downloading 2026 OOS data...")

def dl(ticker):
    d = yf.download(ticker, start=START_OOS, end=END_OOS,
                    auto_adjust=False, progress=False)  # auto_adjust=False keeps 'Adj Close'
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d

qqq_oos  = dl('QQQ')
vix_oos  = dl('^VIX')
tnx_oos  = dl('^TNX')
dxy_oos  = dl('DX-Y.NYB')
sox_oos  = dl('^SOX')
gold_oos = dl('GC=F')
cop_oos  = dl('HG=F')
vxn_oos  = dl('^VXN')
spy_oos  = dl('SPY')
print("  QQQ/macro data downloaded")

try:
    lev_oos = yf.download(['TQQQ','SQQQ'], start=START_OOS, end=END_OOS,
                          auto_adjust=False, progress=False)
    tqqq_vol = lev_oos[('Volume','TQQQ')] if isinstance(lev_oos.columns, pd.MultiIndex) else lev_oos['Volume']['TQQQ']
    sqqq_vol = lev_oos[('Volume','SQQQ')] if isinstance(lev_oos.columns, pd.MultiIndex) else lev_oos['Volume']['SQQQ']
    print("  TQQQ/SQQQ downloaded")
except Exception as e:
    tqqq_vol = None
    print(f"  TQQQ/SQQQ failed: {e}")

# ── Step 2: Build df_all (warm-up + OOS) ─────────────────────
# Use last 120 days of df_fe as warm-up for rolling windows
df_hist = df_fe[df_fe.index < TEST_START].tail(120)[
    ['Open','High','Low','Close','Adj Close','Volume']].copy()

df_new = qqq_oos[['Open','High','Low','Close','Adj Close','Volume']].copy()
df_all = pd.concat([df_hist, df_new])
df_all = df_all[~df_all.index.duplicated()].sort_index()

# ── Step 3: Replicate ALL feature computations ────────────────
# Returns
df_all['ret_log']    = np.log(df_all['Adj Close'] / df_all['Adj Close'].shift(1))
df_all['ret_simple'] = df_all['Adj Close'].pct_change()

# Momentum
for w in [5, 10]:
    df_all[f'ret_ma_{w}'] = df_all['ret_log'].rolling(w).mean()

# Trend
for w in [20, 60]:
    df_all[f'price_ma_{w}'] = df_all['Adj Close'].rolling(w).mean()
df_all['trend_pos']         = df_all['Adj Close'] - df_all['price_ma_20']
df_all['trend_pos_norm']    = df_all['trend_pos'] / df_all['Adj Close']
df_all['ma20_slope']        = (df_all['price_ma_20'] - df_all['price_ma_20'].shift(5)) / 5
df_all['ma60_slope']        = (df_all['price_ma_60'] - df_all['price_ma_60'].shift(10)) / 10
df_all['golden_cross']      = df_all['price_ma_20'] - df_all['price_ma_60']
df_all['golden_cross_norm'] = df_all['golden_cross'] / df_all['price_ma_60']

# Bollinger Bands
df_all['bb_std']   = df_all['Adj Close'].rolling(20).std()
df_all['bb_upper'] = df_all['price_ma_20'] + 2 * df_all['bb_std']
df_all['bb_lower'] = df_all['price_ma_20'] - 2 * df_all['bb_std']
bb_rng = (df_all['bb_upper'] - df_all['bb_lower']).replace(0, np.nan)
df_all['bb_position']     = (df_all['Adj Close'] - df_all['bb_lower']) / bb_rng
df_all['bb_width']        = bb_rng / df_all['price_ma_20']
df_all['bb_breakout_up']  = ((df_all['Adj Close'] > df_all['bb_upper']) &
    (df_all['Adj Close'].shift(1) <= df_all['bb_upper'].shift(1))).astype(int)
df_all['bb_breakout_down']= ((df_all['Adj Close'] < df_all['bb_lower']) &
    (df_all['Adj Close'].shift(1) >= df_all['bb_lower'].shift(1))).astype(int)
df_all['bb_width_q20_60'] = df_all['bb_width'].rolling(60).quantile(0.2)
df_all['bb_squeeze']      = (df_all['bb_width'] < df_all['bb_width_q20_60']).astype(int)

# Volatility
df_all['vol_20']          = df_all['ret_log'].rolling(20).std()
df_all['log_rv_20']       = np.log(1 + df_all['vol_20'])
df_all['vol_regime']      = df_all['vol_20'] / df_all['vol_20'].rolling(60).median()
df_all['vol_trend']       = df_all['vol_20'] / df_all['vol_20'].shift(5)
df_all['daily_range']     = (df_all['High'] - df_all['Low']) / df_all['Close']
df_all['range_5d_smooth'] = df_all['daily_range'].rolling(5).mean()

# Drawdown
df_all['rolling_max_60']      = df_all['Adj Close'].rolling(60).max()
df_all['rolling_min_60']      = df_all['Adj Close'].rolling(60).min()
df_all['rolling_min_20']      = df_all['Adj Close'].rolling(20).min()
df_all['dd_60']               = df_all['Adj Close'] / df_all['rolling_max_60'] - 1
rng60 = (df_all['rolling_max_60'] - df_all['rolling_min_60']).replace(0, np.nan)
df_all['price_percentile_60'] = (df_all['Adj Close'] - df_all['rolling_min_60']) / rng60
df_all['recovery_strength']   = (df_all['Adj Close'] - df_all['rolling_min_20']) / df_all['rolling_min_20']
df_all['dd_vol_ratio']        = df_all['dd_60'].abs() / df_all['vol_20'].replace(0, np.nan)
df_all['recovery_speed']      = df_all['recovery_strength'] / 1
df_all['log_dd_60']           = np.log(df_all['dd_60'].abs() + 1e-9)

# Volume
df_all['log_vol']     = np.log(1 + df_all['Volume'])
df_all['vol_mean_20'] = df_all['log_vol'].rolling(20).mean()
df_all['vol_std_20']  = df_all['log_vol'].rolling(20).std()
df_all['vol_z_20']    = (df_all['log_vol'] - df_all['vol_mean_20']) / df_all['vol_std_20']

# up_down_vol_ratio (vectorized, matches original)
_up   = df_all['ret_log'] > 0
_dn   = df_all['ret_log'] < 0
_up_vol_sum  = (df_all['log_vol'] * _up).rolling(20, min_periods=1).sum()
_dn_vol_sum  = (df_all['log_vol'] * _dn).rolling(20, min_periods=1).sum()
_up_cnt      = _up.rolling(20, min_periods=1).sum()
_dn_cnt      = _dn.rolling(20, min_periods=1).sum()
_up_mean     = _up_vol_sum / _up_cnt.replace(0, np.nan)
_dn_mean     = _dn_vol_sum / _dn_cnt.replace(0, np.nan)
df_all['up_down_vol_ratio'] = (_up_mean / _dn_mean).clip(0.1, 10).fillna(1.0)

# breakout_volume_flag
_breakout = (
    ((df_all['Adj Close'] > df_all['price_ma_20']) &
     (df_all['Adj Close'].shift(1) <= df_all['price_ma_20'].shift(1))) |
    ((df_all['Adj Close'] > df_all['bb_upper']) &
     (df_all['Adj Close'].shift(1) <= df_all['bb_upper'].shift(1)))
)
df_all['breakout_volume_flag'] = (_breakout & (df_all['vol_z_20'] > 1.0)).astype(int)

# Macro
for col, series in {
    'vix'         : vix_oos['Close'],
    'yield_10y'   : tnx_oos['Close'] / 100,
    'dxy'         : dxy_oos['Close'],
    'semi'        : sox_oos['Close'],
    'gold_copper' : gold_oos['Close'] / cop_oos['Close'].replace(0, np.nan),
}.items():
    df_all[col] = series.reindex(df_all.index, method='ffill')

# FRED macro
try:
    df_all['real_rate']   = pdr.DataReader('DFII10','fred',START_OOS,END_OOS).reindex(df_all.index,method='ffill')
    dgs10 = pdr.DataReader('DGS10','fred',START_OOS,END_OOS)
    dgs2  = pdr.DataReader('DGS2', 'fred',START_OOS,END_OOS)
    df_all['yield_curve'] = (dgs10['DGS10'] - dgs2['DGS2']).reindex(df_all.index,method='ffill')
    df_all['credit_spread'] = pdr.DataReader('BAMLC0A4CBBB','fred',START_OOS,END_OOS).reindex(df_all.index,method='ffill')
except:
    for col in ['real_rate','yield_curve','credit_spread']:
        df_all[col] = df_fe[col].reindex(df_all.index, method='ffill')

# VIX-derived features (cell 54 in training)
df_all['vix_chg_1d']    = df_all['vix'].pct_change(1)
df_all['vix_chg_5d']    = df_all['vix'].pct_change(5)
df_all['vix_zscore_20'] = ((df_all['vix'] - df_all['vix'].rolling(20).mean()) /
                            df_all['vix'].rolling(20).std())
df_all['overnight_gap']     = (df_all['Open'] - df_all['Adj Close'].shift(1)) / df_all['Adj Close'].shift(1)
df_all['overnight_gap_z20'] = ((df_all['overnight_gap'] - df_all['overnight_gap'].rolling(20).mean()) /
                                df_all['overnight_gap'].rolling(20).std())

# VXN features (cell 55 in training)
df_all['vxn'] = vxn_oos['Close'].reindex(df_all.index, method='ffill')
df_all['vxn_vix_ratio']  = df_all['vxn'] / df_all['vix'].replace(0, np.nan)
df_all['vxn_chg_1d']    = df_all['vxn'].pct_change(1)
df_all['vxn_zscore_20'] = ((df_all['vxn'] - df_all['vxn'].rolling(20).mean()) /
                            df_all['vxn'].rolling(20).std())

# TQQQ/SQQQ ratio
if tqqq_vol is not None:
    ts_ratio = (tqqq_vol.squeeze() / (sqqq_vol.squeeze() + 1e-6))
    df_all['tqqq_sqqq_ratio'] = ts_ratio.reindex(df_all.index, method='ffill')
    df_all['tqqq_sqqq_log']   = np.log(df_all['tqqq_sqqq_ratio'].clip(lower=0.01))
    df_all['tqqq_sqqq_zscore']= ((df_all['tqqq_sqqq_log'] - df_all['tqqq_sqqq_log'].rolling(20).mean()) /
                                  df_all['tqqq_sqqq_log'].rolling(20).std())
    df_all['tqqq_sqqq_ma5']   = df_all['tqqq_sqqq_log'].rolling(5).mean()
else:
    for c in ['tqqq_sqqq_ratio','tqqq_sqqq_log','tqqq_sqqq_zscore','tqqq_sqqq_ma5']:
        df_all[c] = np.nan

# QQQ/SPY ratio
df_all['spy_close_t'] = spy_oos['Close'].reindex(df_all.index, method='ffill')
df_all['qqq_spy_ratio']      = df_all['Adj Close'] / df_all['spy_close_t'].replace(0, np.nan)
df_all['qqq_spy_ratio_chg5'] = df_all['qqq_spy_ratio'].pct_change(5)
df_all['qqq_spy_ratio_zscore']= ((df_all['qqq_spy_ratio'] - df_all['qqq_spy_ratio'].rolling(20).mean()) /
                                   df_all['qqq_spy_ratio'].rolling(20).std())
df_all.drop(columns=['spy_close_t'], inplace=True)

# P/C proxy (SQQQ/TQQQ inverted) — same as training fallback
if tqqq_vol is not None:
    df_all['pc_proxy']        = 1.0 / df_all['tqqq_sqqq_ratio'].clip(lower=0.01)
    df_all['pc_proxy_log']    = np.log(df_all['pc_proxy'].clip(lower=0.01))
    df_all['pc_proxy_ma5']    = df_all['pc_proxy_log'].rolling(5).mean()
    df_all['pc_proxy_zscore'] = ((df_all['pc_proxy_log'] - df_all['pc_proxy_log'].rolling(20).mean()) /
                                  df_all['pc_proxy_log'].rolling(20).std())
else:
    for c in ['pc_proxy','pc_proxy_log','pc_proxy_ma5','pc_proxy_zscore']:
        df_all[c] = np.nan

# ── Step 4: Extract 2026 only ─────────────────────────────────
df_oos   = df_all[df_all.index >= TEST_START].copy()
ret_oos  = df_oos['ret_log'].fillna(0)
total_bh = (1 + ret_oos).prod() - 1

print(f"\n2026 test period: {len(df_oos)} trading days")
print(f"  {df_oos.index.min().date()} → {df_oos.index.max().date()}")
print(f"  QQQ total return: {total_bh:.2%}")

# ── Step 5: Scale & predict ───────────────────────────────────
# Only use feature_cols that exist in df_oos; fill rest with 0
oos_feat_cols = [c for c in feature_cols if c in df_oos.columns]
missing_cols  = [c for c in feature_cols if c not in df_oos.columns]
if missing_cols:
    print(f"\nWARNING: {len(missing_cols)} features not in OOS data, filled with 0:")
    print(f"  {missing_cols}")

X_oos_raw = pd.DataFrame(0.0, index=df_oos.index, columns=feature_cols)
X_oos_raw[oos_feat_cols] = df_oos[oos_feat_cols]
X_oos_raw = X_oos_raw.fillna(0)

X_oos_sc = pd.DataFrame(
    scaler.transform(X_oos_raw),
    columns=feature_cols,
    index=df_oos.index
).fillna(0)

# Meta-features from all base learners
meta_oos = pd.DataFrame(index=X_oos_sc.index)
for name, model in base_learners.items():
    col = name.lower().replace(' ', '_')
    meta_oos[col] = model.predict_proba(X_oos_sc)[:, 1]

# Ensemble probability from the (uncalibrated) blending meta-model
prob_oos = pd.Series(meta_model.predict_proba(meta_oos)[:, 1], index=ret_oos.index)
ensemble_label = 'Blending Ensemble'

print(f"\nProbability stats (2026, {ensemble_label}):")
print(f"  Min={prob_oos.min():.4f}  Max={prob_oos.max():.4f}  Std={prob_oos.std():.4f}  >0.5: {(prob_oos>0.5).mean():.1%}")

# ── Step 6: Backtest 2026 ─────────────────────────────────────
bt_oos = {}

bt_oos['Buy & Hold'] = backtest_strategy(
    pd.Series(1.0, index=ret_oos.index), ret_oos, 'Buy & Hold')

bt_oos['Logistic Regression'] = backtest_strategy(
    pd.Series(base_learners['Logistic Regression'].predict_proba(X_oos_sc)[:, 1],
              index=ret_oos.index),
    ret_oos, 'Logistic Regression')

for thresh in [0.01, 0.02, 0.03, 0.04, 0.05]:
    label = f'Blend thresh={thresh:.0%}'
    bt_oos[label] = backtest_strategy(prob_oos, ret_oos, label, confidence_threshold=thresh)

bt_oos[ensemble_label] = backtest_strategy(prob_oos, ret_oos, ensemble_label)

# ── Step 7: Display (same columns as main backtest table) ─────
print("\n" + "="*120)
print("2026 OUT-OF-SAMPLE RESULTS — same metrics as in-sample table")
print("="*120)

oos_rows = []
for name, bt in bt_oos.items():
    oos_rows.append({
        'Strategy'    : name,
        'Total Return': f"{bt['total_return']:.2%}",
        'Annual Return': f"{bt['annualized_return']:.2%}",
        'Volatility'  : f"{bt['volatility']:.2%}",
        'Sharpe Ratio': f"{bt['sharpe_ratio']:.3f}",
        'Max Drawdown': f"{bt['max_drawdown']:.2%}",
        'Calmar Ratio': f"{bt['calmar_ratio']:.3f}",
        'Win Rate'    : f"{bt['win_rate']:.2%}" if not np.isnan(bt['win_rate']) else 'N/A',
        'In Market'   : f"{bt.get('in_market_pct',1.0):.0%}",
    })
oos_df = pd.DataFrame(oos_rows).set_index('Strategy')
oos_df['_s'] = [bt_oos[idx]['sharpe_ratio'] for idx in oos_df.index]
oos_df = oos_df.sort_values('_s', ascending=False).drop('_s', axis=1)
print(oos_df.to_string())
print("="*120)

# ── Step 8: Side-by-side in-sample vs OOS comparison ──────────
print("\n" + "="*85)
print("IN-SAMPLE (test 2024-25)  vs  OUT-OF-SAMPLE (2026)")
print("="*85)
print(f"{'Strategy':<26} {'IS Sharpe':>9} {'IS MaxDD':>8} {'IS Ann%':>7}  "
      f"{'OOS Sharpe':>10} {'OOS MaxDD':>9} {'OOS Ann%':>8}")
print("-"*85)

compare_pairs = [
    ('Buy & Hold',          'Buy & Hold'),
    ('Blending Ensemble',   ensemble_label),
    ('Logistic Regression', 'Logistic Regression'),
]
for is_key, oos_key in compare_pairs:
    bu = backtest_results.get(is_key, {})
    oo = bt_oos.get(oos_key, {})
    if bu and oo:
        print(f"{is_key:<26} "
              f"{bu['sharpe_ratio']:>9.3f} "
              f"{bu['max_drawdown']:>7.2%} "
              f"{bu['annualized_return']:>6.2%}  "
              f"{oo['sharpe_ratio']:>10.3f} "
              f"{oo['max_drawdown']:>8.2%} "
              f"{oo['annualized_return']:>7.2%}")


  QQQ/macro data downloaded
  TQQQ/SQQQ downloaded

2026 test period: 145 trading days
  2026-01-02 → 2026-07-31
  QQQ total return: 10.75%



Probability stats (2026, Blending Ensemble):
  Min=0.6389  Max=0.7651  Std=0.0301  >0.5: 100.0%

2026 OUT-OF-SAMPLE RESULTS — same metrics as in-sample table
                    Total Return Annual Return Volatility Sharpe Ratio Max Drawdown Calmar Ratio Win Rate In Market
Strategy                                                                                                           
Logistic Regression        8.40%        15.16%     13.09%        1.158       -5.05%        3.000   57.14%       34%
Buy & Hold                10.97%        19.98%     21.77%        0.918      -11.98%        1.668   54.86%      100%
Blend thresh=1%           10.97%        19.98%     21.77%        0.918      -11.98%        1.668   54.86%      100%
Blend thresh=2%           10.97%        19.98%     21.77%        0.918      -11.98%        1.668   54.86%      100%
Blend thresh=3%           10.97%        19.98%     21.77%        0.918      -11.98%        1.668   54.86%      100%
Blend thresh=4%           10.

## 7.4 Discussion, Limitations & Further Development

**This section is a template — fill in the bracketed [ ] numbers once you've run the notebook.** It intentionally does not carry over the price-direction version's specific findings (its numbers do not apply to a different target variable); only the methodology points that are still true by construction are kept.

### Strengths
- OOF blending with TimeSeriesSplit respects temporal ordering, preventing data leakage across all folds
- `backtest_strategy`'s look-ahead-bias fix (T's signal scored against T+1's return, `shift_returns=True`) carries over unchanged from the price-direction version — this target swap does not reintroduce that bug
- The label polarity (`label=1` ⇔ "calm ahead") was chosen specifically so the existing `prob > 0.5 -> long` trading rule stays economically correct without touching any backtest code
- Volatility is a fundamentally different forecasting problem from price direction: it clusters and mean-reverts (ARCH/GARCH effects are well documented for equity indices) rather than approximating a random walk, and several existing features (`vix`, `vxn`, `vol_20`, `vol_regime`, `daily_range`, `range_5d_smooth`, `bb_width`) are direct or closely related measures of realized/implied volatility — so, unlike price direction, there is a genuine economic reason to expect AUC meaningfully above 0.5 here

### Limitations
- **Forecast horizon vs. rebalancing frequency mismatch**: the label looks 20 trading days ahead, but the backtest (reused unchanged from Section 7.2) rebalances daily off a rolling 20-day-ahead forecast. This is standard practice for a continuously-updated exposure overlay, but it means consecutive days' positions are highly autocorrelated (the label windows overlap by 19/20 days) — the effective number of independent observations is much smaller than the row count, so significance/robustness claims should account for that
- **[FILL IN] Classification performance**: report AUC/balanced accuracy per base learner and the ensemble here once run, and compare against the price-direction version's ~0.50-0.51 AUCs
- **[FILL IN] Backtest performance**: report Sharpe/annual return/max drawdown for the ensemble vs. Buy & Hold here once run
- **[FILL IN] 2026 OOS check (Section 7.3)**: report whether the OOS probability range saturates the way the price-direction version's did (min=0.503, 100% one-sided) — a similarly saturated range here would indicate the same underlying issue (short, one-regime OOS window), not a new problem
- **Run-to-run instability**: the price-direction version documented material Sharpe/AUC swings between identical repeated runs, traced partly to nested `n_jobs=-1` and unpinned multi-threaded BLAS — the same base-learner code is reused here unchanged, so the same instability should be expected and headline numbers should be read as one draw, not a fixed result
- **Transaction costs**: backtest assumes zero transaction costs and perfect execution at close price

### Further Development
- If [FILL IN] AUC confirms a real edge, test sensitivity to the forecast horizon (5d/10d/20d/60d forward vol) rather than only 20d
- Compare against a simple GARCH(1,1) or EWMA volatility forecast baseline — the bar for "the ML ensemble adds value" is beating that, not beating a coin flip
- Extend training window to include multiple volatility regimes (2008, 2020 crash, 2022 bear market) since the current 2020-2025 window is dominated by a single, mostly-calm bull regime
- Add transaction cost model and slippage assumptions
- Replace binary long/flat with fractional position sizing tied to the (raw) probability magnitude, closer to a real vol-targeting overlay
